# Copperbelt surface-water and vegetation monitoring with NDWI | v5 multi-site (physically and statistically consistent figures)
**Question:** are the Copperbelt's water bodies shrinking, and are the plants around them drying out? Each water body is analysed **on its own** (so a stable dam is not averaged with a failing one), then the results are combined into one regional conclusion.

**Index definitions**
- Open water: NDWI = (Green - NIR) / (Green + NIR)
- Vegetation water content: NDWI (Gao) = (NIR - SWIR) / (NIR + SWIR)

**Sites (pins from Google Places; Cell 7 checks that real water lies near each pin)**
| Site | Town | Type | Why it is here |
|---|---|---|---|
| Mindolo Dam | Kitwe | reservoir | Benchmark from v3 |
| Makoma Dam | Luanshya | reservoir | Large former mine-recreation lake |
| Kafubu Dam | Ndola | reservoir | Ndola water-supply reservoir |
| Fitula Dam | Chingola | reservoir | Western Copperbelt |
| Kaluba Dam | Mufulira | reservoir | Northern Copperbelt |

**Method in brief**
1. **Data:** Landsat 5/7/8/9 Collection 2 Level-2 (TM/ETM+/OLI, 30 m), 1996-2025, two seasons per year: end of the rains (Mar-May) and end of the dry season (Aug-Oct). Sentinel-2 MSI (2015-2025) validates Landsat. MODIS is not used: at 250-500 m it cannot resolve dams of 40-300 ha.
2. **Open water:** NDWI = (Green - NIR)/(Green + NIR) > 0. Each water body is the connected water nearest its pin that was wet in at least 10% of years (its usual footprint); water cover = share of that footprint under water, so one dam is never mixed with neighbouring ponds.
3. **Canopy water:** NDWI = (NIR - SWIR)/(NIR + SWIR) of natural vegetation within 300 m of the shore (the early-warning signal for plant water deficit), with NDVI alongside to test whether NDWI drops first.
4. **Trends:** Mann-Kendall with the Hamed-Rao correction for year-to-year memory, Sen slope with its 95% range, a minimum effect size of 5% per decade for water area, false-discovery-rate checks, and a reliability flag per dam (size, Landsat vs Sentinel-2 agreement, consistency between Landsat generations).
5. **Climate:** ERA5 rainfall for every dam; one drought definition (SPI-6, Dec-May rains) used in every figure; a rainfall test asks whether a water trend remains after the rainfall effect is removed.


**Run order:** Cells 1 to 5 once, Cell 7 (look at the gallery and fix any site flagged ENLARGE or EXCLUDE), then the rest top to bottom. Step 3 (climate) now runs straight after Step 2 so every later figure marks the same drought years. Your cached downloads are reused (the cache key has not changed). The first download is slow (5 sites x 30 years x 2 seasons); later runs read the cache. For a quick test set `ACTIVE_SITES = ["mindolo", "makoma"]` in Cell 2.

**Figures**
Core figures, in reading order (`figures/`):
| # | Figure | What it answers |
|---|---|---|
| 01 | `01_data_availability` | Which years have usable data at each dam (the evidence for 1996-2025) |
| 02 | `02_study_sites` | The five water bodies and their tracked outlines |
| 03 | `03_ndwi_seasonal_trend` | **The headline visual:** are open-water NDWI and canopy NDWI declining? (blue = high water / canopy moisture, brown = low moisture / stress) |
| 04 | `04_ndwi_monthly_heatmap` | The same NDWI month by month |
| 05 | `05_site_<dam>` | One page per dam: where water was lost or gained in each of the three decades, water cover with decade means, shoreline canopy NDWI, drought index |
| 06 | `06_seasonal_drawdown` | Reservoir surface-area fluctuation between the rains and the dry season |
| 07 | `07_canopy_stress_<dam>` | Spatial assessment of canopy water stress around each dam |
| 08 | `08_early_warning_yearly` | In drought years, does canopy NDWI drop while NDVI is still normal? |
| 09 | `09_early_warning_monthly` | Does canopy NDWI drop in an earlier month than NDVI? |
| 10 | `10_trend_scoreboard` | All dams side by side: water and canopy trends with 95% ranges |

Supplementary figures (`figures/supplementary/`, switch off with `MAKE_SUPPLEMENTARY = False` in Cell 2):
| # | Figure | Purpose |
|---|---|---|
| S1 | `S1_sensor_harmonisation` | Method: the Landsat 5/7 to 8/9 offset that is removed |
| S2 | `S2_landsat_vs_sentinel2` | Method: does Landsat agree with Sentinel-2? |
| S3 | `S3_landuse_<dam>` | Land cover around each dam (natural vegetation vs cropland) |
| S4 | `S4_vegetation_trends_<dam>` | Pixel maps of greenness and canopy-water trends (FDR-corrected) |
| S5 | `S5_vegetation_summary` | Land cover, greenness trend and MNDWI cross-check for all dams |
| S6 | `S6_drought_condition_timeline` | Share of vegetation in drought condition, all dams, every year |
| S7 | `S7_drought_years` | How each dam responded in the regional drought years |
| S8 | `S8_threshold_sensitivity` | How much the NDWI water threshold changes the area |
| S9 | `S9_climate_drivers` | Correlation of each dam with each climate variable (FDR-marked) |
| S10 | `S10_rain_timescales` | Over how many months of rain each dam responds |
| S11 | `S11_shoreline_retreat` | Waterline retreat in metres, decade by decade (1996-2005, 2006-2015, 2016-2025) |
| S12 | `S12_synchrony` | Do the dams rise and fall together? |
| S13 | `S13_site_map` | Map of trends by site |
| S14 | `S14_conclusion` | The auto-written conclusion as a page |


In [ ]:
# Cell 1: install (run once, then restart the kernel if pip asks you to)
%pip install -q pystac-client planetary-computer odc-stac rioxarray xarray "dask[complete]" netcdf4 pandas numpy scipy matplotlib pymannkendall openpyxl adlfs zarr rasterio requests pyproj folium

In [ ]:
# Cell 2: CONFIGURATION (multi-site Copperbelt study)
from pathlib import Path
import json, warnings, hashlib, copy
warnings.filterwarnings("ignore")

AREA_NAME = "Copperbelt_water_bodies"
CRS = "EPSG:32735"          # UTM 35S covers 24E-30E: every site below sits inside it

# ---------------- the sites ----------------
# kind: "reservoir" (area trend), "new_reservoir" (filling story, no 30-yr baseline), "river" (reach), "sinkhole" (tiny, stable reference)
# half_deg = half-width of the analysis box around the pin (0.05 deg is about 5.5 km). Raise it if the screening gallery says the water touches the box edge.
# Pins come from Google Places. Cell 7 checks that real water lies within ~1 km of every pin before any long download.
SITES = {
    "mindolo":    dict(name="Mindolo Dam",             town="Kitwe",     lat=-12.7944, lon=28.1401, kind="reservoir",     half_deg=0.08),
    "makoma":     dict(name="Makoma Dam",              town="Luanshya",  lat=-13.1528, lon=28.3710, kind="reservoir",     half_deg=0.05),
    "kafubu":     dict(name="Kafubu Dam",              town="Ndola",     lat=-13.0821, lon=28.5882, kind="reservoir",     half_deg=0.05),
    "fitula":     dict(name="Fitula Dam",              town="Chingola",  lat=-12.6650, lon=27.8602, kind="reservoir",     half_deg=0.05),
    "kaluba":     dict(name="Kaluba Dam",              town="Mufulira",  lat=-12.4738, lon=28.2389, kind="reservoir",     half_deg=0.06),
}
ACTIVE_SITES = list(SITES)          # the five study reservoirs. For a quick test use e.g. ["mindolo", "makoma"]
SENSORS_BY_KIND = {"reservoir": ["landsat", "s2"], "new_reservoir": ["landsat", "s2"], "river": ["landsat", "s2"], "sinkhole": ["s2fine"]}

# ---------------- time ----------------
LANDSAT_YEARS = list(range(1996, 2026))   # 30 years, 1996-2025: the ONE analysis period for every dam, figure, window and trend.
                                          # Why 1996: it is the first year with Landsat images at every dam in the archive record used here, and 30 years is the
                                          # standard (WMO) length of a climate normal. Years inside the period with too few clear images are dropped (Cell 9a shows which and why).
S2_YEARS = list(range(2015, 2026))         # Sentinel-2 starts in 2015 because the satellite was launched in June 2015 (validation only)
SEASONS = {"dry": (8, 10), "mam": (3, 5)}      # end of dry season (water low) and end of rains (water high / vegetation green)
WATER_SEASON, VEG_SEASON = "dry", "mam"
DECADES = [(1996, 2005), (2006, 2015), (2016, 2025)]   # three EQUAL 10-year windows covering the whole period: decade maps, decade means, shoreline change
DECADE_LABEL = [f"{a}-{b}" for a, b in DECADES]
EARLY, LATE = DECADES[0], DECADES[-1]                  # first and last decade (used only for the PARTIAL-record check)
SCREEN_DATES = "2024-08-01/2025-10-31"        # recent dry season, used only for the pre-download screening gallery
RING_WIDE_M = 1500                           # natural vegetation within 1.5 km of the shore: zone for the canopy-stress maps and the vegetation validation
SENSOR_BREAK = 2013
LANDSAT_PLATFORMS = ["landsat-5", "landsat-7", "landsat-8", "landsat-9"]   # TM (L5), ETM+ (L7), OLI (L8/L9): all that flew 1996-2025

# ---------------- resolution / memory ----------------
S2_FINE_RES = 10
BATCH_ITEMS = 6

# ---------------- thresholds ----------------
WATER_THR = 0.0
STRESS_Z = -1.0
USE_WORLDCOVER = True
NDVI_VEG_THR = 0.3
S2_VALID_SCL = [4, 5, 6, 7]
MAX_TILE_CLOUD = 70
MIN_VALID_FRAC_SCENE = 0.5
MIN_COVERAGE_YEAR = 0.9
MIN_SCENES_YEAR = 3

# ---------------- site analysis rules ----------------
FOOT_OCC = 0.10            # a pixel is in the lake's "usual footprint" if it was water in >= 10% of usable years
RING_M = 300               # shoreline vegetation band: 0 to 300 m outside the footprint
MIN_FOOT_COV = 0.90        # a year is used only if >= 90% of the footprint was seen clearly
MIN_AREA_TREND_HA = 20     # smaller than this, 30 m pixels are mostly mixed and trends are unreliable
MAX_PIN_DIST_M = 1000      # the pin must be within 1 km of detected water
USE_HAMED_RAO = True       # trend significance uses the Hamed-Rao test, which allows for year-to-year memory in the series (the plain Mann-Kendall test is too eager)
MIN_EFFECT_PCT_DEC = 5     # a water-area trend is only called a decline/increase if it is also at least this big (% of the mean per decade)

# ---------------- physical and statistical consistency rules (v5) ----------------
HARMONISE_VEG = True       # put Landsat 5/7 (TM/ETM+) vegetation indices on the Landsat 8/9 (OLI) scale before ANY trend or anomaly.
                           # OLI's narrower NIR band reads NDVI and canopy NDWI higher; without this the 2013 sensor switch looks like "greening / wetting".
HARM_YEARS = (2013, 2020)  # offset estimated only from years when both sensor families observed the same season (L7 orbit drift grows after ~2020)
L7_DRIFT_YEAR = 2022       # Landsat 7 scenes from this year on (drifted, lowered orbit) are left out of scene-level series
MAX_REFILL_PTS = 20        # physics check: with no inflow between May and October, a dam cannot hold MORE water at the end of the dry season than at the end of the rains.
                           # If the end-of-rains cover is lower by more than this many % points, that end-of-rains value is a detection failure (haze, turbid or plant-covered water) and is dropped.
CLIMATE_SOURCE = "era5"    # one climate product for EVERY site ("era5" = ERA5 via Open-Meteo, 1960-2025; "terraclimate" ends 2024). Mixing products between sites is not allowed.
DROUGHT_SPI = "spi6_mam"   # ONE drought definition everywhere: SPI-6 ending May (= Dec-May rains) below STRESS_Z. It precedes both the end-of-rains and the end-of-dry-season observations (cause before effect).
MIN_YEARS_PER_END = 3      # a trend counts as FULL-RECORD only with >= 3 usable years in BOTH the first and the last 10 years; otherwise it is labelled 'partial record (from YYYY)'
FDR_Q = 0.05               # false-discovery-rate level used for maps of per-pixel trends and for the climate-correlation grids

MAKE_SUPPLEMENTARY = True  # True = also draw the supplementary figures S1-S14 into figures/supplementary/. False = only the 10 core figures (faster).

FORCE_REPROCESS = False

ROOT = Path("ndwi_copperbelt")
DIRS = {k: ROOT / k for k in ["raw_meta", "processed", "tables", "figures", "gis"]}
DIRS["supp"] = DIRS["figures"] / "supplementary"          # supplementary figures (S1-S14) go here; the core figures (01-10) stay in figures/
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

def make_run_id(bbox):
    return hashlib.md5(json.dumps([bbox, CRS, WATER_THR, S2_VALID_SCL, MAX_TILE_CLOUD, USE_WORLDCOVER, NDVI_VEG_THR, S2_FINE_RES,
                                   list(SEASONS.values())]).encode()).hexdigest()[:8]

BBOX = None; CENTER_LAT = None; CENTER_LON = None; RUN_ID = None; SID = None
def set_site(sid):
    """Points the whole pipeline (Cell 5) at one site: sets BBOX and the cache key."""
    global BBOX, CENTER_LAT, CENTER_LON, RUN_ID, SID
    s = SITES[sid]; hd = s["half_deg"]
    BBOX = [round(s["lon"] - hd, 4), round(s["lat"] - hd, 4), round(s["lon"] + hd, 4), round(s["lat"] + hd, 4)]
    CENTER_LAT, CENTER_LON = s["lat"], s["lon"]; RUN_ID = make_run_id(BBOX); SID = sid
    return BBOX

json.dump({k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, list, tuple, dict))},
          open(ROOT / "config_used.json", "w"), indent=2, default=str)
print("Project folder:", ROOT.resolve(), "|", len(ACTIVE_SITES), "sites:", ", ".join(ACTIVE_SITES))


In [ ]:
# Cell 3: imports and connection to the Planetary Computer catalogue
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
import pystac_client, planetary_computer, odc.stac, rioxarray
from scipy import stats
import pymannkendall as mk

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})
print("Connected to:", catalog.title)

In [ ]:
# Cell 4: helper functions (indices, statistics, trend tests)
import numpy as np
from scipy import stats

def savefig(name):
    p = DIRS["figures"] / f"{name}.png"
    plt.savefig(p, dpi=300, bbox_inches="tight")
    print("saved", p)

def fmt_p(p):
    return "p<0.001" if p < 0.001 else f"p={p:.3f}"

def season_range(year, months):
    m0, m1 = months
    last = pd.Timestamp(year=year, month=m1, day=1) + pd.offsets.MonthEnd(0)
    return f"{year}-{m0:02d}-01/{last:%Y-%m-%d}"

def safe_nd(a, b):
    """Normalised difference (a-b)/(a+b), masked where the denominator is not positive."""
    den = a + b
    return ((a - b) / den).where(den > 0)

def _utc_naive(dt):
    ts = pd.Timestamp(dt)
    ts = ts.tz_localize("UTC") if ts.tzinfo is None else ts.tz_convert("UTC")
    return ts.tz_localize(None)

def nearest_item_attr(times, items, fn):
    """For every loaded time slice return fn(item) of the STAC item closest in time."""
    it_t = np.array([_utc_naive(i.datetime).to_datetime64() for i in items])
    vals = [fn(i) for i in items]
    return [vals[int(np.argmin(np.abs(it_t - np.datetime64(t))))] for t in times]

def s2_offset(item):
    """Sentinel-2 processing baseline >= 04.00 (from 2022-01-25) adds +1000 to every DN."""
    pb = item.properties.get("s2:processing_baseline")
    if pb:
        return 1000.0 if float(pb) >= 4.0 else 0.0
    return 1000.0 if _utc_naive(item.datetime) >= pd.Timestamp("2022-01-25") else 0.0

# ---------------- statistics ----------------
def pettitt(x):
    """Pettitt change-point test. Returns (index of last element BEFORE the change, p-value)."""
    x = np.asarray(x, float); n = len(x)
    r = stats.rankdata(x)
    U = np.abs(2 * np.cumsum(r) - np.arange(1, n + 1) * (n + 1))
    t = int(np.argmax(U[:-1])); K = U[t]
    return t, float(min(1.0, 2 * np.exp(-6 * K ** 2 / (n ** 3 + n ** 2))))

def pixel_trend(cube, years, min_n=8, rows_per_chunk=40):
    """Vectorised Mann-Kendall + Sen's slope for every pixel. cube: (time, y, x) with NaN gaps."""
    t = np.asarray(years, float); nt = len(t)
    iu, ju = np.triu_indices(nt, 1)
    dt = (t[ju] - t[iu])[:, None, None].astype("float32")
    ny, nx = cube.shape[1:]
    slope = np.full((ny, nx), np.nan, "float32"); pval = np.full((ny, nx), np.nan, "float32")
    for r0 in range(0, ny, rows_per_chunk):
        c = cube[:, r0:r0 + rows_per_chunk].astype("float32")
        d = c[ju] - c[iu]
        sl = np.nanmedian(d / dt, axis=0)
        S = np.nansum(np.sign(d), axis=0)
        n = np.isfinite(c).sum(0).astype(float)
        var = n * (n - 1) * (2 * n + 5) / 18
        z = np.where(S > 0, S - 1, np.where(S < 0, S + 1, 0)) / np.sqrt(np.where(var > 0, var, np.nan))
        p = 2 * (1 - stats.norm.cdf(np.abs(z)))
        bad = n < min_n
        sl[bad] = np.nan; p[bad] = np.nan
        slope[r0:r0 + rows_per_chunk] = sl; pval[r0:r0 + rows_per_chunk] = p
    return slope, pval

def spi(series, window=12, ref=None):
    """Standardised Precipitation Index from a monthly precipitation Series (gamma fit per calendar month)."""
    s = series.rolling(window).sum()
    out = pd.Series(np.nan, index=s.index)
    for m in range(1, 13):
        sm = s[s.index.month == m].dropna()
        rv = sm if ref is None else sm[(sm.index.year >= ref[0]) & (sm.index.year <= ref[1])]
        pos = rv[rv > 0]
        if len(pos) < 10:
            continue
        q0 = 1 - len(pos) / len(rv)
        a, loc, sc = stats.gamma.fit(pos, floc=0)
        cdf = q0 + (1 - q0) * stats.gamma.cdf(sm, a, loc=0, scale=sc)
        out[sm.index] = stats.norm.ppf(np.clip(cdf, 1e-6, 1 - 1e-6))
    return out

def agreement(x, y):
    """Agreement of y (e.g. Landsat) against reference x (e.g. Sentinel-2)."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y); x, y = x[m], y[m]
    if len(x) < 3:
        return {"n": len(x)}
    out = {"n": len(x), "bias": float(np.mean(y - x)), "rmse": float(np.sqrt(np.mean((y - x) ** 2))),
           "mae": float(np.mean(np.abs(y - x)))}
    if np.ptp(x) == 0 or np.ptp(y) == 0:          # a constant series has no correlation or regression
        out.update(pearson_r=np.nan, pearson_p=np.nan, spearman_rho=np.nan, spearman_p=np.nan,
                   slope=np.nan, intercept=np.nan)
        return out
    r, pr = stats.pearsonr(x, y); rho, ps = stats.spearmanr(x, y)
    lr = stats.linregress(x, y)
    out.update(pearson_r=r, pearson_p=pr, spearman_rho=rho, spearman_p=ps, slope=lr.slope, intercept=lr.intercept)
    return out

def zscore(s):
    s = pd.Series(s, dtype=float)
    sd = s.std(ddof=1)
    return (s - s.mean()) / sd if sd and np.isfinite(sd) and sd > 0 else s * np.nan

def robust_z(x):
    x = x.astype(float)
    if x.notna().sum() < 4:
        return pd.Series(0.0, index=x.index)
    med = x.median(); mad = (x - med).abs().median()
    return (x - med) / (1.4826 * mad) if mad > 0 else pd.Series(0.0, index=x.index)

def record_cover(years):
    """How much of the analysis period a series really covers: usable years in the first and last 10 years of LANDSAT_YEARS."""
    y = np.asarray(years, float); g = globals()
    e0, l1 = g["EARLY"], g["LATE"]
    n_e = int(((y >= e0[0]) & (y <= e0[1])).sum()); n_l = int(((y >= l1[0]) & (y <= l1[1])).sum())
    return {"n_first_decade": n_e, "n_last_decade": n_l, "full_record": bool(n_e >= g.get("MIN_YEARS_PER_END", 3) and n_l >= g.get("MIN_YEARS_PER_END", 3))}

def span_label(t):
    """'1996-2025, n=25' or 'PARTIAL record 1998-2025, n=21' for a trend-table row."""
    if t is None or not hasattr(t, "get") or not np.isfinite(t.get("first_year", np.nan)): return ""
    s_ = f"{int(t['first_year'])}-{int(t['last_year'])}, n={int(t['n_years'])}"
    return s_ if t.get("full_record", True) else f"PARTIAL record {s_}"

def trend_table(df, metrics, min_n=8):
    """Mann-Kendall (+ Hamed-Rao autocorrelation-corrected), Sen's slope with 95% CI, Pettitt change point."""
    rows = []
    for m, label in metrics.items():
        if m not in df.columns:
            continue
        d = df[["year", m]].dropna()
        if len(d) < min_n or np.ptp(d[m].values) == 0:
            continue
        x, y = d["year"].values.astype(float), d[m].values.astype(float)
        o = mk.original_test(y)
        try:
            hr_p = mk.hamed_rao_modification_test(y).p
        except Exception:
            hr_p = np.nan
        sen = stats.theilslopes(y, x, 0.95)
        cp, cp_p = pettitt(y)
        mean = float(np.mean(y))
        rows.append({"metric": m, "label": label, "n_years": len(y), "mean": mean, "mk_trend": o.trend,
                     "mk_p": o.p, "mk_p_hamed_rao": hr_p, "kendall_tau": o.Tau,
                     "sen_slope_per_year": sen[0], "sen_ci_low": sen[2], "sen_ci_high": sen[3],
                     "pct_change_per_decade": (100 * 10 * sen[0] / abs(mean)) if (mean and not m.endswith("_z") and "pct" not in m) else np.nan,
                     "pettitt_change_year": int(x[cp + 1]), "pettitt_p": cp_p,
                     "first_year": int(x.min()), "last_year": int(x.max()), **record_cover(x)})
    return pd.DataFrame(rows)
def bh_fdr(p):
    """Benjamini-Hochberg q-values for an array of p-values (NaN kept as NaN). Used when many tests are run at once (pixels, correlation grids)."""
    p = np.asarray(p, float); q = np.full(p.shape, np.nan); ok = np.isfinite(p)
    if ok.sum() == 0: return q
    pv = p[ok]; o = np.argsort(pv); m = len(pv)
    qq = pv[o] * m / np.arange(1, m + 1); qq = np.minimum.accumulate(qq[::-1])[::-1]
    out = np.empty(m); out[o] = np.clip(qq, 0, 1); q[ok] = out
    return q

def ci_mean_diff(a, b, conf=0.95):
    """Difference of two means (b - a) with a Welch t confidence interval."""
    a, b = np.asarray(a, float), np.asarray(b, float); a, b = a[np.isfinite(a)], b[np.isfinite(b)]
    if len(a) < 2 or len(b) < 2: return np.nan, np.nan, np.nan
    d = b.mean() - a.mean(); va, vb = a.var(ddof=1) / len(a), b.var(ddof=1) / len(b); se = np.sqrt(va + vb)
    df = (va + vb) ** 2 / (va ** 2 / (len(a) - 1) + vb ** 2 / (len(b) - 1)) if (va + vb) > 0 else 1
    h = stats.t.ppf(0.5 + conf / 2, df) * se
    return d, d - h, d + h
print("helpers ready")

In [ ]:
# Cell 4b (NEW): STANDARD FIGURE STYLE. Run once, right after Cell 4.
# Every figure in the notebook uses the colours, labels and layout defined here, so the whole set looks like one report.
import calendar, textwrap
import matplotlib as mpl
from matplotlib.patches import Patch, Rectangle
from matplotlib.lines import Line2D
from matplotlib.colors import ListedColormap, TwoSlopeNorm

# ---------- colour language (the same colour always means the same thing) ----------
C_WATER, C_WATER_L = "#2166ac", "#9ecae1"     # water / wetter / rainfall      -> BLUE
C_VEG, C_VEG_L = "#2e7d32", "#a1d99b"         # vegetation                      -> GREEN
C_DRY, C_DRY_L = "#a6611a", "#dfc27d"         # drier / loss / drought          -> BROWN
C_HOT = "#d7301f"                             # temperature / significant trend -> RED
C_CROP = "#9e5fa3"                            # cropland (farming signal)       -> PURPLE
C_GREY = "#6b6b6b"
SENSOR_COL = {"Landsat": "#4d4d4d", "Sentinel-2": "#e08214"}
FAMILY_COL = {"TM/ETM+ (L5/L7)": "#8c510a", "OLI (L8/L9)": "#01665e"}
PLATFORM_COL = {"landsat-5": "#8c510a", "landsat-7": "#d8b365", "landsat-8": "#5ab4ac", "landsat-9": "#01665e"}
PLATFORM_NAME = {"landsat-5": "Landsat 5", "landsat-7": "Landsat 7", "landsat-8": "Landsat 8", "landsat-9": "Landsat 9"}
C_MOIST = C_WATER                             # canopy water content (Gao NDWI) uses the SAME blue as water: the colour scale is blue = high water / canopy moisture, brown = low moisture / stress
C_RAINS, C_DRYS = "#08306b", "#6baed6"        # season of observation: end of rains = dark blue (fuller), end of dry season = light blue. Brown is kept ONLY for deficits.
SITE_COL = {"mindolo": "#0072B2", "makoma": "#E69F00", "kafubu": "#009E73", "fitula": "#CC79A7", "kaluba": "#56B4E9"}   # Okabe-Ito (colour-blind safe), fixed for every figure
MONTHS_RY = [11, 12, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]            # rain-year order (Nov first)
MON_NAME = {m: calendar.month_abbr[m] for m in range(1, 13)}
FOOTER = "Copperbelt Province, Zambia"

plt.rcParams.update({"figure.dpi": 100, "figure.facecolor": "white", "axes.facecolor": "white",
                     "font.size": 10.5, "axes.titlesize": 11, "axes.titleweight": "bold", "axes.labelsize": 10,
                     "axes.grid": True, "axes.grid.axis": "y", "grid.color": "#dddddd", "grid.alpha": 0.8,
                     "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
                     "xtick.labelsize": 9, "ytick.labelsize": 9})

def season_label(sk):
    m0, m1 = SEASONS[sk]
    nm = {"dry": "End of dry season", "mam": "End of rains"}.get(sk, sk)
    return f"{nm} ({calendar.month_abbr[m0]}-{calendar.month_abbr[m1]})"

METRIC_TITLE = {
    "water_ha": "Surface-water area (dam + river)",
    "ndvi_nat_mean": "Greenness of natural vegetation (NDVI)",
    "moist_nat_mean": "Water content of natural vegetation (moisture NDWI)",
    "stress_pct_z": "Natural vegetation under moisture stress (% of area)",
    "ndvi_crop_mean": "Greenness of cropland (NDVI) - farming signal",
    "moist_crop_mean": "Water content of cropland (NDWI) - farming signal",
    "water_ndwi_mean": "Mean NDWI of water pixels"}
METRIC_YLAB = {"water_ha": "Water area (ha)", "ndvi_nat_mean": "NDVI (higher = greener)", "moist_nat_mean": "NDWI (higher = wetter)",
               "stress_pct_z": "% of natural vegetation", "ndvi_crop_mean": "NDVI (higher = greener)", "moist_crop_mean": "NDWI (higher = wetter)"}
METRIC_UNIT = {"water_ha": "ha", "ndvi_nat_mean": "NDVI", "moist_nat_mean": "NDWI", "stress_pct_z": "% points",
               "ndvi_crop_mean": "NDVI", "moist_crop_mean": "NDWI"}
METRIC_KIND = {"water_ha": "water", "ndvi_nat_mean": "veg", "moist_nat_mean": "veg", "stress_pct_z": "dry",
               "ndvi_crop_mean": "crop", "moist_crop_mean": "crop"}
KIND_COL = {"water": C_WATER, "veg": C_VEG, "dry": C_DRY, "crop": C_CROP}

DRIVER_LABEL = {
    "wet_season_ppt_mm": "Wet-season rainfall (Nov-Apr)", "hydro_year_ppt_mm": "Rain-year rainfall (Nov-Oct)",
    "water_balance_mm": "Rainfall minus evaporation demand", "wet_season_ppt_lag1": "Wet-season rainfall, previous year",
    "spi3_mam": "Drought index SPI-3 (end of May)", "spi6_mam": "Drought index SPI-6 (end of May)",
    "spi3_dry": "Drought index SPI-3 (end of Oct)", "spi6_dry": "Drought index SPI-6 (end of Oct)",
    "spi12_dry": "Drought index SPI-12 (end of Oct)", "spi12_dry_lag1": "Drought index SPI-12, previous year",
    "pdsi_mam": "Palmer drought index (Mar-May)", "pdsi_dry": "Palmer drought index (Aug-Oct)",
    "soil_mm_mam": "Soil moisture (Mar-May)", "soil_mm_dry": "Soil moisture (Aug-Oct)",
    "tmax_mam": "Max temperature (Mar-May)", "tmax_dry": "Max temperature (Aug-Oct)", "pet_dry_mm": "Evaporation demand (Aug-Oct)"}
DRIVER_ORDER = list(DRIVER_LABEL)
RESP_LABEL = {"water_ha": "Dam water area", "ndvi_nat_mean": "Vegetation greenness", "moist_nat_mean": "Vegetation water content",
              "stress_pct_z": "Vegetation under moisture stress"}

# ---------- helpers ----------
def finish(fig, name, title, subtitle=None, source="Landsat 5/7/8/9 Collection 2 Level-2 (30 m), Microsoft Planetary Computer",
           handles=None, legend_cols=6, adjust=None, pad=1.08, w_pad=None, h_pad=None, bot_extra=0.0, top_extra=0.0):
    """Adds the headline (what the viewer should take away), a plain-language sub-line, optional legend and footer, then saves + shows."""
    supp = name.startswith("S")                                  # S1-S14 = supplementary figures
    if supp and not globals().get("MAKE_SUPPLEMENTARY", True):
        plt.close(fig); print(f"skipped supplementary figure {name} (MAKE_SUPPLEMENTARY = False)"); return
    w, h = fig.get_size_inches()
    t_lines = textwrap.wrap(title, max(30, int(w / 0.125)))
    s_lines = textwrap.wrap(subtitle, max(40, int(w / 0.080))) if subtitle else []
    top_in = 0.14 + 0.28 * len(t_lines) + ((0.08 + 0.18 * len(s_lines)) if s_lines else 0) + 0.15 + top_extra
    bot_in = 0.42 + (0.42 if handles else 0) + (0.35 if adjust is not None else 0) + bot_extra
    if adjust is not None:
        fig.subplots_adjust(top=1 - top_in / h, bottom=bot_in / h, **adjust)
    else:
        fig.tight_layout(rect=[0, bot_in / h, 1, 1 - top_in / h], pad=pad, w_pad=w_pad, h_pad=h_pad)
    fig.text(0.006, 1 - 0.12 / h, "\n".join(t_lines), fontsize=14, fontweight="bold", va="top", ha="left", linespacing=1.15)
    if s_lines:
        fig.text(0.006, 1 - (0.12 + 0.28 * len(t_lines) + 0.05) / h, "\n".join(s_lines), fontsize=9.5, color="#444444", va="top", ha="left")
    if handles:
        fig.legend(handles=handles, loc="lower center", ncol=min(len(handles), legend_cols), fontsize=9, bbox_to_anchor=(0.5, 0.30 / h))
    fig.text(0.006, 0.09 / h, f"{FOOTER}  |  Data: {source}", fontsize=7, color=C_GREY, va="bottom", ha="left")
    p = (DIRS["supp"] if supp else DIRS["figures"]) / f"{name}.png"
    fig.savefig(p, dpi=300, bbox_inches="tight", facecolor="white")
    print("saved", p)
    plt.show()

def phrase_p(p, trend):
    """Plain-language verdict. Uses the FINAL trend label (after the Hamed-Rao and effect-size rules), so a tiny-but-'significant' change is never called a decline."""
    if not np.isfinite(p): return "not testable"
    if trend in ("increasing", "decreasing"):
        return f"{trend.capitalize()} - statistically significant ({fmt_p(p)})"
    if p < 0.05:
        return f"No meaningful trend - detectable ({fmt_p(p)}) but too small to matter"
    return f"No clear trend - not significant ({fmt_p(p)})"

def gap_xy(x, y, max_step=1):
    """Insert NaN where consecutive years are more than max_step apart, so lines are NOT drawn across missing years (no invented data)."""
    x = np.asarray(x, float); y = np.asarray(y, float); o = np.argsort(x); x, y = x[o], y[o]
    if len(x) < 2: return x, y
    brk = np.where(np.diff(x) > max_step)[0]
    return np.insert(x, brk + 1, np.nan), np.insert(y, brk + 1, np.nan)

def gap_line(ax, x, y, color, marker="o", ms=4.5, lw=1.5, label=None, mfc=None, ls="-", zorder=3, alpha=1.0, mec="white"):
    """Markers for every value, line segments only between consecutive years."""
    gx, gy = gap_xy(x, y)
    ax.plot(gx, gy, ls=ls, color=color, lw=lw, zorder=zorder, alpha=alpha)
    ax.plot(x, y, marker, color=color, ms=ms, mec=mec, mew=0.6 if mec == "white" else 0.9, mfc=mfc or color, ls="none", label=label, zorder=zorder + 1, alpha=alpha)

def sen_fit(x, y):
    """Sen slope, its 95% CI, and the Conover intercept (median of y - slope * x)."""
    x = np.asarray(x, float); y = np.asarray(y, float); m = np.isfinite(x) & np.isfinite(y); x, y = x[m], y[m]
    s, _, lo, hi = stats.theilslopes(y, x, 0.95)
    return s, lo, hi, float(np.median(y - s * x)), x, y

def sen_band(ax, x, y, color, sig=True, band=True, lw=2.0):
    """Sen trend line (solid if significant, dashed if not) with a shaded 95% confidence wedge pivoting on the data centre."""
    s, lo, hi, b, x, y = sen_fit(x, y)
    if len(x) < 3: return
    xs = np.linspace(x.min(), x.max(), 50); xc, yc = np.median(x), b + s * np.median(x)
    if band:
        ax.fill_between(xs, yc + lo * (xs - xc), yc + hi * (xs - xc), color=color, alpha=0.12, lw=0, zorder=1)
    ax.plot(xs, b + s * xs, color=color, lw=lw if sig else 1.3, ls="-" if sig else "--", zorder=5)

def lat_fmt(v, _=None): return f"{abs(v):.2f}°{'S' if v < 0 else 'N'}"
def lon_fmt(v, _=None): return f"{abs(v):.2f}°{'E' if v >= 0 else 'W'}"

def quick_trend(x, y, min_n=8):
    d = pd.DataFrame({"x": np.asarray(x, float), "y": np.asarray(y, float)}).dropna()
    if len(d) < min_n or d.y.nunique() < 3:
        return None
    o = mk.original_test(d.y.values); sl = stats.theilslopes(d.y.values, d.x.values, 0.95); p = float(o.p)
    if globals().get("USE_HAMED_RAO", False):                    # allow for year-to-year memory in the series
        try: p = float(mk.hamed_rao_modification_test(d.y.values).p)
        except Exception: pass
    tr = ("increasing" if sl[0] > 0 else "decreasing") if (p < 0.05 and sl[0] != 0) else "no trend"
    return {"slope": float(sl[0]), "lo": float(sl[2]), "hi": float(sl[3]), "p": p, "trend": tr, "n": len(d), "mean": float(d.y.mean())}

def shade_eras(ax, x0=None, x1=None):
    """Light background bands: which Landsat generation supplied the images (so sensor switches are visible, not hidden)."""
    x0 = LANDSAT_YEARS[0] - 0.7 if x0 is None else x0; x1 = LANDSAT_YEARS[-1] + 0.7 if x1 is None else x1
    ax.axvspan(x0, SENSOR_BREAK - 0.5, color=FAMILY_COL["TM/ETM+ (L5/L7)"], alpha=0.07, lw=0, zorder=0)
    ax.axvspan(SENSOR_BREAK - 0.5, x1, color=FAMILY_COL["OLI (L8/L9)"], alpha=0.07, lw=0, zorder=0)
    ax.set_xlim(x0, x1)

ERA_HANDLES = [Patch(color=FAMILY_COL["TM/ETM+ (L5/L7)"], alpha=0.25, label="Landsat 5/7 period"),
               Patch(color=FAMILY_COL["OLI (L8/L9)"], alpha=0.25, label="Landsat 8/9 period")]
TREND_HANDLES = [Line2D([0], [0], marker="o", color="#555555", lw=0.8, ms=5, label="Yearly value"),
                 Line2D([0], [0], color="#222222", lw=2.4, label="5-year average"),
                 Line2D([0], [0], color=C_HOT, lw=2, label="Trend line: statistically significant"),
                 Line2D([0], [0], color=C_GREY, lw=1.6, ls="--", label="Trend line: not significant")] + ERA_HANDLES

def trend_panel(ax, d, m, t, title=None, ylabel=None, zero_based=False):
    """One yearly time-series panel with 5-year average, Sen trend line and a plain-language verdict."""
    col = KIND_COL[METRIC_KIND.get(m, "veg")]; unit = METRIC_UNIT.get(m, "")
    x = d["year"].values.astype(float); y = d[m].values.astype(float); sl = float(t.sen_slope_per_year)
    shade_eras(ax)
    ax.plot(x, y, "-", color=col, lw=0.8, alpha=0.5, zorder=2)
    ax.plot(x, y, "o", color=col, ms=5.5, mec="white", mew=0.6, zorder=4)
    ax.plot(x, pd.Series(y, index=x).rolling(5, center=True, min_periods=3).mean().values, color="#222222", lw=2.4, zorder=3)
    sig = t.mk_p < 0.05; icpt = np.median(y - sl * x); xs = np.array([x.min(), x.max()])
    ax.plot(xs, icpt + sl * xs, color=C_HOT if sig else C_GREY, lw=2 if sig else 1.6, ls="-" if sig else "--", zorder=5)
    pdec = t.get("pct_change_per_decade", np.nan) if hasattr(t, "get") else np.nan
    extra = f" ({pdec:+.0f}% of average)" if np.isfinite(pdec) else ""
    ax.text(0.015, 0.97, f"{phrase_p(t.mk_p, t.mk_trend)}\n{sl * 10:+.3g} {unit} per decade{extra}", transform=ax.transAxes, va="top", ha="left",
            fontsize=8.8, bbox=dict(fc="white", ec="#cccccc", alpha=0.92, boxstyle="round,pad=0.3"), zorder=6)
    ax.set_title(title or METRIC_TITLE.get(m, m), loc="left", fontsize=10.5)
    ax.set_ylabel(ylabel or METRIC_YLAB.get(m, ""))
    if zero_based:
        top = np.nanmax(y) * 1.35
        ax.set_ylim(0, top)
    else:
        lo, hi = np.nanmin(y), np.nanmax(y); pad = (hi - lo) * 0.12 + 1e-9
        ax.set_ylim(lo - pad, hi + pad * 3.0)           # head-room for the verdict box

def anomaly_bars(ax, x, pct, kind="water", width=0.8):
    """Bars = % above / below the long-term average. Blue (or green) = more than normal, brown = less than normal."""
    pos = C_WATER if kind in ("water", "moist") else C_VEG
    ax.bar(x, pct, width=width, color=np.where(np.asarray(pct) >= 0, pos, C_DRY), zorder=3)
    ax.axhline(0, color="k", lw=0.9, zorder=4)

def stripes(fig, ax, mat, ylabels, vmax=2.5, cbar_label="Anomaly (standard deviations from the 30-year average)"):
    """Heat-strip: rows = indicators, columns = years. Brown = below average, blue-green = above average, grey = no data."""
    cm = mpl.colormaps["BrBG"].copy(); cm.set_bad("#e6e6e6")
    im = ax.imshow(np.ma.masked_invalid(mat.values.astype(float)), aspect="auto", cmap=cm, norm=TwoSlopeNorm(0, -vmax, vmax), interpolation="nearest")
    yrs = list(mat.columns)
    ax.set_xticks(range(0, len(yrs), 2)); ax.set_xticklabels([yrs[i] for i in range(0, len(yrs), 2)], rotation=0)
    ax.set_yticks(range(len(ylabels))); ax.set_yticklabels(ylabels); ax.grid(False)
    for s in ax.spines.values(): s.set_visible(False)
    cb = fig.colorbar(im, ax=ax, pad=0.012, shrink=0.95); cb.set_label(cbar_label); cb.set_ticks([-2, -1, 0, 1, 2])
    return im

def km_extent(da):
    return [0, (float(da.x.max()) - float(da.x.min())) / 1000, 0, (float(da.y.max()) - float(da.y.min())) / 1000]

def show_map(ax, da, cmap, vmin=None, vmax=None, bad="#eeeeee", title=None):
    """Map in km from the south-west corner, north up, grey where there is no data."""
    cm = mpl.colormaps[cmap].copy() if isinstance(cmap, str) else cmap.copy(); cm.set_bad(bad)
    arr = np.ma.masked_invalid(np.asarray(da.values, float))
    if float(da.y[0]) < float(da.y[-1]): arr = arr[::-1]
    im = ax.imshow(arr, cmap=cm, vmin=vmin, vmax=vmax, extent=km_extent(da), origin="upper", interpolation="nearest")
    ax.set_xlabel("km east"); ax.set_ylabel("km north"); ax.grid(False); ax.set_aspect("equal")
    ax.annotate("", xy=(0.94, 0.90), xytext=(0.94, 0.79), xycoords="axes fraction", arrowprops=dict(arrowstyle="-|>", color="k", lw=1.2))
    ax.text(0.94, 0.915, "N", transform=ax.transAxes, ha="center", va="bottom", fontsize=9, fontweight="bold", bbox=dict(fc="white", ec="none", alpha=0.6, pad=0.5))
    if title: ax.set_title("\n".join(textwrap.fill(l_, 60) for l_ in title.split("\n")), loc="left", fontsize=10.5)
    return im

def metric_scatter(ax, x, y, xlabel, ylabel, title, color, xtxt="r"):
    """Agreement scatter with 1:1 line and a stats box."""
    x, y = np.asarray(x, float), np.asarray(y, float); m = np.isfinite(x) & np.isfinite(y); x, y = x[m], y[m]
    ax.scatter(x, y, color=color, s=40, edgecolor="white", lw=0.6, zorder=3)
    lo, hi = min(x.min(), y.min()), max(x.max(), y.max()); pad = (hi - lo) * 0.06 + 1e-9
    ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], "k--", lw=1, zorder=2)
    ax.set_xlim(lo - pad, hi + pad); ax.set_ylim(lo - pad, hi + pad); ax.set_aspect("equal", adjustable="box")
    g = agreement(x, y)
    ax.text(0.04, 0.96, f"r = {g.get('pearson_r', np.nan):.2f}\nbias = {g['bias']:+.3g}\nn = {int(g['n'])}", transform=ax.transAxes, va="top", fontsize=8.8,
            bbox=dict(fc="white", ec="#cccccc", alpha=0.92, boxstyle="round,pad=0.3"))
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel); ax.set_title(title, loc="left", fontsize=10); ax.grid(True, axis="both")

print("figure style ready (colours: blue = water/wetter, green = vegetation, brown = drier/loss, red = significant trend / heat)")


In [ ]:
# Cell 5: the data pipeline (land cover -> load in batches -> mask -> indices -> composites + scene table)
LCM = {}   # resolution (m) -> {"source": "worldcover"|"ndvi", "classes", "natural", "crop"}

def load_landcover(res):
    """ESA WorldCover 2021 on the analysis grid. natural = tree/shrub/grass (10,20,30); crop = 40."""
    if not USE_WORLDCOVER:
        return {"source": "ndvi"}
    try:
        items = list(catalog.search(collections=["esa-worldcover"], bbox=BBOX,
                                    datetime="2021-01-01/2021-12-31").item_collection())
        if not items:
            raise RuntimeError("no WorldCover tiles found")
        lc = odc.stac.load(items, bands=["map"], crs=CRS, resolution=res, bbox=BBOX, groupby="solar_day",
                           resampling="mode" if res > 10 else "nearest", chunks={}).compute()["map"]
        if "time" in lc.dims:
            lc = lc.isel(time=0, drop=True)
        lc = lc.astype("float32").where(lc > 0)
        return {"source": "worldcover", "classes": lc, "natural": lc.isin([10, 20, 30]), "crop": lc == 40}
    except Exception as e:
        print(f"  WorldCover unavailable at {res} m ({type(e).__name__}: {e}); falling back to an NDVI-based mask")
        return {"source": "ndvi"}

def _mask2d(m, like):
    if m.shape != (like.sizes["y"], like.sizes["x"]):
        raise ValueError(f"land-cover grid {m.shape} does not match the image grid "
                         f"{(like.sizes['y'], like.sizes['x'])}; delete ndwi_project/processed and re-run")
    return xr.DataArray(np.asarray(m.values, bool), coords={"y": like.y, "x": like.x}, dims=("y", "x"))


def load_batch(items, sensor, res):
    """Load a handful of scenes -> masked index cube (float32), validity mask, platform list."""
    if sensor == "landsat":
        bands = ["green", "red", "nir08", "swir16", "qa_pixel"]
        resampling = "nearest"
    else:
        bands = ["B03", "B04", "B08", "B11", "SCL"]
        resampling = {"*": "nearest", "B11": "bilinear"} if res <= 10 else {"*": "average", "SCL": "nearest"}

    ds = odc.stac.load(items, bands=bands, crs=CRS, resolution=res, bbox=BBOX, groupby="solar_day",
                       chunks={"x": 1024, "y": 1024}, resampling=resampling, fail_on_error=False).compute()

    def dn(name):
        a = ds[name].astype("float32")
        return a.where(a > 0)                                   # DN 0 = no data

    if sensor == "landsat":
        # Collection-2 Level-2: reflectance = DN * 0.0000275 - 0.2 (the offset matters for ratios)
        sr = {k: (dn(b) * 0.0000275 - 0.2).clip(1e-4, 1.0)
              for k, b in dict(g="green", r="red", n="nir08", s="swir16").items()}
        qa = ds["qa_pixel"].fillna(1).astype("uint16")
        valid = (qa & 0b111111) == 0                            # fill, dilated cloud, cirrus, cloud, shadow, snow
        platform = nearest_item_attr(ds.time.values, items, lambda i: i.properties.get("platform", "landsat"))
    else:
        # Sentinel-2: reflectance = (DN - offset) / 10000, offset = 1000 for processing baseline >= 04.00
        off = xr.DataArray(nearest_item_attr(ds.time.values, items, s2_offset),
                           coords={"time": ds.time}, dims="time")
        sr = {k: ((dn(b) - off) / 1e4).clip(1e-4, 1.0)
              for k, b in dict(g="B03", r="B04", n="B08", s="B11").items()}
        valid = ds["SCL"].isin(S2_VALID_SCL)
        platform = nearest_item_attr(ds.time.values, items, lambda i: i.properties.get("platform", "sentinel-2"))

    for k in sr:
        valid = valid & sr[k].notnull()
    g, r, n, s = (sr[k] for k in "grns")
    idx = xr.Dataset({"ndwi_water": safe_nd(g, n),    # McFeeters: open water
                      "ndwi_moist": safe_nd(n, s),    # Gao / NDMI: vegetation moisture
                      "mndwi": safe_nd(g, s),         # modified NDWI, used as a robustness check
                      "ndvi": safe_nd(n, r)})
    valid = valid & idx.ndwi_water.notnull() & idx.ndwi_moist.notnull() & idx.ndvi.notnull()
    return idx.where(valid), valid, platform


def process_items(items, sensor, res):
    """All scenes of one season for the study box -> (seasonal composite Dataset, per-scene statistics)."""
    items = sorted(items, key=lambda i: _utc_naive(i.datetime))
    parts = [load_batch(items[b:b + BATCH_ITEMS], sensor, res) for b in range(0, len(items), BATCH_ITEMS)]
    idx = xr.concat([p[0] for p in parts], dim="time")
    valid = xr.concat([p[1] for p in parts], dim="time")
    platform = [x for p in parts for x in p[2]]
    px_ha = res * res / 1e4

    # ---------- seasonal composite ----------
    wf = (idx.ndwi_water > WATER_THR).astype("float32").where(valid)
    comp = xr.Dataset({
        "ndwi_water": idx.ndwi_water.median("time"),
        "ndwi_moist": idx.ndwi_moist.median("time"),
        "mndwi": idx.mndwi.median("time"),
        "ndvi": idx.ndvi.median("time"),
        "water_freq": wf.mean("time"),                         # share of clear looks classified as water
        "n_obs": valid.sum("time"),
    }).astype("float32")

    # ---------- per-scene statistics (fixed land-cover masks) ----------
    dims = ("y", "x"); total = valid.sizes["y"] * valid.sizes["x"]
    water = (idx.ndwi_water > WATER_THR) & valid
    lc = LCM.get(res, {"source": "ndvi"})
    if lc["source"] == "worldcover":
        nat = valid & ~water & _mask2d(lc["natural"], idx)
        crp = valid & ~water & _mask2d(lc["crop"], idx)
    else:                                                       # fallback: dynamic NDVI mask, no cropland split
        nat = valid & ~water & (idx.ndvi > NDVI_VEG_THR)
        crp = nat & False
    n_valid = valid.sum(dims).astype(float); nv = n_valid.where(n_valid > 0)
    n_nat = nat.sum(dims).astype(float)
    sc = pd.DataFrame({
        "date": pd.to_datetime(idx.time.values), "sensor": sensor, "platform": platform,
        "valid_frac": (n_valid / total).values,
        "water_frac_valid": (water.sum(dims) / nv).values,
        "nat_frac_valid": (n_nat / nv).values,
        "water_ndwi_mean": idx.ndwi_water.where(water).mean(dims).values,
        "moist_nat_mean": idx.ndwi_moist.where(nat).mean(dims).values,
        "ndvi_nat_mean": idx.ndvi.where(nat).mean(dims).values,
        "moist_crop_mean": idx.ndwi_moist.where(crp).mean(dims).values,
        "ndvi_crop_mean": idx.ndvi.where(crp).mean(dims).values,
    })
    sc["water_ha_equiv"] = sc.water_frac_valid * total * px_ha   # water area scaled to the full study box
    sc["year"] = sc.date.dt.year; sc["month"] = sc.date.dt.month
    return comp, sc


def _search_year(collection, y, months, max_cloud, extra_query):
    q = {"eo:cloud_cover": {"lt": max_cloud}, **(extra_query or {})}
    return list(catalog.search(collections=[collection], bbox=BBOX, datetime=season_range(y, months), query=q).item_collection())

def run_sensor(tag, sensor, collection, years, res, months, extra_query=None, max_cloud=MAX_TILE_CLOUD):
    """Search + download + process one sensor for all years of one season. Cached on disk by RUN_ID.
    v5: the cache is INCREMENTAL. If you extend the record (e.g. you change LANDSAT_YEARS in Cell 2), only the years that were never searched
    are downloaded and merged into the existing cache; the years you already have are not downloaded again.
    A year whose search failed (network error) is not written to the inventory, so it is retried on the next run."""
    base = f"{tag}_{RUN_ID}"
    nc = DIRS["processed"] / f"{base}_annual.nc"
    sc_csv = DIRS["tables"] / f"{base}_scenes_raw.csv"
    inv_csv = DIRS["raw_meta"] / f"{base}_inventory.csv"
    old_annual, old_sc, old_inv = None, None, pd.DataFrame(columns=["year", "n_items", "platforms", "median_tile_cloud"])
    if nc.exists() and sc_csv.exists() and inv_csv.exists() and not FORCE_REPROCESS:
        with xr.open_dataset(nc) as ds_: old_annual = ds_.load()
        old_sc = pd.read_csv(sc_csv, parse_dates=["date"]); old_inv = pd.read_csv(inv_csv)
    todo = [y for y in years if y not in set(old_inv.year.astype(int))]
    if old_annual is not None and not todo:
        print(f"[{tag}] loaded from cache ({int(old_inv.year.min())}-{int(old_inv.year.max())})")
        keep = [y for y in old_annual.year.values if y in set(years)]
        return {"annual": old_annual.sel(year=keep), "scenes_raw": old_sc[old_sc.year.isin(years)], "inv": old_inv[old_inv.year.isin(years)], "res": res, "months": months}
    if old_annual is not None: print(f"[{tag}] cache has {int(old_inv.year.min())}-{int(old_inv.year.max())}; downloading {len(todo)} missing years: {todo[0]}-{todo[-1]}")

    comps, scs, inv = [], [], []
    for y in todo:
        try:
            items = _search_year(collection, y, months, max_cloud, extra_query)
        except Exception as e:
            print(f"  [{tag} {y}] search failed ({type(e).__name__}); will retry next run"); continue
        clouds = [i.properties.get("eo:cloud_cover", np.nan) for i in items]
        inv.append({"year": y, "n_items": len(items),
                    "platforms": ",".join(sorted({i.properties.get("platform", "?") for i in items})),
                    "median_tile_cloud": float(np.nanmedian(clouds)) if clouds else np.nan})
        if not items:
            print(f"  [{tag} {y}] no scenes in the archive"); continue
        try:
            comp, sc = process_items(items, sensor, res)
        except Exception as e:
            inv.pop(); print(f"  [{tag} {y}] FAILED: {type(e).__name__}: {e} (will retry next run)"); continue
        comps.append(comp.expand_dims(year=[y])); scs.append(sc)
        print(f"  [{tag} {y}] {len(items):>2} items -> {len(sc):>2} dates | clear obs/pixel {float(comp.n_obs.mean()):.1f}")
    if old_annual is not None and comps:                        # same box, CRS and resolution -> identical grid; force exact coordinates before merging
        comps = [c.assign_coords(x=old_annual.x, y=old_annual.y) if c.sizes["x"] == old_annual.sizes["x"] and c.sizes["y"] == old_annual.sizes["y"] else c for c in comps]
    parts = ([old_annual] if old_annual is not None else []) + comps
    if not parts:
        raise RuntimeError(f"No data could be processed for {tag}")
    annual = xr.concat(parts, dim="year").sortby("year"); annual.attrs = {}
    scenes = pd.concat(([old_sc] if old_sc is not None else []) + scs, ignore_index=True).sort_values("date").reset_index(drop=True)
    inv = pd.concat([old_inv, pd.DataFrame(inv)], ignore_index=True).sort_values("year").reset_index(drop=True) if inv else old_inv
    if comps or len(inv) != len(old_inv):
        annual.to_netcdf(nc); scenes.to_csv(sc_csv, index=False); inv.to_csv(inv_csv, index=False)
        print(f"[{tag}] saved {nc.name} ({int(annual.year.min())}-{int(annual.year.max())}, {annual.sizes['year']} years with data)")
    keep = [y for y in annual.year.values if y in set(years)]
    return {"annual": annual.sel(year=keep), "scenes_raw": scenes[scenes.year.isin(years)], "inv": inv[inv.year.isin(years)], "res": res, "months": months}
print("pipeline ready")

## Step 0. Screening: do the pins sit on real water, and is every water body inside its box?
The gallery shows Sentinel-2 imagery of every site. **Check three things:** the cyan outline is the water body you expect; the pin (star) is on it; no red warning. If a site says ENLARGE BOX, raise its `half_deg` in Cell 2 and re-run Cells 2 and 7. Sites that say EXCLUDE are skipped until you correct the pin.

In [ ]:
# Cell 7: SITE SCREENING GALLERY. Real Sentinel-2 imagery of every site + automatic check that water exists where the pin is.
from pyproj import Transformer
from scipy import ndimage
_tr = Transformer.from_crs("EPSG:4326", CRS, always_xy=True)

def _covers(item, bb):
    from shapely.geometry import shape, box
    try: return shape(item.geometry).contains(box(*bb))
    except Exception: return True

def screen_site(sid, res=20):
    bb = set_site(sid); s = SITES[sid]
    for dates in (SCREEN_DATES, "2022-08-01/2025-10-31"):          # widen the search if the first window gives too little clear imagery
        items = list(catalog.search(collections=["sentinel-2-l2a"], bbox=bb, datetime=dates, query={"eo:cloud_cover": {"lt": 30}}).item_collection())
        items = [i for i in items if _utc_naive(i.datetime).month in (8, 9, 10)]
        full = [i for i in items if _covers(i, bb)]             # scenes that cover the WHOLE box (a tile edge gives black strips)
        items = sorted(full or items, key=lambda i: i.properties.get("eo:cloud_cover", 100))[:16]
        if not items:
            continue
        ds = odc.stac.load(items, bands=["B02", "B03", "B04", "B08", "SCL"], crs=CRS, resolution=res, bbox=bb, groupby="solar_day", chunks={},
                           resampling={"*": "average", "SCL": "nearest"}, fail_on_error=False).compute()
        valid = ds["SCL"].isin(S2_VALID_SCL)
        if float(valid.any("time").mean()) >= 0.9:
            break
    if not items:
        return None
    off = xr.DataArray(nearest_item_attr(ds.time.values, items, s2_offset), coords={"time": ds.time}, dims="time")
    refl = {k: ((ds[k].astype("float32").where(ds[k] > 0) - off) / 1e4).clip(0, 1) for k in ["B02", "B03", "B04", "B08"]}
    ndwi = safe_nd(refl["B03"], refl["B08"]).where(valid).median("time").values
    rgb = np.dstack([refl[k].where(valid).median("time").values for k in ["B04", "B03", "B02"]])
    clear_frac = float(np.isfinite(rgb).all(-1).mean())
    rgb = np.nan_to_num(np.clip(rgb / 0.25, 0, 1) ** (1 / 1.8), nan=1.0)    # no data shows white, not black
    wat = np.isfinite(ndwi) & (ndwi > WATER_THR)
    lab, n = ndimage.label(wat, structure=np.ones((3, 3)))
    x0, y0 = float(ds.x.min()), float(ds.y.min()); px, py = _tr.transform(s["lon"], s["lat"])
    out = dict(sid=sid, rgb=rgb, extent_km=[0, (float(ds.x.max()) - x0) / 1000, 0, (float(ds.y.max()) - y0) / 1000], clear_frac=clear_frac,
               pin_km=((px - x0) / 1000, (py - y0) / 1000), n_scenes=len(items), area_ha=0.0, dist_m=np.nan, touches_edge=False, comp=np.zeros_like(wat), suggest=None)
    if n == 0:
        return out
    sizes = ndimage.sum(wat, lab, index=range(1, n + 1)); big = int(np.argmax(sizes)) + 1          # largest water body in the box -> suggested pin
    cy, cx = ndimage.center_of_mass(lab == big); inv = Transformer.from_crs(CRS, "EPSG:4326", always_xy=True)
    sx, sy = float(ds.x.values[int(round(cx))]), float(ds.y.values[int(round(cy))]); slon, slat = inv.transform(sx, sy)
    out["suggest"] = (round(slat, 4), round(slon, 4), float(sizes[big - 1] * res * res / 1e4))
    ix = int(np.abs(ds.x.values - px).argmin()); iy = int(np.abs(ds.y.values - py).argmin())
    d, (jy, jx) = ndimage.distance_transform_edt(lab == 0, return_indices=True)
    comp = lab[jy[iy, ix], jx[iy, ix]]; mask = lab == comp; ys, xs = np.where(mask)
    out.update(comp=mask, area_ha=float(mask.sum() * res * res / 1e4), dist_m=float(d[iy, ix] * res),
               touches_edge=bool(ys.min() == 0 or xs.min() == 0 or ys.max() == mask.shape[0] - 1 or xs.max() == mask.shape[1] - 1))
    return out

SCREEN = {}
for sid in ACTIVE_SITES:
    try:
        SCREEN[sid] = screen_site(sid)
        r = SCREEN[sid]
        print(f"{SITES[sid]['name']:<30} " + ("no clear Sentinel-2 scene found" if r is None else f"water {r['area_ha']:.0f} ha | pin-to-water {r['dist_m']:.0f} m | touches box edge: {r['touches_edge']} | clear imagery {100 * r['clear_frac']:.0f}%" + (f" | largest water in box {r['suggest'][2]:.0f} ha at lat {r['suggest'][0]}, lon {r['suggest'][1]}" if r['suggest'] else "")))
    except Exception as e:
        SCREEN[sid] = None; print(f"{SITES[sid]['name']:<30} FAILED: {type(e).__name__}: {e}")

def verdict(sid):
    s, r = SITES[sid], SCREEN.get(sid)
    if r is not None and r["clear_frac"] < 0.8:
        return "NO CLEAR IMAGERY: re-run Cell 7 or widen SCREEN_DATES in Cell 2 (not a pin problem)"
    if r is None or r["area_ha"] == 0 or not np.isfinite(r["dist_m"]) or r["dist_m"] > MAX_PIN_DIST_M:
        return "EXCLUDE: no water found within 1 km of the pin (check the pin)"
    if r["touches_edge"] and s["kind"] != "river":
        return "ENLARGE BOX: water touches the edge (raise half_deg in Cell 2)"
    if s["kind"] == "sinkhole":
        return "REFERENCE: tiny stable lake, analysed with Sentinel-2 10 m only"
    if s["kind"] == "river":
        return "OK (river reach): trends need extra caution" if r["area_ha"] >= 10 else "WEAK: channel too narrow for 30 m pixels"
    if r["area_ha"] < MIN_AREA_TREND_HA:
        return "WEAK: under 20 ha, trends will be unreliable"
    return "OK for area trends"

scr = pd.DataFrame([{"site": sid, "name": SITES[sid]["name"], "town": SITES[sid]["town"], "kind": SITES[sid]["kind"],
                     "water_ha_recent": None if SCREEN[sid] is None else round(SCREEN[sid]["area_ha"]),
                     "pin_to_water_m": None if SCREEN[sid] is None else round(SCREEN[sid]["dist_m"]) if np.isfinite(SCREEN[sid]["dist_m"]) else None,
                     "verdict": verdict(sid)} for sid in ACTIVE_SITES])
display(scr); scr.to_csv(DIRS["tables"] / "site_screening.csv", index=False)
EXCLUDED = [r.site for r in scr.itertuples() if r.verdict.startswith(("EXCLUDE", "ENLARGE", "NO CLEAR"))]
print("\nSites that will be skipped until fixed:", EXCLUDED or "none")

shown = [s for s in ACTIVE_SITES if SCREEN.get(s) is not None]
ncol = 4; nrow = int(np.ceil(max(len(shown), 1) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4.6 * ncol, 4.5 * nrow), squeeze=False)
for ax in axes.ravel(): ax.axis("off")
for ax, sid in zip(axes.ravel(), shown):
    r = SCREEN[sid]; ax.axis("on"); ax.imshow(r["rgb"], extent=r["extent_km"], origin="upper")
    if r["comp"].any():
        ex = r["extent_km"]; ny, nx = r["comp"].shape
        ax.contour(np.linspace(ex[0], ex[1], nx), np.linspace(ex[3], ex[2], ny), r["comp"].astype(float), levels=[0.5], colors="#00e5ff", linewidths=1.3)
    ax.plot(*r["pin_km"], marker="*", ms=13, mfc="#ffd400", mec="black", ls="none")
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    ax.set_title(f"{SITES[sid]['name']}  ({SITES[sid]['town']})\n{r['area_ha']:.0f} ha of water | {SITES[sid]['kind'].replace('_', ' ')}", fontsize=9.5, loc="left")
    if r["touches_edge"] and SITES[sid]["kind"] != "river":
        ax.text(0.5, 0.03, "water touches box edge: enlarge", transform=ax.transAxes, ha="center", color="red", fontsize=8.5, bbox=dict(fc="white", alpha=0.85, pad=1.5))
hs = [Line2D([0], [0], color="#00e5ff", lw=2, label="Detected water body (NDWI > 0)"), Line2D([0], [0], marker="*", color="w", mfc="#ffd400", mec="black", ms=13, label="Site pin")]
finish(fig, "02_study_sites", "The Copperbelt water bodies we will study, as seen from space",
       "Sentinel-2 true-colour images, Aug-Oct 2024/25 (end of dry season). Each box is the analysis area for that site. The cyan outline is the water body the code will track.",
       source="Sentinel-2 L2A (20 m preview), Microsoft Planetary Computer", handles=hs, legend_cols=2)


## Step 1. Collect data for every site
Landsat 5/7/8/9, 1996-2025 (30 years, two seasons), plus Sentinel-2 2015-2025 at 30 m for validation. Each site is cached separately; one failing site does not stop the others. Your existing cache already covers 1996-2025, so nothing is downloaded again.

In [ ]:
# Cell 8: DATA COLLECTION FOR EVERY SITE (Landsat 1996-2025 + Sentinel-2 2015-2025 validation; sinkholes use Sentinel-2 10 m only). Cached on disk per site.
SITE_DATA = {}
RUN_SITES = [s for s in ACTIVE_SITES if s not in EXCLUDED]
for sid in RUN_SITES:
    s = SITES[sid]; set_site(sid)
    print(f"\n########## {s['name']} ({s['town']}) | box {BBOX} | cache {RUN_ID} ##########")
    try:
        LCM.clear()
        sens = SENSORS_BY_KIND[s["kind"]]
        for res in ([30] if "landsat" in sens else []) + ([S2_FINE_RES] if "s2fine" in sens else []):
            LCM[res] = load_landcover(res)
        D = {}
        if "landsat" in sens:
            for sk, months in SEASONS.items():
                D[("landsat", sk)] = run_sensor(f"landsat30_{sk}", "landsat", "landsat-c2-l2", LANDSAT_YEARS, 30, months, extra_query={"platform": {"in": LANDSAT_PLATFORMS}})
            for sk, months in SEASONS.items():
                D[("s2_30", sk)] = run_sensor(f"s2_30m_{sk}", "s2", "sentinel-2-l2a", S2_YEARS, 30, months)
        if "s2fine" in sens:
            for sk, months in SEASONS.items():
                D[("s2_fine", sk)] = run_sensor(f"s2_{S2_FINE_RES}m_{sk}", "s2", "sentinel-2-l2a", S2_YEARS, S2_FINE_RES, months)
        SITE_DATA[sid] = {"D": D, "LCM": copy.copy(LCM), "bbox": list(BBOX)}
    except Exception as e:
        print(f"  !! {s['name']} FAILED: {type(e).__name__}: {e}  (the other sites continue)")
print("\nCollected:", list(SITE_DATA))


## Step 2. Analyse each water body on its own
Footprint, yearly water cover, shoreline vegetation, trend tests (Mann-Kendall, Sen slope, Landsat-generation check), Landsat vs Sentinel-2 validation and a reliability flag.

In [ ]:
# Cell 9: PER-SITE ANALYSIS. For each water body: find its footprint, measure water and shoreline vegetation every year, test trends, check reliability.
METRICS_SITE = {"water_ha": "Water-covered area (ha)",
                "water_pct_foot": "Water share of the usual footprint (%)",
                "ring_moist_mean": "Shoreline vegetation water content (NDWI)",
                "ring_ndvi_mean": "Shoreline vegetation greenness (NDVI)",
                "stress_pct_z": "Shoreline vegetation more than 1 SD drier than its own normal (% of pixels)"}

def clean_scenes(sc):
    sc = sc.copy()
    sc["water_rz"] = sc.groupby("year")["water_frac_valid"].transform(robust_z)
    sc["ok_valid"] = sc.valid_frac >= MIN_VALID_FRAC_SCENE; sc["outlier"] = sc.water_rz.abs() > 3.5; sc["scene_ok"] = sc.ok_valid & ~sc.outlier
    return sc

def year_qc(annual, scenes_c):
    cov = (annual.n_obs > 0).mean(("y", "x")).to_series().rename("coverage")
    mean_obs = annual.n_obs.where(annual.n_obs > 0).mean(("y", "x")).to_series().rename("mean_clear_obs_per_px")
    nsc = scenes_c[scenes_c.scene_ok].groupby("year").size().rename("n_scenes_usable")
    qc = pd.concat([cov, mean_obs, nsc], axis=1).fillna({"n_scenes_usable": 0}).reset_index()
    qc["ok"] = (qc.coverage >= MIN_COVERAGE_YEAR) & (qc.n_scenes_usable >= MIN_SCENES_YEAR)
    return qc

def build_footprint(A, res, lon, lat, mode="occ"):
    """The water body nearest the pin: connected pixels that were water in >= FOOT_OCC of usable years (or in the latest years for a new dam)."""
    wm = ((A.water_freq >= 0.5) & (A.n_obs > 0)).astype("float32")
    occ = wm.mean("year").values
    cand = (wm.isel(year=slice(-3, None)).mean("year").values >= 0.5) if mode == "recent" else (occ >= FOOT_OCC)
    lab, n = ndimage.label(cand, structure=np.ones((3, 3)))
    if n == 0:
        return None
    px, py = _tr.transform(lon, lat)
    ix = int(np.abs(A.x.values - px).argmin()); iy = int(np.abs(A.y.values - py).argmin())
    d, (jy, jx) = ndimage.distance_transform_edt(lab == 0, return_indices=True)
    comp = lab[jy[iy, ix], jx[iy, ix]]; foot = lab == comp
    dist_f = ndimage.distance_transform_edt(~foot) * res
    ring = (dist_f > 0) & (dist_f <= RING_M) & (lab == 0)
    ys, xs = np.where(foot)
    return dict(foot=foot, ring=ring, occ=occ, dist_m=float(d[iy, ix] * res), area_ha=float(foot.sum() * res * res / 1e4),
                touches_edge=bool(ys.min() == 0 or xs.min() == 0 or ys.max() == foot.shape[0] - 1 or xs.max() == foot.shape[1] - 1))

def site_stats(annual, res, qc, tag, sk, fp, lc):
    px_ha = res * res / 1e4; ok_years = qc.loc[qc.ok, "year"].values; foot, ring = fp["foot"], fp["ring"]
    if lc and lc.get("source") == "worldcover":
        nat = _mask2d(lc["natural"], annual).values
    else:
        nat = ((annual.ndvi.sel(year=ok_years) > NDVI_VEG_THR).mean("year") >= 0.5).values
    use_ring = (ring & nat) if (ring & nat).sum() >= 15 else ring
    nfoot = max(int(foot.sum()), 1); rows = []
    if len(ok_years) >= 3:                                      # pixel z-score of vegetation moisture against that pixel's own record (as in v2 Cell 11)
        base_ = annual["ndwi_moist"].sel(year=ok_years); sd_ = base_.std("year", ddof=1)
        zm_all = ((annual["ndwi_moist"] - base_.mean("year")) / sd_.where(sd_ > 1e-6)).values
    else:
        zm_all = np.full(annual["ndwi_moist"].shape, np.nan)
    for iy_, y in enumerate(annual.year.values):
        a = annual.sel(year=y); obs = (a.n_obs > 0).values
        cov = float((obs & foot).sum()) / nfoot
        wat = (a.water_freq.values >= 0.5) & obs & foot; ro = use_ring & obs
        rows.append({"year": int(y), "sensor": tag, "season": sk, "foot_cov": cov, "water_ha": wat.sum() * px_ha, "water_pct_foot": 100 * wat.sum() / nfoot,
                     "ring_moist_mean": float(np.nanmean(a.ndwi_moist.values[ro])) if ro.sum() >= 5 else np.nan,
                     "ring_ndvi_mean": float(np.nanmean(a.ndvi.values[ro])) if ro.sum() >= 5 else np.nan,
                     "stress_pct_z": 100 * float(((zm_all[iy_] < STRESS_Z) & ro).sum()) / max(int(ro.sum()), 1) if ro.sum() >= 5 else np.nan})
    df = pd.DataFrame(rows).merge(qc[["year", "coverage", "n_scenes_usable", "ok"]], on="year", how="left")
    df["ok"] = df["ok"].fillna(False).astype(bool); df["ok_site"] = df.ok & (df.foot_cov >= MIN_FOOT_COV)
    df.loc[df.foot_cov < MIN_FOOT_COV, ["water_ha", "water_pct_foot"]] = np.nan
    for m in METRICS_SITE:
        d = df.loc[df.ok_site, m]; sd = d.std(ddof=1)
        df[m + "_z"] = (df[m] - d.mean()) / sd if (len(d) > 2 and sd and np.isfinite(sd) and sd > 0) else np.nan
    return df

def pct_row(tt, m):
    if tt is None or not len(tt): return None
    r = tt[tt.metric == m]
    if not len(r): return None
    r = r.iloc[0].copy(); mean = r["mean"]
    r["pct_dec"] = 1000 * r.sen_slope_per_year / abs(mean) if mean else np.nan
    r["ci_lo"] = 1000 * r.sen_ci_low / abs(mean) if mean else np.nan; r["ci_hi"] = 1000 * r.sen_ci_high / abs(mean) if mean else np.nan
    return r

def harden_trends(tt):
    """Stricter verdicts: significance from the Hamed-Rao test (allows for year-to-year memory) and, for water area, the change must also be big enough to matter."""
    if tt is None or not len(tt): return tt
    tt = tt.copy(); tt["mk_p_original"] = tt["mk_p"]; tt["mk_trend_original"] = tt["mk_trend"]; tt["note"] = ""
    for i, r in tt.iterrows():
        p = r["mk_p_hamed_rao"] if (USE_HAMED_RAO and np.isfinite(r["mk_p_hamed_rao"])) else r["mk_p"]
        tr = ("increasing" if r["sen_slope_per_year"] > 0 else "decreasing") if p < 0.05 else "no trend"; note = ""
        if r["metric"] in ("water_ha", "water_pct_foot") and r["mean"]:
            pct = 1000 * r["sen_slope_per_year"] / abs(r["mean"])
            if tr != "no trend" and abs(pct) < MIN_EFFECT_PCT_DEC:
                note = f"statistically detectable (p={p:.3f}) but only {pct:+.1f}% per decade, below the {MIN_EFFECT_PCT_DEC}% we treat as meaningful"; tr = "no trend"
        if r["mk_trend_original"] != "no trend" and p >= 0.05 and not note:
            note = f"plain Mann-Kendall said significant (p={r['mk_p_original']:.3f}) but not after allowing for year-to-year memory (p={p:.3f})"
        tt.at[i, "mk_p"] = p; tt.at[i, "mk_trend"] = tr; tt.at[i, "note"] = note
    return tt

def reliability(kind, foot_ha, agree_z, agree_raw, sub, edge=False):
    why = []; severe = False
    if edge and kind != "river": why.append("the water body reaches the edge of the analysis box, so its area is only partly measured (raise half_deg in Cell 2)")
    if kind == "river": why.append("river reach: narrow channel, mixed pixels at 30 m")
    if foot_ha < MIN_AREA_TREND_HA: why.append(f"small water body ({foot_ha:.0f} ha < {MIN_AREA_TREND_HA} ha)")
    if agree_z is not None and agree_z.get("n", 0) >= 6:
        r = agree_z.get("pearson_r", np.nan)
        if not np.isfinite(r) or r < 0.6: why.append(f"Landsat and Sentinel-2 disagree on year-to-year change (r={r:.2f})")
        if not np.isfinite(r) or r < 0.3: severe = True
        mean_s2 = agree_raw.get("_mean_s2", np.nan)
        if np.isfinite(mean_s2) and mean_s2 > 0 and abs(agree_raw.get("bias", 0)) / mean_s2 > 0.25:
            why.append(f"Landsat area differs from Sentinel-2 by {100 * abs(agree_raw['bias']) / mean_s2:.0f}%")
    elif agree_z is not None:
        why.append("too few Landsat/Sentinel-2 overlap years to validate")
    if sub is not None and len(sub) == 2:
        s1, s2 = sub.sen_slope_per_year.values; p1, p2 = sub.mk_p.values
        if s1 * s2 < 0 and min(p1, p2) < 0.1: why.append("trend direction differs between Landsat generations (sensor artefact risk)"); severe = True
    label = "High" if not why else ("Low" if (severe or len(why) >= 2) else "Medium")
    return label, why

LC_NAMES = {10: "Tree cover", 20: "Shrubland", 30: "Grassland", 40: "Cropland", 50: "Built-up", 60: "Bare / sparse", 80: "Water", 90: "Herbaceous wetland"}
LC_COLORS = {10: "#006400", 20: "#ffbb22", 30: "#ffff4c", 40: "#f096ff", 50: "#fa0000", 60: "#b4b4b4", 80: "#0064c8", 90: "#0096a0"}

def landuse_around(sid, r, res=30):
    """What is around the lake, and do the natural vegetation and the farmland there behave differently? (the v3 land-cover analysis, one lake at a time)"""
    SD = SITE_DATA[sid]; lc = SD["LCM"].get(res); D = SD["D"]; fp = r["fp"]
    if not lc or lc.get("source") != "worldcover" or ("landsat", VEG_SEASON) not in D: return None
    d = D[("landsat", VEG_SEASON)]; okY = d["qc"].loc[d["qc"].ok, "year"].values
    if len(okY) < 8: return None
    B = d["annual"].sel(year=okY)
    zone = (ndimage.distance_transform_edt(~fp["foot"]) * res <= RING_WIDE_M) & ~fp["foot"]
    cls = np.asarray(lc["classes"].values, float); nat = _mask2d(lc["natural"], B).values; crop = _mask2d(lc["crop"], B).values
    tot = max(int((zone & np.isfinite(cls)).sum()), 1)
    share = {LC_NAMES[k]: 100 * int(((cls == k) & zone).sum()) / tot for k in LC_NAMES if int(((cls == k) & zone).sum()) > 0}
    rows = []
    for var in ["ndvi", "ndwi_moist"]:
        sl, p = pixel_trend(B[var].values, B.year.values); p = bh_fdr(p)          # FDR across all pixels: thousands of tests at once
        for cname, m in {"Natural vegetation": nat, "Cropland": crop}.items():
            sel = m & zone & np.isfinite(sl); n = int(sel.sum())
            if n < 10: continue
            rows.append({"site": sid, "variable": var, "land_cover": cname, "n_pixels": n,
                         "pct_greening_or_wetter": 100 * int(((sl > 0) & (p < 0.05) & sel).sum()) / n,
                         "pct_browning_or_drier": 100 * int(((sl < 0) & (p < 0.05) & sel).sum()) / n,
                         "mean_change_per_decade": 10 * float(np.nanmean(sl[sel]))})
    return dict(zone=zone, cls=cls, share=share, table=pd.DataFrame(rows), x=B.x.values, y=B.y.values)

# ---------------- v5: sensor harmonisation of the vegetation indices ----------------
# Landsat 8/9 OLI has a narrower NIR band than Landsat 5/7 TM/ETM+, so it reads NDVI and canopy NDWI systematically HIGHER for the same canopy
# (Roy et al. 2016, Remote Sens. Environ. 185). A trend fitted across 2013 then shows "greening / wetting" that is only the sensor switch.
# Fix (relative cross-calibration): in years when BOTH families observed the same season, the median OLI minus TM/ETM+ difference is the offset;
# every TM/ETM+ contribution is moved onto the OLI scale. Composites mix both families after 2013, so each year's composite is shifted by
# offset x (share of that year's clear looks that came from TM/ETM+). Water indices are not touched (their offset is ~0, see Cell 9b).
# Second correction: canopy NDWI falls ~0.08 per month from March to May, so a year whose clear images happen to be late in the season reads 'drier'.
# Each composite is moved to the record's mean image date with the within-season slope (fitted with a sensor dummy, so the two corrections do not mix).
FAMILY = lambda p: "OLI (L8/L9)" if p in ("landsat-8", "landsat-9") else "TM/ETM+ (L5/L7)"
HARM_VARS = {"ndvi": "ndvi_nat_mean", "ndwi_moist": "moist_nat_mean"}     # composite variable -> scene-level metric used to estimate the offset
HARM_ROWS = []

def _clean_for_offset(scenes):
    return scenes[scenes.scene_ok & ~((scenes.platform == "landsat-7") & (scenes.year >= L7_DRIFT_YEAR))].copy()

def season_slope(scenes, metric):
    """Within-season change per day (e.g. canopy NDWI dries ~0.08 per month from March to May), fitted jointly with a sensor-family dummy so the
    sensor offset does not leak into the slope. Needed because each year's composite is built from whatever dates happened to be cloud-free."""
    s = _clean_for_offset(scenes).dropna(subset=[metric]); s["oli"] = (s.platform.map(FAMILY) == "OLI (L8/L9)").astype(float)
    if len(s) < 20: return 0.0
    X = np.c_[np.ones(len(s)), s.date.dt.dayofyear.values, s.oli.values]; coef = np.linalg.lstsq(X, s[metric].values, rcond=None)[0]
    return float(coef[1])

def family_offset(scenes, metric, slope=0.0):
    """Sensor offset = coefficient of an OLI dummy in  metric ~ 1 + image date + OLI,  fitted ONLY on the overlap years when both families flew.
    Same years (so no climate trend leaks in) and image date in the model (so a different date mix of the two families is not mistaken for an offset)."""
    s = _clean_for_offset(scenes).dropna(subset=[metric]); s = s[(s.year >= HARM_YEARS[0]) & (s.year <= HARM_YEARS[1])]
    s["oli"] = (s.platform.map(FAMILY) == "OLI (L8/L9)").astype(float)
    yrs_both = s.groupby("year").oli.agg(lambda v: 0 < v.mean() < 1); n = int(yrs_both.sum())
    if n < 4 or s.oli.nunique() < 2: return np.nan, n
    X = np.c_[np.ones(len(s)), s.date.dt.dayofyear.values, s.oli.values]
    return float(np.linalg.lstsq(X, s[metric].values, rcond=None)[0][2]), n

def harmonise_landsat(d, sid, sk):
    """Idempotent: always starts from the raw cached composite / raw scene table.
    Composite(year) is moved by  + offset x share_of_TM/ETM+_looks(year)  - season_slope x (mean look date(year) - mean look date(all years))."""
    d.setdefault("annual_raw", d["annual"]); A = d["annual_raw"].copy()
    sc = d["scenes"]; use = sc[sc.valid_frac > 0].copy(); use["etm"] = (use.platform.map(FAMILY) == "TM/ETM+ (L5/L7)").astype(float); use["doy"] = use.date.dt.dayofyear
    w = use.groupby("year").apply(lambda g: np.average(g.etm, weights=g.valid_frac) if g.valid_frac.sum() > 0 else 1.0)
    dy = use.groupby("year").apply(lambda g: np.average(g.doy, weights=g.valid_frac) if g.valid_frac.sum() > 0 else np.nan)
    etm_share = xr.DataArray(w.reindex(A.year.values).fillna(1.0).values, coords={"year": A.year}, dims="year")
    doy_y = dy.reindex(A.year.values); doy_dev = xr.DataArray((doy_y - doy_y.mean()).fillna(0.0).values, coords={"year": A.year}, dims="year")
    for var, met in HARM_VARS.items():
        b_ = season_slope(sc, met); off, n = family_offset(sc, met, b_); off_used = off if np.isfinite(off) else 0.0
        A[var] = A[var] + off_used * etm_share - b_ * doy_dev
        sc[met + "_raw"] = sc[met]; sc[met] = sc[met] + np.where(sc.platform.map(FAMILY) == "TM/ETM+ (L5/L7)", off_used, 0.0)
        HARM_ROWS.append({"site": sid, "season": sk, "variable": var, "offset_OLI_minus_TM_ETM": off, "overlap_years": n, "season_slope_per_30d": 30 * b_,
                          "max_date_shift_days": float(np.nanmax(np.abs(doy_y - doy_y.mean()))), "applied": bool(np.isfinite(off))})
    d["annual"] = A; d["etm_share"] = etm_share.to_series(); d["doy_mean"] = doy_y

def physics_qc(st_mam, st_dry):
    """A reservoir cannot gain water between the end of the rains (Mar-May) and the end of the dry season (Aug-Oct): there is almost no rain from June to October.
    If the end-of-rains water cover is more than MAX_REFILL_PTS below the same year's end-of-dry-season cover, the end-of-rains value is a detection failure
    (haze/thin cloud the QA mask missed, sediment-laden or plant-covered water, which push McFeeters NDWI below 0). That value is removed, not 'corrected'."""
    st = st_mam.copy(); dry = st_dry.set_index("year")["water_pct_foot"]
    d = st.year.map(dry) - st.water_pct_foot
    st["phys_fail"] = (d > MAX_REFILL_PTS).fillna(False)
    st.loc[st.phys_fail, ["water_ha", "water_pct_foot"]] = np.nan
    for m in ["water_ha", "water_pct_foot"]:
        dd = st.loc[st.ok_site, m]; sd = dd.std(ddof=1)
        st[m + "_z"] = (st[m] - dd.mean()) / sd if (dd.notna().sum() > 2 and sd and np.isfinite(sd) and sd > 0) else np.nan
    return st

RES = {}; HARM_ROWS.clear()
def analyse_site(sid):
    SD = SITE_DATA[sid]; D = SD["D"]; s = SITES[sid]; out = {"sid": sid, "stats": {}, "trend": {}, "sub": {}, "agree": {}}
    prim = "landsat" if ("landsat", "dry") in D else "s2_fine"; res_p = 30 if prim == "landsat" else S2_FINE_RES
    out["prim"] = prim
    for k, d in D.items():
        d["scenes"] = clean_scenes(d["scenes_raw"]); d["qc"] = year_qc(d.get("annual_raw", d["annual"]), d["scenes"])
        if HARMONISE_VEG and k[0] == "landsat": harmonise_landsat(d, sid, k[1])
    qc = D[(prim, "dry")]["qc"]; okY = qc.loc[qc.ok, "year"].values
    if len(okY) < 5:
        out["error"] = f"only {len(okY)} usable dry-season years"; return out
    A = D[(prim, "dry")]["annual"].sel(year=okY)
    fp = build_footprint(A, res_p, s["lon"], s["lat"], "recent" if s["kind"] == "new_reservoir" else "occ")
    if fp is None:
        out["error"] = "no water detected near the pin"; return out
    out["fp"] = fp; out["xy"] = (A.x.values, A.y.values)
    for (tag, sk), d in D.items():
        res_t = {"landsat": 30, "s2_30": 30, "s2_fine": S2_FINE_RES}[tag]
        same = (d["annual"].sizes["y"], d["annual"].sizes["x"]) == fp["foot"].shape
        fpt = fp if same else build_footprint(d["annual"].sel(year=d["qc"].loc[d["qc"].ok, "year"].values), res_t, s["lon"], s["lat"])
        if fpt is None: continue
        out["stats"][(tag, sk)] = site_stats(d["annual"], res_t, d["qc"], tag, sk, fpt, SD["LCM"].get(res_t))
    out["mam_fail"] = {}
    for tag in ("landsat", "s2_30", "s2_fine"):
        a, b = out["stats"].get((tag, "mam")), out["stats"].get((tag, "dry"))
        if a is not None and b is not None:
            out["stats"][(tag, "mam")] = physics_qc(a, b); q_ = out["stats"][(tag, "mam")]
            out["mam_fail"][tag] = (int(q_.phys_fail[q_.ok_site].sum()), int(q_.ok_site.sum()))
    for sk in SEASONS:
        st = out["stats"].get((prim, sk))
        if st is None: continue
        ok = st[st.ok_site]
        if s["kind"] != "new_reservoir":
            out["trend"][sk] = harden_trends(trend_table(ok, METRICS_SITE, min_n=8))
            if prim == "landsat":
                out["sub"][sk] = pd.concat([trend_table(g, {"water_ha": "w"}, min_n=6).assign(era=e) for e, g in
                                            {"TM/ETM+": ok[ok.year < SENSOR_BREAK], "OLI": ok[ok.year >= SENSOR_BREAK]}.items()], ignore_index=True)
    out["agree_z"], out["agree_raw"] = None, None
    if prim == "landsat":
        for sk in SEASONS:
            L, S = out["stats"].get(("landsat", sk)), out["stats"].get(("s2_30", sk))
            if L is None or S is None: continue
            ov = sorted(set(L[L.ok_site].year) & set(S[S.ok_site].year))
            if len(ov) >= 3:
                Lv, Sv = L.set_index("year").loc[ov, "water_ha"].values, S.set_index("year").loc[ov, "water_ha"].values
                ar = agreement(Sv, Lv); ar["_mean_s2"] = float(np.nanmean(Sv)); az = agreement(zscore(Sv).values, zscore(Lv).values)
                out["agree"][sk] = {"raw": ar, "z": az, "years": ov}
        if WATER_SEASON in out["agree"]:
            out["agree_z"], out["agree_raw"] = out["agree"][WATER_SEASON]["z"], out["agree"][WATER_SEASON]["raw"]
        else:
            out["agree_z"] = {"n": 0}; out["agree_raw"] = {}
        # three-decade change map, end of dry season: a pixel is "water in a decade" if it was water in at least half of that decade's usable years
        wmask = ((A.water_freq >= 0.5)).astype("float32").where(A.n_obs > 0)
        dec_n = [int(wmask.sel(year=slice(*w)).sizes["year"]) for w in DECADES]
        out["change_n"] = dict(zip(DECADE_LABEL, dec_n))               # usable years per decade (the map needs >= 3 in every decade)
        if min(dec_n) >= 3:
            occ_d = [wmask.sel(year=slice(*w)).mean("year") for w in DECADES]; ok_all = occ_d[0].notnull() & occ_d[1].notnull() & occ_d[2].notnull()
            w1, w2, w3 = [(o >= 0.5) for o in occ_d]
            # 0 never water | 1 water in all 3 decades | 2 lost by 2006-2015 | 3 lost by 2016-2025 | 4 new by 2006-2015 | 5 new by 2016-2025 | 6 intermittent (on-off-on / off-on-off)
            cat = xr.where(w1 & w2 & w3, 1, xr.where(w1 & ~w2 & ~w3, 2, xr.where(w1 & w2 & ~w3, 3, xr.where(~w1 & w2 & w3, 4, xr.where(~w1 & ~w2 & w3, 5, xr.where(w1 | w2 | w3, 6, 0))))))
            out["change"] = cat.where(ok_all)
            # hectares are counted for THIS dam only: its usual footprint plus RING_M around it (so growth beyond the footprint is still counted,
            # but other ponds and tailings dams in the analysis box are not)
            dam_zone = xr.DataArray(ndimage.distance_transform_edt(~fp["foot"]) * res_p <= RING_M, coords={"y": A.y, "x": A.x}, dims=("y", "x"))
            out["dam_zone"] = dam_zone
            out["decade_water_ha"] = dict(zip(DECADE_LABEL, [float(((o >= 0.5) & ok_all & dam_zone).sum()) * 0.09 for o in occ_d]))
        else:
            out["change"] = None
    else:
        out["change"] = None
    try: out["landuse"] = landuse_around(sid, out, 30) if prim == "landsat" else None
    except Exception as e: out["landuse"] = None; print(f"  (land-cover split skipped for {sid}: {type(e).__name__}: {e})")
    out["occ_da"] = xr.DataArray(fp["occ"], coords={"y": A.y, "x": A.x}, dims=("y", "x"))
    out["foot_da"] = xr.DataArray(fp["foot"].astype("float32"), coords={"y": A.y, "x": A.x}, dims=("y", "x"))
    sub_w = out["sub"].get(WATER_SEASON); sub_w = sub_w[sub_w.metric == "water_ha"] if sub_w is not None and len(sub_w) else None
    out["rel"], out["why"] = reliability(s["kind"], fp["area_ha"], out["agree_z"] if prim == "landsat" else None, out["agree_raw"] if prim == "landsat" else None, sub_w, edge=fp["touches_edge"])
    if s["kind"] == "sinkhole" and fp["area_ha"] < MIN_AREA_TREND_HA:
        out["rel"], out["why"] = "Reference", ["tiny stable lake: used to test for sensor drift, not for area trends"]
    return out

for sid in SITE_DATA:
    try:
        RES[sid] = analyse_site(sid)
        r = RES[sid]
        if not r.get("error") and r["fp"]["touches_edge"] and SITES[sid]["kind"] != "river": print(f"  WARNING {SITES[sid]['name']}: the water body touches the box edge, enlarge half_deg in Cell 2 and re-run")
        print(f"{SITES[sid]['name']:<30}", r.get("error") or f"footprint {r['fp']['area_ha']:.0f} ha | reliability: {r['rel']}" + (f" ({'; '.join(r['why'])})" if r["why"] else ""))
        if not r.get("error") and r.get("mam_fail", {}).get(r["prim"]):
            nf, nt = r["mam_fail"][r["prim"]]
            if nf: print(f"     physics check: {nf} of {nt} end-of-rains water values were BELOW the following dry season by >{MAX_REFILL_PTS} points (impossible without inflow) and were removed")
        for sk, tt in r.get("trend", {}).items():
            x = pct_row(tt, "water_ha")
            if x is not None:
                print(f"     {season_label(sk):<28} water area: {phrase_p(x.mk_p, x.mk_trend)}; {x.pct_dec:+.1f}% per decade ({span_label(x)})")
                if x.note: print(f"        note: {x.note}")
    except Exception as e:
        RES[sid] = {"sid": sid, "error": f"{type(e).__name__}: {e}"}; print(f"{SITES[sid]['name']:<30} FAILED: {RES[sid]['error']}")

# Multiple-testing check: ten water-cover tests (5 dams x 2 seasons) are run, so a few "significant" results are expected by chance. Benjamini-Hochberg false-discovery-rate adjustment:
_fd = [(sid, sk, float(pct_row(tt, "water_pct_foot").mk_p)) for sid, r in RES.items() if not r.get("error") for sk, tt in r.get("trend", {}).items() if pct_row(tt, "water_pct_foot") is not None]
FDR_LINE = None
if _fd:
    ps = np.array([p for _, _, p in _fd]); order = np.argsort(ps); m_ = len(ps); q = np.empty(m_); run_ = 1.0
    for rank, idx in zip(range(m_, 0, -1), order[::-1]):
        run_ = min(run_, ps[idx] * m_ / rank); q[idx] = run_
    nraw = int((ps < 0.05).sum()); nfdr = int((q < 0.05).sum())
    for (sid, sk, p), qq in zip(_fd, q):
        tt = RES[sid]["trend"][sk]; tt.loc[tt.metric == "water_pct_foot", "p_fdr"] = qq
    FDR_LINE = f"Multiple-testing check (water-cover share of footprint): {nraw} of {m_} water-cover tests are significant at p<0.05; {nfdr} survive a false-discovery-rate correction (with {m_} tests, about {0.05 * m_:.1f} would be expected by chance alone)."
    print("\n" + FDR_LINE)

HARM_TAB = pd.DataFrame(HARM_ROWS)
if len(HARM_TAB):
    print("\nSensor harmonisation (added to every TM/ETM+ look so all years are on the Landsat 8/9 OLI scale):")
    display(HARM_TAB.round(4))


## Step 2a. Data availability 1996-2025: which years exist, and why some are dropped
Every dam is analysed over the same 30-year period, 1996-2025. This figure (`01_data_availability`) shows, for each dam, season and year, whether Landsat images exist and whether the year passed the quality rules. It is the evidence for the analysis period: 1996 is the first year with images at every dam, and where a dam's series starts later inside the period, the figure shows why.

In [ ]:
# Cell 9a: DATA AVAILABILITY over the analysis period (LANDSAT_YEARS = 1996-2025). One row per dam and season; one column per year. Explains every year that is missing from a trend.
AV_CAT = {0: ("Search not run / failed (re-run Cell 8)", "#ffffff"), 1: ("No Landsat image in the archive", "#d9d9d9"), 2: (f"Fewer than {MIN_SCENES_YEAR} clear images", "#fdd0a2"),
          3: (f"Less than {int(100 * MIN_COVERAGE_YEAR)}% of the box or {int(100 * MIN_FOOT_COV)}% of the lake seen clearly", "#fd8d3c"), 4: ("Usable", "#2166ac"), 5: ("Usable, but end-of-rains water failed the physics check", "#9ecae1")}
AV_ROWS = []
for sid, r in RES.items():
    if r.get("error") or r.get("prim") != "landsat": continue
    for sk in SEASONS:
        d = SITE_DATA[sid]["D"].get(("landsat", sk)); st = r["stats"].get(("landsat", sk))
        if d is None or st is None: continue
        inv = d["inv"].set_index("year"); qc = d["qc"].set_index("year"); st_ = st.set_index("year")
        for y in LANDSAT_YEARS:
            if y not in inv.index: c = 0
            elif inv.loc[y, "n_items"] == 0 or y not in qc.index: c = 1
            elif qc.loc[y, "n_scenes_usable"] < MIN_SCENES_YEAR: c = 2
            elif not bool(st_.loc[y, "ok_site"]) if y in st_.index else True: c = 3
            elif "phys_fail" in st_ and bool(st_.loc[y, "phys_fail"]): c = 5
            else: c = 4
            AV_ROWS.append({"site": sid, "season": sk, "year": y, "category": c, "reason": AV_CAT[c][0],
                            "n_archive_images": int(inv.loc[y, "n_items"]) if y in inv.index else np.nan,
                            "n_clear_images": int(qc.loc[y, "n_scenes_usable"]) if y in qc.index else 0,
                            "platforms": inv.loc[y, "platforms"] if y in inv.index else ""})
AV = pd.DataFrame(AV_ROWS)
if len(AV):
    first = AV[AV.category >= 4].groupby(["site", "season"]).year.agg(["min", "max", "count"]).rename(columns={"min": "first_usable", "max": "last_usable", "count": "usable_years"})
    first["years_in_period"] = len(LANDSAT_YEARS)
    first = first.join(AV[AV.category == 1].groupby(["site", "season"]).size().rename("years_no_image")).join(AV[AV.category == 2].groupby(["site", "season"]).size().rename("years_too_few_clear")).fillna(0)
    print(f"Analysis period for every site: {LANDSAT_YEARS[0]}-{LANDSAT_YEARS[-1]} ({len(LANDSAT_YEARS)} years). First / last usable year and why other years are missing:")
    display(first.astype({c: int for c in first.columns}))
    nf = AV[AV.category == 0]
    if len(nf): print(f"\nWARNING: {nf[['site', 'season']].drop_duplicates().shape[0]} site-seasons have years that were never searched (network failure). Re-run Cell 8 to fill them: years {', '.join(str(int(v)) for v in sorted(nf.year.unique()))}")
    rows_ = list(AV[["site", "season"]].drop_duplicates().itertuples(index=False, name=None))
    M = np.array([[AV[(AV.site == s_) & (AV.season == k_) & (AV.year == y)].category.iloc[0] for y in LANDSAT_YEARS] for s_, k_ in rows_], float)
    cmap_ = ListedColormap([AV_CAT[k][1] for k in sorted(AV_CAT)])
    fig, ax = plt.subplots(figsize=(15.5, 0.42 * len(rows_) + 2.6))
    ax.imshow(M, cmap=cmap_, vmin=-0.5, vmax=len(AV_CAT) - 0.5, aspect="auto", interpolation="nearest")
    for i, (s_, k_) in enumerate(rows_):
        for j, y in enumerate(LANDSAT_YEARS):
            n_ = AV[(AV.site == s_) & (AV.season == k_) & (AV.year == y)].n_clear_images.iloc[0]
            if M[i, j] >= 2: ax.text(j, i, str(int(n_)), ha="center", va="center", fontsize=6.5, color="white" if M[i, j] == 4 else "#333333")
    ax.set_yticks(range(len(rows_))); ax.set_yticklabels([f"{SITES[s_]['name']}, {'end of rains' if k_ == 'mam' else 'end of dry season'}" for s_, k_ in rows_], fontsize=8.5)
    ax.set_xticks(range(0, len(LANDSAT_YEARS), 5)); ax.set_xticklabels(LANDSAT_YEARS[::5]); ax.grid(False)
    ax.set_xticks(np.arange(-0.5, len(LANDSAT_YEARS)), minor=True); ax.set_yticks(np.arange(-0.5, len(rows_)), minor=True); ax.grid(which="minor", color="white", lw=0.8); ax.tick_params(which="minor", length=0)
    for k in range(1, len(rows_) // 2): ax.axhline(2 * k - 0.5, color="k", lw=0.8)
    for yb, lab in [(1999, "Landsat 7 launched"), (2003, "L7 scan-line failure"), (2013, "Landsat 8")]:
        if yb in LANDSAT_YEARS: j = LANDSAT_YEARS.index(yb); ax.axvline(j - 0.5, color="#555555", ls="--", lw=0.8); ax.text(j - 0.4, -0.7, lab, fontsize=7.5, color="#555555", ha="left", va="bottom")
    hs = [Patch(facecolor=AV_CAT[k][1], edgecolor="#999999", label=AV_CAT[k][0]) for k in sorted(AV_CAT) if (M == k).any()]
    finish(fig, "01_data_availability", f"Every dam is analysed over the same {LANDSAT_YEARS[0]}-{LANDSAT_YEARS[-1]} period - this is which years actually have usable data",
           f"Each cell is one dam, season and year; the number is how many clear images that composite used. Before 1999 only Landsat 5 flew (one image every 16 days), and the end of the rains is cloudy, "
           f"so early years are often dropped for having fewer than {MIN_SCENES_YEAR} clear images. Trends with fewer than {MIN_YEARS_PER_END} usable years in {EARLY[0]}-{EARLY[1]} are labelled PARTIAL everywhere in this notebook.",
           source="Landsat 5/7/8/9 C2-L2 (30 m), Microsoft Planetary Computer", handles=hs, legend_cols=3, top_extra=0.15)
    EXTRA_AV_LINE = (f"Data availability: the analysis period is {LANDSAT_YEARS[0]}-{LANDSAT_YEARS[-1]} at every dam; the first usable end-of-dry-season year is " +
                     ", ".join(f"{SITES[s_]['name'].replace(' Dam', '')} {int(first.loc[(s_, 'dry'), 'first_usable'])}" for s_ in RES if (s_, "dry") in first.index) + " (figure 01 shows why).")
    ADV_SHEETS_AV = {"data_availability": AV, "first_usable_year": first.reset_index()}


## Step 3. Climate at each site (runs before the vegetation checks so every figure marks the same drought years)
One climate product for all sites (ERA5 by default, `CLIMATE_SOURCE` in Cell 2). Rainfall and drought index per site, and the key test: does water cover follow the rain, and does a decline survive once rainfall is removed?

In [ ]:
# Cell 10: CLIMATE FOR EACH SITE and the rainfall test: does the water follow the rain, and does the trend survive once rainfall is removed?
# v5: (1) ONE climate product for every site (CLIMATE_SOURCE in Cell 2) - mixing TerraClimate at one dam with ERA5 at another makes the sites incomparable;
#     (2) this cell now runs straight after Cell 9 so every later figure can mark the same drought years.
def _load_terraclimate(lat, lon):
    coll = catalog.get_collection("terraclimate"); asset = planetary_computer.sign(coll.assets["zarr-abfs"])   # fresh token for every call (old tokens expire)
    kw = dict(asset.extra_fields.get("xarray:open_kwargs", {})); so = asset.extra_fields.get("xarray:storage_options")
    if so is not None and "storage_options" not in kw: kw["storage_options"] = so
    ds = xr.open_dataset(asset.href, **kw)
    pt = ds[["ppt", "pet", "soil", "pdsi", "tmax"]].sel(lat=lat, lon=lon, method="nearest").load()
    return pt.to_dataframe().drop(columns=["lat", "lon"], errors="ignore"), "TerraClimate (Planetary Computer)"

def _load_era5(lat, lon):
    """ERA5 via Open-Meteo. Cached on disk (raw_meta/climate_era5_<lat>_<lon>.csv) so a re-run never calls the API again.
    The free API limits request rate (HTTP 429 'Too Many Requests'), so each call waits and retries with growing pauses."""
    import requests, time
    cache = DIRS["raw_meta"] / f"climate_era5_{lat:.4f}_{lon:.4f}.csv"
    if cache.exists():
        m = pd.read_csv(cache, index_col=0, parse_dates=True)
        if m.index.max().year >= LANDSAT_YEARS[-1]: return m, "ERA5 reanalysis (Open-Meteo)"
    params = {"latitude": lat, "longitude": lon, "start_date": "1960-01-01", "end_date": f"{LANDSAT_YEARS[-1]}-12-31",
              "daily": "precipitation_sum,et0_fao_evapotranspiration,temperature_2m_max", "timezone": "GMT"}
    for wait in [0, 20, 60, 120, 240]:
        if wait: print(f"    Open-Meteo busy (429), waiting {wait} s and retrying ..."); time.sleep(wait)
        resp = requests.get("https://archive-api.open-meteo.com/v1/archive", timeout=180, params=params)
        if resp.status_code != 429: break
    resp.raise_for_status(); j = resp.json()["daily"]
    dd = pd.DataFrame({"ppt": j["precipitation_sum"], "pet": j["et0_fao_evapotranspiration"], "tmax": j["temperature_2m_max"]}, index=pd.to_datetime(j["time"]))
    m = pd.DataFrame({"ppt": dd.ppt.resample("MS").sum(min_count=20), "pet": dd.pet.resample("MS").sum(min_count=20), "tmax": dd.tmax.resample("MS").mean()})
    m["soil"] = np.nan; m["pdsi"] = np.nan; m = m.dropna(subset=["ppt"])
    m.to_csv(cache); time.sleep(5)                                    # pause before the next site's request
    return m, "ERA5 reanalysis (Open-Meteo)"

def load_climate(lat, lon):
    """Monthly ppt, pet, tmax (+ soil, pdsi for TerraClimate) from the ONE product chosen in Cell 2."""
    return _load_terraclimate(lat, lon) if CLIMATE_SOURCE == "terraclimate" else _load_era5(lat, lon)

def at(s, y, m):
    v = s[(s.index.year == y) & (s.index.month == m)]
    return float(v.iloc[0]) if len(v) else np.nan

def build_drivers(tc):
    ppt = tc["ppt"]; s3, s6, s12 = spi(ppt, 3), spi(ppt, 6), spi(ppt, 12)
    d0, d1 = SEASONS["dry"]; g0, g1 = SEASONS["mam"]; rows = []
    for y in range(tc.index.year.min() + 1, tc.index.year.max() + 1):
        yr, mo = tc.index.year, tc.index.month
        dry = tc[(yr == y) & (mo >= d0) & (mo <= d1)]; mam = tc[(yr == y) & (mo >= g0) & (mo <= g1)]
        hyd = tc[((yr == y - 1) & (mo >= 11)) | ((yr == y) & (mo <= d1))]
        wet = tc[((yr == y - 1) & (mo >= 11)) | ((yr == y) & (mo <= 4))]
        rows.append({"year": y, "wet_season_ppt_mm": wet.ppt.sum(), "hydro_year_ppt_mm": hyd.ppt.sum(),
                     "water_balance_mm": (hyd.ppt - hyd.pet).sum(),
                     "spi3_dry": at(s3, y, d1), "spi6_dry": at(s6, y, d1), "spi12_dry": at(s12, y, d1),
                     "spi3_mam": at(s3, y, g1), "spi6_mam": at(s6, y, g1),
                     "pdsi_dry": dry.pdsi.mean(), "pdsi_mam": mam.pdsi.mean(),
                     "soil_mm_dry": dry.soil.mean(), "soil_mm_mam": mam.soil.mean(),
                     "tmax_dry": dry.tmax.mean(), "tmax_mam": mam.tmax.mean(), "pet_dry_mm": dry.pet.sum()})
    dr = pd.DataFrame(rows).set_index("year")
    dr["spi12_dry_lag1"] = dr.spi12_dry.shift(1); dr["wet_season_ppt_lag1"] = dr.wet_season_ppt_mm.shift(1)
    return dr


CLIM = {}; TC = {}
def site_climate(sid):
    s = SITES[sid]; key = (round(s["lat"] * 4) / 4, round(s["lon"] * 4) / 4)       # ERA5 is ~0.25 deg, so nearby sites share one download
    if key not in CLIM:
        try:
            tc, src = load_climate(s["lat"], s["lon"]); TC[key] = tc; CLIM[key] = (build_drivers(tc), src)
        except Exception as e:
            print(f"  climate failed for {s['name']}: {type(e).__name__}: {e}"); CLIM[key] = (None, None)
    return CLIM[key]

def rain_test(st, drivers, metric="water_pct_foot"):
    """Spearman link to wet-season rain + trend left after removing the rainfall effect (OLS residuals)."""
    if drivers is None or st is None: return {}
    m = st[st.ok_site][["year", metric]].merge(drivers.reset_index()[["year", "wet_season_ppt_mm", "spi12_dry"]], on="year").dropna()
    if len(m) < 8 or m[metric].std() == 0: return {}
    rho, p = stats.spearmanr(m[metric], m.wet_season_ppt_mm)
    lr = stats.linregress(m.wet_season_ppt_mm, m[metric]); resid = m[metric] - (lr.intercept + lr.slope * m.wet_season_ppt_mm)
    q = quick_trend(m.year.values, resid.values, min_n=8)             # same Hamed-Rao rule as every other trend in the notebook
    if q is None: return {"rain_rho": rho, "rain_p": p, "n": len(m)}
    return {"rain_rho": rho, "rain_p": p, "n": len(m), "resid_trend": q["trend"], "resid_p": q["p"], "resid_slope_dec": 10 * q["slope"]}

for _attempt in range(2):                                  # if the chosen product fails at ANY site, switch every site to ERA5 (never mix products)
    CLIM.clear(); TC.clear()
    _fail = [sid for sid, r in RES.items() if not r.get("error") and site_climate(sid)[0] is None]
    if not _fail or CLIMATE_SOURCE == "era5": break
    print(f"{CLIMATE_SOURCE} failed for {_fail}; using ERA5 for ALL sites so they stay comparable"); CLIMATE_SOURCE = "era5"
for sid, r in RES.items():
    if r.get("error"): continue
    drv, src = site_climate(sid); r["drivers"], r["climate_src"] = drv, src; r["tc"] = TC.get((round(SITES[sid]["lat"] * 4) / 4, round(SITES[sid]["lon"] * 4) / 4))
    r["rain"] = {sk: rain_test(r["stats"].get((r["prim"], sk)), drv) for sk in SEASONS}
    rr = r["rain"].get(WATER_SEASON, {})
    if rr: print(f"{SITES[sid]['name']:<30} water vs wet-season rain: rho={rr['rain_rho']:+.2f} ({fmt_p(rr['rain_p'])}); trend after removing rainfall: {rr['resid_trend']} ({fmt_p(rr['resid_p'])})")
print("climate source:", {v[1] for v in CLIM.values() if v[1]})
_noclim = [SITES[sid]["name"] for sid, r in RES.items() if not r.get("error") and r.get("drivers") is None]
if _noclim: print(f"\nWARNING: no climate data for {', '.join(_noclim)}. Their drought shading, rainfall test and early-warning tests will be missing. Re-run this cell (downloads are cached, so it only fetches what is missing).")

for sid, r in RES.items():
    if not r.get("error") and r.get("drivers") is not None:
        dy = [int(y) for y in r["drivers"].index[r["drivers"][DROUGHT_SPI] < STRESS_Z] if LANDSAT_YEARS[0] <= y <= LANDSAT_YEARS[-1]]
        print(f"  {SITES[sid]['name']:<14} drought years ({DROUGHT_SPI} < {STRESS_Z}): {dy}")


## Step 2b. Sensor checks and seasonal drawdown
Core: `06_seasonal_drawdown`, how much water each dam loses between the end of the rains and the end of the dry season (reservoir surface-area fluctuation). Supplementary method checks: `S1_sensor_harmonisation` (the Landsat 5/7 to 8/9 offset removed before every vegetation trend) and `S2_landsat_vs_sentinel2` (does the 30-year Landsat record agree with sharper Sentinel-2, 2015-2025?).

In [ ]:
# Cell 9b: ADVANCED CHECKS carried over from the v2 notebook, now run for every dam: (1) satellite-family offsets, (2) Landsat vs Sentinel-2 pixel and scene agreement, (3) harmonisation, (4) seasonal drawdown.
FAMILY = lambda p: "OLI (L8/L9)" if p in ("landsat-8", "landsat-9") else "TM/ETM+ (L5/L7)"
ADV = {k: [] for k in ["fam_off", "fam_tr", "harm", "harm_tr", "pix", "vpix", "pair", "draw"]}; FAMS = {}; FAMS_H = {}; DRAW = {}; ADV_LINES = []; ADV_SHEETS = {}
FAM_METRICS = {"ndvi_nat_mean": "NDVI, natural vegetation", "moist_nat_mean": "Moisture NDWI, natural vegetation", "water_frac_valid": "Water fraction of clear pixels"}

def family_series(scenes, metric, raw=True):
    """Yearly median per satellite family. raw=True uses the values BEFORE harmonisation (to show the sensor offset itself).
    Landsat 7 scenes from L7_DRIFT_YEAR on are left out (drifted / lowered orbit changes the illumination geometry)."""
    col = metric + "_raw" if (raw and metric + "_raw" in scenes) else metric
    s = scenes[scenes.scene_ok & ~((scenes.platform == "landsat-7") & (scenes.year >= L7_DRIFT_YEAR))].copy(); s["family"] = s.platform.map(FAMILY)
    g = s.groupby(["year", "family"])[col].agg(["median", "count"]).reset_index(); g = g[g["count"] >= 2]
    return g.pivot(index="year", columns="family", values="median")

def _on_grid(da2d, like):
    """Put a 2-D boolean/float DataArray on the grid of `like` (nearest pixel within 30 m)."""
    return da2d.reindex_like(like, method="nearest", tolerance=30).fillna(0).values.astype(bool)

for sid, r in RES.items():
    if r.get("error") or r.get("prim") != "landsat": continue
    D = SITE_DATA[sid]["D"]; name = SITES[sid]["name"]; fp = r["fp"]
    try:
        # ---- 1. satellite-family check (scene level): is a change just the switch from TM/ETM+ to OLI? ----
        for sk in SEASONS:
            scn = D[("landsat", sk)]["scenes"]
            for m, lab in FAM_METRICS.items():
                ser = family_series(scn, m); FAMS[(sid, sk, m)] = ser; FAMS_H[(sid, sk, m)] = family_series(scn, m, raw=False)
                if {"OLI (L8/L9)", "TM/ETM+ (L5/L7)"} <= set(ser.columns):
                    both = ser.dropna(subset=["OLI (L8/L9)", "TM/ETM+ (L5/L7)"])
                    if len(both) >= 4:
                        diff = both["OLI (L8/L9)"] - both["TM/ETM+ (L5/L7)"]
                        try: pw = stats.wilcoxon(diff).pvalue
                        except Exception: pw = np.nan
                        ADV["fam_off"].append({"site": sid, "season": sk, "metric": m, "overlap_years": len(both), "OLI_minus_ETM_mean": diff.mean(), "wilcoxon_p": pw,
                                               "offset_as_pct_of_mean": 100 * diff.mean() / abs(both["TM/ETM+ (L5/L7)"].mean())})
                for fam in ser.columns:
                    d_ = ser[fam].dropna().reset_index().rename(columns={fam: m}); t = trend_table(d_, {m: lab}, min_n=8)
                    if len(t): ADV["fam_tr"].append(t.assign(site=sid, season=sk, family=fam, span=f"{int(d_.year.min())}-{int(d_.year.max())}"))
        # ---- 3. harmonisation: express Landsat on the Sentinel-2 scale, but only where the two agree well (r2 >= 0.7) ----
        for sk, m in [(WATER_SEASON, "water_ha"), (VEG_SEASON, "ring_ndvi_mean"), (VEG_SEASON, "ring_moist_mean")]:
            L, S = r["stats"].get(("landsat", sk)), r["stats"].get(("s2_30", sk))
            if L is None or S is None: continue
            ov = sorted(set(L[L.ok_site].year) & set(S[S.ok_site].year))
            if len(ov) < 5: continue
            Lx, Sx = L.set_index("year").loc[ov, m].values, S.set_index("year").loc[ov, m].values
            if not (np.isfinite(Lx).all() and np.isfinite(Sx).all()) or np.ptp(Lx) == 0: continue
            lr = stats.linregress(Lx, Sx); applied = lr.rvalue ** 2 >= 0.7
            ADV["harm"].append({"site": sid, "season": sk, "metric": m, "intercept": lr.intercept, "slope": lr.slope, "r2": lr.rvalue ** 2, "n": len(ov), "applied": applied})
            if applied:
                Lh = L[L.ok_site].copy(); Lh[m + "_harmonised"] = lr.intercept + lr.slope * Lh[m]
                ADV["harm_tr"].append(trend_table(Lh, {m + "_harmonised": m + " (harmonised)"}, min_n=8).assign(site=sid, season=sk))
        # ---- 2. pixel-by-pixel and scene-pair agreement (Sentinel-2 30 m = reference) ----
        lc = SITE_DATA[sid]["LCM"].get(30); dist = ndimage.distance_transform_edt(~fp["foot"]) * 30
        for sk in SEASONS:
            L, S = r["stats"].get(("landsat", sk)), r["stats"].get(("s2_30", sk))
            if L is None or S is None: continue
            ov = sorted(set(L[L.ok_site].year) & set(S[S.ok_site].year)); la, sa = D[("landsat", sk)]["annual"], D[("s2_30", sk)]["annual"]
            zone_w = xr.DataArray(dist <= 1000, coords={"y": la.y, "x": la.x}, dims=("y", "x"))                       # the lake and 1 km around it
            zone_v = xr.DataArray((dist <= RING_WIDE_M) & ~fp["foot"], coords={"y": la.y, "x": la.x}, dims=("y", "x"))  # land around the lake
            for y in ov:
                s_, l_ = sa.sel(year=y), la.sel(year=y)
                if not (np.array_equal(s_.x, l_.x) and np.array_equal(s_.y, l_.y)): l_ = l_.reindex_like(s_, method="nearest", tolerance=30)
                zw, zv = _on_grid(zone_w, s_), _on_grid(zone_v, s_)
                both = ((s_.n_obs > 0) & (l_.n_obs > 0)).values & zw
                a = (s_.water_freq >= 0.5).values & both; b = (l_.water_freq >= 0.5).values & both
                tp, fn, fpx = int((a & b).sum()), int((a & ~b).sum()), int((~a & b).sum()); tn = int((~a & ~b & both).sum()); N = tp + tn + fpx + fn
                if N:
                    po = (tp + tn) / N; pe = ((tp + fn) * (tp + fpx) + (fpx + tn) * (fn + tn)) / N ** 2
                    ADV["pix"].append({"site": sid, "season": sk, "year": y, "overall_accuracy": po, "IoU": tp / max(tp + fpx + fn, 1), "F1": 2 * tp / max(2 * tp + fpx + fn, 1),
                                       "kappa": (po - pe) / (1 - pe) if pe < 1 else np.nan, "omission_pct": 100 * fn / max(tp + fn, 1), "commission_pct": 100 * fpx / max(tp + fpx, 1)})
                if lc and lc.get("source") == "worldcover":
                    nat = ((s_.n_obs > 0) & (l_.n_obs > 0)).values & zv & ~(s_.water_freq >= 0.5).values & ~(l_.water_freq >= 0.5).values & _mask2d(lc["natural"], s_).values
                    for var in ["ndvi", "ndwi_moist"]:
                        xs, ys = s_[var].values[nat], l_[var].values[nat]; ok_ = np.isfinite(xs) & np.isfinite(ys)
                        if ok_.sum() > 50 and np.ptp(xs[ok_]) > 0 and np.ptp(ys[ok_]) > 0:
                            ADV["vpix"].append({"site": sid, "season": sk, "year": y, "variable": var, "n_pixels": int(ok_.sum()), "pixel_r": float(np.corrcoef(xs[ok_], ys[ok_])[0, 1]),
                                                "bias_landsat_minus_s2": float(np.mean(ys[ok_] - xs[ok_]))})
            a_ = D[("s2_30", sk)]["scenes"].query("scene_ok").sort_values("date"); b_ = D[("landsat", sk)]["scenes"].query("scene_ok").sort_values("date")
            pairs = pd.merge_asof(a_, b_, on="date", direction="nearest", tolerance=pd.Timedelta("3D"), suffixes=("_s2", "_ls")).dropna(subset=["water_frac_valid_ls"])
            if len(pairs) >= 3:
                for m in ["water_frac_valid", "moist_nat_mean", "ndvi_nat_mean"]:
                    ag = agreement(pairs[m + "_s2"], pairs[m + "_ls"]); ADV["pair"].append({"site": sid, "season": sk, "metric": m, "n_pairs": len(pairs), **{k: ag.get(k, np.nan) for k in ["pearson_r", "bias", "rmse"]}})
        # ---- 4. seasonal drawdown: how much water is lost between the end of the rains and the end of the dry season? ----
        a = r["stats"].get(("landsat", "mam")); b = r["stats"].get(("landsat", "dry"))
        if a is not None and b is not None:
            m_ = a[a.ok_site][["year", "water_pct_foot", "water_ha", "phys_fail"]].merge(b[b.ok_site][["year", "water_pct_foot", "water_ha"]], on="year", suffixes=("_mam", "_dry"))
            m_["drawdown_pts"] = m_.water_pct_foot_mam - m_.water_pct_foot_dry; m_["drawdown_ha"] = m_.water_ha_mam - m_.water_ha_dry; DRAW[sid] = m_
            t = trend_table(m_, {"drawdown_pts": "Seasonal drawdown (percentage points of the footprint)"}, min_n=8)
            ADV["draw"].append({"site": sid, "n_years": len(m_), "mean_drawdown_pts": float(m_.drawdown_pts.mean()), "mean_drawdown_ha": float(m_.drawdown_ha.mean()),
                                "trend": (t.iloc[0].mk_trend if len(t) else None), "p_hamed_rao": (t.iloc[0].mk_p_hamed_rao if len(t) else np.nan), "sen_pts_per_decade": (10 * t.iloc[0].sen_slope_per_year if len(t) else np.nan)})
    except Exception as e:
        print(f"advanced checks skipped for {name}: {type(e).__name__}: {e}")

T = {k: (pd.concat(v, ignore_index=True) if v and isinstance(v[0], pd.DataFrame) else pd.DataFrame(v)) for k, v in ADV.items()}
ADV_SHEETS.update({"sensor_family_offsets": T["fam_off"], "trend_single_family": T["fam_tr"], "harmonisation": T["harm"], "trend_harmonised": T["harm_tr"],
                   "agree_water_mask": T["pix"], "agree_vegetation_pixels": T["vpix"], "agree_scene_pairs": T["pair"], "seasonal_drawdown": T["draw"]})

print("1. SATELLITE-FAMILY OFFSETS (OLI minus TM/ETM+, years both observed; a large offset means a trend across 2013 may be a sensor artefact):")
if len(T["fam_off"]): display(T["fam_off"].round(4))
if len(T["fam_tr"]): print("\nTrend inside ONE satellite family at a time:"); display(T["fam_tr"][["site", "season", "family", "span", "metric", "n_years", "mk_trend", "mk_p_hamed_rao", "sen_slope_per_year"]].round(4))
print("\n2. LANDSAT vs SENTINEL-2 (mean over overlap years):")
if len(T["pix"]): print("Water-mask agreement:"); display(T["pix"].groupby(["site", "season"])[["overall_accuracy", "IoU", "F1", "kappa", "omission_pct", "commission_pct"]].mean().round(3))
if len(T["vpix"]): print("Vegetation pixel agreement:"); display(T["vpix"].groupby(["site", "season", "variable"])[["pixel_r", "bias_landsat_minus_s2"]].mean().round(3))
if len(T["pair"]): print("Scene pairs within 3 days:"); display(T["pair"].round(3))
print("\n3. HARMONISATION (Sentinel-2 = a + b x Landsat; applied only if r2 >= 0.7):")
if len(T["harm"]): display(T["harm"].round(3))
else: print("  not enough overlap years")
if len(T["harm_tr"]): display(T["harm_tr"][["site", "season", "metric", "n_years", "mk_trend", "mk_p_hamed_rao", "sen_slope_per_year"]].round(4))
print("\n4. SEASONAL DRAWDOWN:")
if len(T["draw"]): display(T["draw"].round(3))

# ---- plain-language lines for the conclusion ----
fo = T["fam_off"]
if len(fo):
    nv = fo[(fo.metric == "ndvi_nat_mean") & (fo.season == VEG_SEASON)]
    if len(nv): ADV_LINES.append(f"Sensor check: in years when both Landsat generations observed a dam, OLI (Landsat 8/9) read {nv.offset_as_pct_of_mean.median():+.0f}% (median over dams) higher than TM/ETM+ (Landsat 5/7) for natural-vegetation greenness before harmonisation; this offset is removed in every trend (figure S1).")
px = T["pix"]
if len(px):
    d_ = px[px.season == WATER_SEASON].groupby("site").IoU.mean()
    if len(d_): ADV_LINES.append(f"Landsat and Sentinel-2 water masks overlap by IoU {d_.min():.2f} to {d_.max():.2f} across dams (1.0 = identical), so lake outlines are consistent but not exact.")

# ---- figures ----
shown = [s for s in RES if not RES[s].get("error") and (s, VEG_SEASON, "ndvi_nat_mean") in FAMS]
if shown:
    fig, axs = plt.subplots(len(shown), 2, figsize=(14, 2.3 * len(shown) + 1.2), squeeze=False, sharex=True)
    FC = {"TM/ETM+ (L5/L7)": (FAMILY_COL["TM/ETM+ (L5/L7)"], "s"), "OLI (L8/L9)": (FAMILY_COL["OLI (L8/L9)"], "o")}
    for i, sid in enumerate(shown):
        for j, (m, lab) in enumerate([("ndvi_nat_mean", "Greenness (NDVI)"), ("moist_nat_mean", "Canopy water content (NDWI, NIR-SWIR)")]):
            ax = axs[i, j]; ser = FAMS[(sid, VEG_SEASON, m)]; serh = FAMS_H.get((sid, VEG_SEASON, m))
            ax.axvspan(HARM_YEARS[0] - 0.5, HARM_YEARS[1] + 0.5, color="#f0f0f0", lw=0, zorder=0)
            for fam in ser.columns: gap_line(ax, ser.index, ser[fam], FC[fam][0], marker=FC[fam][1], ms=4, lw=1.3, label=f"{fam}, as measured")
            if serh is not None and "TM/ETM+ (L5/L7)" in serh:
                gap_line(ax, serh.index, serh["TM/ETM+ (L5/L7)"], FC["TM/ETM+ (L5/L7)"][0], marker="s", ms=3.5, lw=1.0, mfc="white", ls=":", label="TM/ETM+ after harmonisation (used in all trends)", alpha=0.9, mec=FC["TM/ETM+ (L5/L7)"][0])
            ax.axvline(SENSOR_BREAK - 0.5, c="k", ls="--", lw=0.8)
            h = HARM_TAB[(HARM_TAB.site == sid) & (HARM_TAB.season == VEG_SEASON) & (HARM_TAB.variable == {"ndvi_nat_mean": "ndvi", "moist_nat_mean": "ndwi_moist"}[m])] if len(HARM_TAB) else []
            if len(h): ax.text(0.01, 0.05, f"offset OLI - TM/ETM+ = {h.iloc[0].offset_OLI_minus_TM_ETM:+.3f} ({int(h.iloc[0].overlap_years)} shared years)", transform=ax.transAxes, fontsize=8, bbox=dict(fc="white", ec="#cccccc", alpha=0.92, boxstyle="round,pad=0.25"))
            if j == 0: ax.set_ylabel(SITES[sid]["name"] + "\n" + "index value", fontsize=9)
            if i == 0: ax.set_title(lab, loc="left", fontsize=10.5)
    hs = [Line2D([0], [0], color=FC["TM/ETM+ (L5/L7)"][0], marker="s", mec="white", label="Landsat 5/7 (TM/ETM+), as measured"), Line2D([0], [0], color=FC["OLI (L8/L9)"][0], marker="o", mec="white", label="Landsat 8/9 (OLI)"),
          Line2D([0], [0], color=FC["TM/ETM+ (L5/L7)"][0], marker="s", mfc="white", ls=":", label="Landsat 5/7 after harmonisation"), Patch(color="#f0f0f0", label=f"years used to estimate the offset ({HARM_YEARS[0]}-{HARM_YEARS[1]})")]
    axs[-1, 0].set_xlabel("Year"); axs[-1, 1].set_xlabel("Year")
    finish(fig, "S1_sensor_harmonisation", "The 2013 satellite switch creates a fake step up in greenness and canopy water, so it is removed before any trend is tested",
           f"Scene medians of natural vegetation in each study box ({season_label(VEG_SEASON)}). In years when both satellite families flew, Landsat 8/9 reads higher than Landsat 5/7 for the same plants (narrower NIR band). "
           f"That offset is added to every Landsat 5/7 look (dotted line), so 1996-2012 and 2013-2025 are on one scale. Landsat 7 after {L7_DRIFT_YEAR - 1} (drifting orbit) is left out.",
           source="Landsat 5/7/8/9 C2-L2 (30 m)", handles=hs, legend_cols=4)
if len(T["pix"]) or len(T["vpix"]) or len(T["pair"]):
    fig, axs = plt.subplots(1, 3, figsize=(17, 5.2)); sites_ = [s for s in RES if not RES[s].get("error")]; nm = [SITES[s]["name"].replace(" Dam", "") for s in sites_]; xx = np.arange(len(sites_))
    if len(T["pix"]):
        g_ = T["pix"][T["pix"].season == WATER_SEASON].groupby("site")[["IoU", "F1", "kappa"]].mean().reindex(sites_)
        for k, (c_, col) in enumerate([("IoU", C_WATER), ("F1", C_WATER_L), ("kappa", "#999999")]): axs[0].bar(xx + (k - 1) * 0.26, g_[c_], 0.26, color=col, edgecolor="#444444", label=c_, zorder=3)
        axs[0].legend(fontsize=8.5, loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3)
    axs[0].set_xticks(xx); axs[0].set_xticklabels(nm); axs[0].set_ylim(0, 1.05); axs[0].axhline(0.8, color=C_DRY, ls=":", lw=1.2); axs[0].set_ylabel("Agreement (1 = identical)"); axs[0].set_title("A. Do both satellites draw the same lake?", loc="left", fontsize=10.5)
    if len(T["vpix"]):
        g_ = T["vpix"][T["vpix"].season == VEG_SEASON].groupby(["site", "variable"]).pixel_r.mean().unstack().reindex(sites_)
        for k, (c_, col) in enumerate([("ndvi", C_VEG), ("ndwi_moist", C_WATER)]):
            if c_ in g_: axs[1].bar(xx + (k - 0.5) * 0.36, g_[c_], 0.36, color=col, edgecolor="#444444", label={"ndvi": "Greenness (NDVI)", "ndwi_moist": "Vegetation water (NDWI)"}[c_], zorder=3)
        axs[1].legend(fontsize=8.5, loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2)
    axs[1].set_xticks(xx); axs[1].set_xticklabels(nm); axs[1].set_ylim(0, 1.05); axs[1].set_ylabel("Pixel-by-pixel correlation"); axs[1].set_title("B. Do they see the same vegetation, pixel by pixel?", loc="left", fontsize=10.5)
    if len(T["pair"]):
        g_ = T["pair"][T["pair"].season == WATER_SEASON].pivot_table(index="site", columns="metric", values="pearson_r").reindex(sites_)
        for k, (c_, col, lb) in enumerate([("water_frac_valid", C_WATER, "Water fraction"), ("ndvi_nat_mean", C_VEG, "NDVI")]):
            if c_ in g_: axs[2].bar(xx + (k - 0.5) * 0.36, g_[c_], 0.36, color=col, edgecolor="#444444", label=lb, zorder=3)
        axs[2].legend(fontsize=8.5, loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2)
    axs[2].set_xticks(xx); axs[2].set_xticklabels(nm); axs[2].set_ylim(-0.2, 1.05); axs[2].axhline(0, c="k", lw=0.8); axs[2].set_ylabel("Correlation"); axs[2].set_title("C. Same-week scene pairs (end of dry season)", loc="left", fontsize=10.5)
    for a_ in axs: a_.grid(True, axis="y"); a_.grid(False, axis="x")
    finish(fig, "S2_landsat_vs_sentinel2", "Can we trust Landsat for the 30-year record? Comparing it with Sentinel-2",
           "Sentinel-2 is sharper (10 to 20 m) and is used as the reference where both satellites overlap (2015 to 2025). Values near 1 mean the long Landsat record tells the same story; low values mean its year-to-year changes at that dam should be read with caution.",
           source="Landsat 5/7/8/9 C2-L2 and Sentinel-2 L2A (30 m)", handles=None)
if DRAW:
    ns_ = len(DRAW); ncol = 3; nrow = int(np.ceil(ns_ / ncol)); fig, axs = plt.subplots(nrow, ncol, figsize=(5.6 * ncol, 3.7 * nrow), squeeze=False, sharey=True); [a_.axis("off") for a_ in axs.ravel()]
    for ax, (sid, m_) in zip(axs.ravel(), DRAW.items()):
        ax.axis("on"); shade_eras(ax); okb = m_.water_pct_foot_mam.notna()
        yy_ = m_.year.values.astype(float); lo_ = m_.water_pct_foot_dry.values; hi_ = m_.water_pct_foot_mam.values
        for y0, a0, b0 in zip(yy_[okb], lo_[okb], hi_[okb]):        # one bar per year = that year's drawdown (no band drawn across missing years)
            ax.plot([y0, y0], [min(a0, b0), max(a0, b0)], color=C_DRY if b0 >= a0 else C_GREY, lw=3.2, alpha=0.55, solid_capstyle="butt", zorder=2)
        gap_line(ax, m_.year[okb], m_.water_pct_foot_mam[okb], C_RAINS, "o", ms=4, lw=1.3)
        gap_line(ax, m_.year, m_.water_pct_foot_dry, C_DRYS, "s", ms=4, lw=1.3)
        fail = m_[m_.phys_fail.fillna(False).astype(bool)]
        if len(fail): ax.plot(fail.year, np.full(len(fail), 3.0), "x", color=C_HOT, ms=6, mew=1.6, zorder=6)
        mdd = m_.drawdown_pts.median(); nf = len(fail); n_ = len(m_)
        ax.set_ylim(0, 105); ax.set_ylabel("% of usual footprint under water")
        ttl_ = f"median drawdown {mdd:.0f} points" if mdd >= 1 else f"no measurable drawdown (median {mdd:+.0f} points, within error)"
        ax.set_title(f"{SITES[sid]['name']}: {ttl_}" + (f"\n{nf} of {n_} end-of-rains values failed the physics check" if nf else ""), loc="left", fontsize=10)
    hs = [Line2D([0], [0], color=C_RAINS, marker="o", mec="white", label=season_label("mam")), Line2D([0], [0], color=C_DRYS, marker="s", mec="white", label=season_label("dry")),
          Line2D([0], [0], color=C_DRY, lw=4, alpha=0.55, label="Drawdown: water lost over the dry season"), Line2D([0], [0], color=C_GREY, lw=4, alpha=0.55, label=f"Small apparent gain (< {MAX_REFILL_PTS} points, within error)"),
          Line2D([0], [0], color=C_HOT, marker="x", ls="none", mew=1.6, label="End-of-rains value removed (physically impossible)")] + ERA_HANDLES
    finish(fig, "06_seasonal_drawdown", "How much water does each dam lose between the end of the rains and the end of the dry season?",
           f"Each vertical bar is one year: from the end-of-rains water cover (dark blue) down to the end-of-dry-season cover (light blue). With almost no rain from June to October a dam cannot gain water, "
           f"so an end-of-rains value more than {MAX_REFILL_PTS} points BELOW the dry season is a detection failure (haze, muddy or plant-covered water) and is removed (red x) instead of being plotted as a 'negative drawdown'. Lines are not drawn across missing years.",
           source="Landsat 5/7/8/9 C2-L2 (30 m)", handles=hs, legend_cols=4)


## Step 2c. Vegetation around each dam (supplementary)
Natural vegetation vs cropland around each dam, where greenness and canopy water are changing (FDR-corrected pixel-trend maps), an interactive web map per dam, and a second water index (MNDWI) as a cross-check of the open-water trend. Figures `S3`-`S5` and `supplementary/interactive_map_*.html`.

In [ ]:
# Cell 9c: VEGETATION AROUND EACH DAM. What type of land surrounds it (natural vegetation vs cropland), where is the vegetation greening or browning, and how do the two land types behave over 30 years? Plus a second water index (MNDWI) as a cross-check.
import rioxarray  # adds the .rio accessor used to reproject for the web map
from matplotlib.colors import ListedColormap, LinearSegmentedColormap
try:
    import folium
except ImportError:
    import subprocess, sys; subprocess.run([sys.executable, "-m", "pip", "install", "-q", "folium"]); import folium
CM_GREEN = LinearSegmentedColormap.from_list("brown_green", [C_DRY, "#f5f5f5", C_VEG]); CM_BLUE = LinearSegmentedColormap.from_list("brown_blue", [C_DRY, "#f5f5f5", C_WATER])
VEG, VEG_SHEETS, VEG_LINES, MNDWI_ROWS = {}, {}, [], []
LC_RGBA = {k: [int(c[i:i + 2], 16) for i in (1, 3, 5)] + [170] for k, c in LC_COLORS.items()}

def _win(mask, pad=10):
    ys, xs = np.where(mask); return slice(max(ys.min() - pad, 0), ys.max() + pad + 1), slice(max(xs.min() - pad, 0), xs.max() + pad + 1)

def _overlay(arr2d, B, rgba_fn):
    """Reproject a 2-D array on the analysis grid to lat/lon and colour it, for the interactive web map."""
    da = xr.DataArray(np.asarray(arr2d, "float32"), coords={"y": B.y.values, "x": B.x.values}, dims=("y", "x")).rio.write_crs(CRS).rio.write_nodata(np.nan).rio.reproject("EPSG:4326")
    if float(da.y[0]) < float(da.y[-1]): da = da.isel(y=slice(None, None, -1))
    img = rgba_fn(da.values); return img, [[float(da.y.min()), float(da.x.min())], [float(da.y.max()), float(da.x.max())]]

def interactive_map(sid, B, cls, foot, sig_n, sig_m, zone):
    s = SITES[sid]; m = folium.Map(location=[s["lat"], s["lon"]], zoom_start=14, tiles="OpenStreetMap")
    def lc_rgba(v):
        out = np.zeros(v.shape + (4,), np.uint8)
        for k, c in LC_RGBA.items(): out[v == k] = c
        return out
    def cont_rgba(v, cmap, lim):
        out = np.zeros(v.shape + (4,), np.uint8); ok = np.isfinite(v)
        if ok.any(): out[ok] = (cmap(np.clip((v[ok] + lim) / (2 * lim), 0, 1)) * 255).astype(np.uint8); out[ok, 3] = 200
        return out
    def mask_rgba(v, rgb):
        out = np.zeros(v.shape + (4,), np.uint8); out[v > 0.5] = rgb + [150]; return out
    layers = [("ESA WorldCover land cover", _overlay(cls, B, lc_rgba), True), ("Lake usual footprint", _overlay(foot.astype("float32"), B, lambda v: mask_rgba(v, [0, 229, 255])), True)]
    for nm, arr in [("NDVI trend (significant pixels: brown = browning, green = greening)", sig_n), ("Vegetation water (NDWI) trend (significant pixels: brown = drier, blue = wetter)", sig_m)]:
        a = np.where(zone, arr, np.nan); lim = float(np.nanpercentile(np.abs(a), 98)) if np.isfinite(a).any() else 0.01; lim = lim if lim > 0 else 0.01
        layers.append((nm, _overlay(a, B, lambda v, c=(CM_GREEN if "NDVI" in nm else CM_BLUE), l=lim: cont_rgba(v, c, l)), False))
    for nm, (img, bounds), show in layers:
        folium.raster_layers.ImageOverlay(image=img, bounds=bounds, name=nm, opacity=0.9, interactive=False, show=show).add_to(m)
    folium.Marker([s["lat"], s["lon"]], popup=f"{s['name']} (pin)", tooltip=s["name"]).add_to(m); folium.LayerControl(collapsed=False).add_to(m)
    p = DIRS["supp"] / f"interactive_map_{sid}.html"; m.save(str(p)); return m, p

for sid, r in RES.items():
    if r.get("error") or r.get("prim") != "landsat": continue
    try:
        D = SITE_DATA[sid]["D"]; lc = SITE_DATA[sid]["LCM"].get(30); d = D[("landsat", VEG_SEASON)]; okY = d["qc"].loc[d["qc"].ok, "year"].values
        if lc is None or lc.get("source") != "worldcover" or len(okY) < 8: print(f"  vegetation map skipped for {SITES[sid]['name']} (needs WorldCover and 8 usable '{VEG_SEASON}' years)"); continue
        B = d["annual"].sel(year=okY); foot = r["fp"]["foot"]; res = 30
        zone = (ndimage.distance_transform_edt(~foot) * res <= RING_WIDE_M) & ~foot
        nat = _mask2d(lc["natural"], B).values; crop = _mask2d(lc["crop"], B).values; cls = np.asarray(lc["classes"].values, float)
        sl_n, p_n = pixel_trend(B["ndvi"].values, B.year.values); sl_m, p_m = pixel_trend(B["ndwi_moist"].values, B.year.values)
        zsel = zone & np.isfinite(sl_n)                          # FDR is applied to the pixels we actually look at (within 1.5 km of the lake)
        q_n = np.where(zsel, bh_fdr(np.where(zsel, p_n, np.nan)), np.nan); q_m = np.where(zsel, bh_fdr(np.where(zsel, p_m, np.nan)), np.nan)
        p_n, p_m = q_n, q_m                                        # from here on "p" means the FDR-adjusted q-value
        sig_n = np.where(p_n < FDR_Q, sl_n, np.nan); sig_m = np.where(p_m < FDR_Q, sl_m, np.nan)
        rows, ts = [], []
        for var, sl, pp in [("ndvi", sl_n, p_n), ("ndwi_moist", sl_m, p_m)]:
            for cname, msk in [("Natural vegetation", nat), ("Cropland", crop)]:
                sel = msk & zone & np.isfinite(sl); n = int(sel.sum())
                if n >= 10: rows.append({"site": sid, "variable": var, "land_cover": cname, "n_pixels": n, "pct_up": 100 * int(((sl > 0) & (pp < FDR_Q) & sel).sum()) / n,
                                         "pct_down": 100 * int(((sl < 0) & (pp < FDR_Q) & sel).sum()) / n, "mean_change_per_decade": 10 * float(np.nanmean(sl[sel]))})
        for y in B.year.values:
            a = B.sel(year=y); ok_ = ((a.n_obs > 0).values) & ~(a.water_freq >= 0.5).values
            for cname, msk in [("Natural vegetation", nat), ("Cropland", crop)]:
                sel = msk & zone & ok_
                if sel.sum() >= 10: ts.append({"year": int(y), "land_cover": cname, "ndvi": float(np.nanmean(a.ndvi.values[sel])), "moist": float(np.nanmean(a.ndwi_moist.values[sel])), "n": int(sel.sum())})
        ts = pd.DataFrame(ts); tab = pd.DataFrame(rows); VEG[sid] = dict(tab=tab, ts=ts)
        # ---------- figure ----------
        flip = B.y.values[0] < B.y.values[-1]; ori = (lambda a_: a_[::-1]) if flip else (lambda a_: a_)
        zo, fo = ori(zone), ori(foot); ry, rx = _win(zo | fo); crop_ = lambda a_: ori(a_)[ry, rx]
        ny, nx = crop_(zone).shape; ext = [0, nx * 0.03, 0, ny * 0.03]
        cat = np.where(foot, 3, np.where(crop, 2, np.where(nat, 1, 0))).astype(float)
        fig = plt.figure(figsize=(17.5, 10.6)); gs = fig.add_gridspec(2, 3, hspace=0.42, wspace=0.28); ax = [fig.add_subplot(gs[i, j]) for i in range(2) for j in range(3)]
        def draw_map(a_, arr, cmap, vmin, vmax, ttl):
            im = a_.imshow(np.ma.masked_invalid(crop_(arr)), cmap=cmap, vmin=vmin, vmax=vmax, extent=ext, origin="upper", interpolation="nearest"); a_.set_xlabel("km east"); a_.set_ylabel("km north"); a_.grid(False); a_.set_aspect("equal")
            xs_, ys_ = np.linspace(0, ext[1], nx), np.linspace(ext[3], 0, ny)
            a_.contour(xs_, ys_, crop_(foot).astype(float), levels=[0.5], colors="#00e5ff", linewidths=1.3); a_.contour(xs_, ys_, crop_(zone | foot).astype(float), levels=[0.5], colors="k", linewidths=0.9, linestyles="--"); a_.set_title(ttl, loc="left", fontsize=10.5); return im
        cm_ = ListedColormap(["#e0e0e0", "#2ca25f", "#f096ff", "#6baed6"]); cm_.set_bad("#ffffff"); draw_map(ax[0], cat, cm_, -0.5, 3.5, "A. The two land types we track")
        ax[0].legend(handles=[Patch(color="#2ca25f", label="Natural vegetation (climate signal)"), Patch(color="#f096ff", label="Cropland (farming signal)"), Patch(color="#6baed6", label="Lake (usual footprint)"), Patch(color="#e0e0e0", label="Other (built-up, bare)")],
                     loc="upper center", bbox_to_anchor=(0.5, -0.17), ncol=2, fontsize=8)
        for a_, arr, cmap, ttl, lab in [(ax[1], sig_n, CM_GREEN, f"B. Greenness trend (NDVI)", "NDVI change per decade"), (ax[2], sig_m, CM_BLUE, f"C. Canopy water trend (NDWI)", "NDWI change per decade")]:
            za = 10 * np.where(zone, arr, np.nan); lim = float(np.nanpercentile(np.abs(za), 98)) if np.isfinite(za).any() else 0.1; lim = lim if lim > 0 else 0.1
            cmx = cmap.copy(); cmx.set_bad("#e6e6e6"); im = draw_map(a_, za, cmx, -lim, lim, ttl); cb = fig.colorbar(im, ax=a_, shrink=0.75, pad=0.03); cb.set_label(lab + f"\nbrown = decline; grey = no trend after FDR (q<{FDR_Q})", fontsize=8)
        if len(tab):
            g_ = tab.copy(); g_["lab"] = g_.variable.map({"ndvi": "Green-\nness", "ndwi_moist": "Veg.\nwater"}) + "\n" + g_.land_cover.str.replace(" vegetation", "").str.lower()
            xx = np.arange(len(g_)); cu = [C_VEG if v == "ndvi" else C_WATER for v in g_.variable]
            ax[3].bar(xx - 0.2, g_.pct_up, 0.4, color=cu, edgecolor="#444444", label="significantly higher (greener / wetter)", zorder=3); ax[3].bar(xx + 0.2, g_.pct_down, 0.4, color=C_DRY, edgecolor="#444444", label="significantly lower (browner / drier)", zorder=3)
            ax[3].set_xticks(xx); ax[3].set_xticklabels(g_.lab, fontsize=8.5); ax[3].set_ylabel(f"% of pixels with a trend (FDR q < {FDR_Q})"); ax[3].set_ylim(0, max(float(max(g_.pct_up.max(), g_.pct_down.max())), 5) * 1.45); ax[3].legend(fontsize=8, loc="upper right")
        ax[3].set_title("D. Share of pixels changing, by land type", loc="left", fontsize=10.5)
        for a_, col, lab, ttl in [(ax[4], "ndvi", "Greenness (NDVI)", "E. Natural vegetation vs cropland: greenness"), (ax[5], "moist", "Vegetation water content (NDWI)", "F. Natural vegetation vs cropland: water content")]:
            if len(ts):
                for cname, colr, mk_ in [("Natural vegetation", C_VEG, "o"), ("Cropland", C_CROP, "s")]:
                    t_ = ts[ts.land_cover == cname]
                    if len(t_) < 5: continue
                    q = quick_trend(t_.year, t_[col]); ph = "no clear trend" if (q is None or q["trend"] == "no trend") else q["trend"]
                    gap_line(a_, t_.year, t_[col], colr, mk_, ms=4, lw=1.3, label=f"{cname} ({ph})")
                    if q is not None: sen_band(a_, t_.year, t_[col], colr, sig=q["trend"] != "no trend", band=q["trend"] != "no trend", lw=1.8)
                a_.axvline(SENSOR_BREAK - 0.5, c="k", ls="--", lw=0.8); a_.legend(fontsize=8, loc="upper left")
            a_.set_ylabel(lab); a_.set_xlabel("Year"); a_.set_title(ttl, loc="left", fontsize=10.5)
        finish(fig, f"S4_vegetation_trends_{sid}", f"{SITES[sid]['name']}: what grows around the lake, and is it greening or drying?",
               f"Within 1.5 km of the lake (black dashed line), {season_label(VEG_SEASON)}, Landsat 5/7 harmonised to Landsat 8/9. Maps colour only pixels whose trend survives a false-discovery-rate correction (q < {FDR_Q}); with thousands of pixels, about 5% would look 'significant' by chance without it. "
               "E-F: solid line + band = significant Sen trend with its 95% range, dashed = not significant; lines are broken where years are missing.",
               source="ESA WorldCover 2021, Landsat 5/7/8/9 C2-L2 (30 m)", bot_extra=0.2)
        try:
            m_, pth = interactive_map(sid, B, cls, foot, sig_n, sig_m, zone); VEG[sid]["map"] = pth; print(f"  interactive map saved: {pth}")
            if sid == list(RES)[0]: display(m_)
        except Exception as e: print(f"  interactive map skipped for {SITES[sid]['name']}: {type(e).__name__}: {e}")
        # ---------- second water index (MNDWI) as a cross-check of the NDWI water trend ----------
        dd = D[("landsat", WATER_SEASON)]; A = dd["annual"]; st = r["stats"][("landsat", WATER_SEASON)]; okS = st[st.ok_site].year.values
        mser = pd.DataFrame({"year": okS, "m": [100 * float((((A.sel(year=y).mndwi > 0) & (A.sel(year=y).n_obs > 0)).values & foot).sum()) / max(int(foot.sum()), 1) for y in okS]})
        tm = trend_table(mser, {"m": "MNDWI water cover"}, min_n=8); xn = pct_row(r["trend"].get(WATER_SEASON), "water_pct_foot")
        if len(tm) and xn is not None:
            tm = harden_trends(tm).iloc[0]; MNDWI_ROWS.append({"site": sid, "ndwi_pts_per_decade": 10 * xn.sen_slope_per_year, "ndwi_trend": xn.mk_trend, "ndwi_p": xn.mk_p,
                                                             "mndwi_pts_per_decade": 10 * tm.sen_slope_per_year, "mndwi_trend": tm.mk_trend, "mndwi_p": tm.mk_p,
                                                             "series_rho": float(stats.spearmanr(st[st.ok_site].water_pct_foot.values, mser.m.values)[0]) if mser.m.nunique() > 2 and st[st.ok_site].water_pct_foot.nunique() > 2 else np.nan})
    except Exception as e:
        print(f"vegetation analysis skipped for {SITES[sid]['name']}: {type(e).__name__}: {e}")

if VEG:
    allt = pd.concat([v["tab"] for v in VEG.values() if len(v["tab"])], ignore_index=True); VEG_SHEETS["veg_trend_by_landcover"] = allt
    if len(allt):
        for var, nm in [("ndvi", "greenness"), ("ndwi_moist", "vegetation water content")]:
            nn = allt[(allt.variable == var) & (allt.land_cover == "Natural vegetation")]; cc = allt[(allt.variable == var) & (allt.land_cover == "Cropland")]
            if len(nn): VEG_LINES.append(f"Vegetation around the dams: natural-vegetation {nm} is rising near {int((nn.mean_change_per_decade > 0).sum())} of {len(nn)} dams and falling near {int((nn.mean_change_per_decade < 0).sum())}" + (f"; on cropland it is rising near {int((cc.mean_change_per_decade > 0).sum())} of {len(cc)} dams." if len(cc) else "."))
    sites_v = list(VEG); nm = [SITES[s]["name"].replace(" Dam", "") for s in sites_v]; xx = np.arange(len(sites_v))
    fig, axs = plt.subplots(1, 3, figsize=(18, 5.6)); bottom = np.zeros(len(sites_v)); shares = {s: (RES[s].get("landuse") or {}).get("share", {}) for s in sites_v}
    for k, lab in LC_NAMES.items():
        v = np.array([shares[s].get(lab, 0) for s in sites_v], float)
        if v.sum() > 0: axs[0].bar(xx, v, bottom=bottom, color=LC_COLORS[k], edgecolor="white", label=lab, zorder=3); bottom += v
    axs[0].set_xticks(xx); axs[0].set_xticklabels(nm); axs[0].set_ylabel("% of land within 1.5 km of the lake"); axs[0].set_title("A. What surrounds each dam?", loc="left", fontsize=10.5); axs[0].legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=3)
    for j, (var, ttl, unit) in enumerate([("ndvi", "B. Greenness change per decade", "NDVI per decade")]):
        for k, (cname, colr) in enumerate([("Natural vegetation", C_VEG), ("Cropland", C_CROP)]):
            v = [float(allt[(allt.site == s) & (allt.variable == var) & (allt.land_cover == cname)].mean_change_per_decade.iloc[0]) if len(allt[(allt.site == s) & (allt.variable == var) & (allt.land_cover == cname)]) else np.nan for s in sites_v]
            axs[1].bar(xx + (k - 0.5) * 0.36, v, 0.36, color=colr, edgecolor="#444444", label=cname, zorder=3)
        axs[1].axhline(0, c="k", lw=0.8); axs[1].set_xticks(xx); axs[1].set_xticklabels(nm); axs[1].set_ylabel(unit); axs[1].set_title(ttl, loc="left", fontsize=10.5); axs[1].legend(fontsize=8.5, loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=2)
    mt = pd.DataFrame(MNDWI_ROWS)
    if len(mt):
        VEG_SHEETS["water_index_crosscheck"] = mt; xm = np.arange(len(mt)); nmm = [SITES[s]["name"].replace(" Dam", "") for s in mt.site]
        for k, (c_, col, lab) in enumerate([("ndwi", C_WATER, "NDWI water mask (used in the analysis)"), ("mndwi", C_WATER_L, "MNDWI water mask (cross-check)")]):
            axs[2].bar(xm + (k - 0.5) * 0.36, mt[f"{c_}_pts_per_decade"], 0.36, color=col, edgecolor="#444444", label=lab, zorder=3, hatch="" if k == 0 else "//")
        axs[2].axhline(0, c="k", lw=0.8); axs[2].set_xticks(xm); axs[2].set_xticklabels(nmm); axs[2].set_ylabel("Water cover change\n(points per decade)"); axs[2].legend(fontsize=8.5, loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=1)
        same = int((np.sign(mt.ndwi_pts_per_decade) == np.sign(mt.mndwi_pts_per_decade)).sum()); VEG_LINES.append(f"Second water index check: MNDWI gives the same direction of change in water cover as NDWI at {same} of {len(mt)} dams (series correlation {mt.series_rho.min():.2f} to {mt.series_rho.max():.2f}).")
    axs[2].set_title("C. Does a second water index agree?", loc="left", fontsize=10.5)
    for a_ in axs: a_.grid(True, axis="y"); a_.grid(False, axis="x")
    finish(fig, "S5_vegetation_summary", "Land around the Copperbelt dams, and how it is changing",
           "A: land cover within 1.5 km of each dam. B: greenness trend of natural vegetation vs cropland (only natural vegetation carries a climate signal). C: water-cover trend measured with two different water indices.",
           source="ESA WorldCover 2021, Landsat 5/7/8/9 C2-L2 (30 m)", bot_extra=0.5)
ADV_LINES.extend(VEG_LINES); ADV_SHEETS.update(VEG_SHEETS)


## Step 2d. Canopy water stress maps (core figures `07_canopy_stress_*`)
Where is the vegetation around each dam under water stress? Each pixel's canopy NDWI (and NDVI) is scored 0-100 against its own 1996-2025 record, for the driest year and the latest year, with the share of vegetation in drought condition each year, a rapid spatial assessment of water stress in vegetation canopies. Supplementary `S6` summarises all dams on one timeline.

In [ ]:
# Cell 9d: DROUGHT-CONDITION MAPS. Vegetation Condition Index (greenness) and Moisture Condition Index (canopy water content): where does each pixel sit between ITS OWN driest and wettest year (0 = worst on record, 100 = best)?
COND_CLASSES = [(0, 10, "Extreme drought", "#7f3b08"), (10, 20, "Severe drought", "#b35806"), (20, 35, "Moderate drought", "#e08214"), (35, 50, "Mild / abnormally dry", "#fdb863"), (50, 101, "No drought", "#a6dba0")]
COND_CM = ListedColormap([c[3] for c in COND_CLASSES]); COND_BOUNDS = [0, 10, 20, 35, 50, 100.01]
from matplotlib.colors import BoundaryNorm
COND, COND_ROWS = {}, []

def cond_index(cube):
    """0-100 position between the pixel's own 5th and 95th percentile over the usable years (a robust version of min/max, so one cloud-shadow year cannot set the scale)."""
    lo, hi = np.nanpercentile(cube, 5, axis=0), np.nanpercentile(cube, 95, axis=0)
    with np.errstate(invalid="ignore", divide="ignore"): c = 100 * (cube - lo) / (hi - lo)
    c[:, (hi - lo) < 1e-4] = np.nan
    return np.clip(c, 0, 100)

for sid, r in RES.items():
    if r.get("error") or r.get("prim") != "landsat": continue
    try:
        D = SITE_DATA[sid]["D"]; lc = SITE_DATA[sid]["LCM"].get(30); d = D[("landsat", VEG_SEASON)]; okY = d["qc"].loc[d["qc"].ok, "year"].values
        if len(okY) < 8: continue
        B = d["annual"].sel(year=okY); foot = r["fp"]["foot"]; zone = (ndimage.distance_transform_edt(~foot) * 30 <= RING_WIDE_M) & ~foot
        if lc is not None and lc.get("source") == "worldcover": veg = _mask2d(lc["natural"], B).values & zone
        else: veg = ((B.ndvi > NDVI_VEG_THR).mean("year") >= 0.5).values & zone
        if veg.sum() < 20: continue
        cubes = {"MCI": cond_index(B["ndwi_moist"].values), "VCI": cond_index(B["ndvi"].values)}
        pct = {k: np.array([100 * float(((c[i] < 35) & veg & np.isfinite(c[i])).sum()) / max(int((veg & np.isfinite(c[i])).sum()), 1) for i in range(len(okY))]) for k, c in cubes.items()}
        yrs = B.year.values.astype(int)
        for k in cubes:
            for y, v in zip(yrs, pct[k]): COND_ROWS.append({"site": sid, "index": k, "year": int(y), "pct_natural_veg_in_drought": v})
        iw = int(np.argmax(pct["MCI"])); il = len(yrs) - 1; COND[sid] = dict(yrs=yrs, pct=pct, worst=int(yrs[iw]), latest=int(yrs[il]))
        flip = B.y.values[0] < B.y.values[-1]; ori = (lambda a_: a_[::-1]) if flip else (lambda a_: a_); ry, rx = _win(ori(zone) | ori(foot)); crop_ = lambda a_: ori(a_)[ry, rx]
        ny, nx = crop_(zone).shape; ext = [0, nx * 0.03, 0, ny * 0.03]; xs_, ys_ = np.linspace(0, ext[1], nx), np.linspace(ext[3], 0, ny)
        fig = plt.figure(figsize=(17, 9.4)); gs = fig.add_gridspec(2, 3, width_ratios=[1, 1, 1.25], hspace=0.42, wspace=0.22); axm = [[fig.add_subplot(gs[i, j]) for j in range(2)] for i in range(2)]; axt = fig.add_subplot(gs[:, 2])
        for i, (k, nm) in enumerate([("MCI", "Vegetation water content"), ("VCI", "Greenness")]):
            for j, (idx, lab) in enumerate([(iw, f"driest year: {yrs[iw]}"), (il, f"latest year: {yrs[il]}")]):
                a_ = axm[i][j]; arr = np.where(veg, cubes[k][idx], np.nan); a_.imshow(np.ma.masked_invalid(crop_(arr)), cmap=COND_CM, norm=BoundaryNorm(COND_BOUNDS, COND_CM.N), extent=ext, origin="upper", interpolation="nearest")
                a_.set_facecolor("#eeeeee"); a_.contour(xs_, ys_, crop_(foot).astype(float), levels=[0.5], colors="#00e5ff", linewidths=1.3); a_.grid(False); a_.set_aspect("equal"); a_.set_xlabel("km east"); a_.set_ylabel("km north")
                a_.set_title(f"{nm}, {lab}\n{pct[k][idx]:.0f}% of natural vegetation in drought", loc="left", fontsize=9.5)
        shade_eras(axt); gap_line(axt, yrs, pct["MCI"], C_MOIST, "o", ms=4, lw=1.6, label="Canopy water content (MCI, from NDWI)"); gap_line(axt, yrs, pct["VCI"], C_VEG, "s", ms=4, lw=1.3, label="Greenness (VCI, from NDVI)")
        drv_ = r.get("drivers")
        if drv_ is not None:
            for y in drv_.index[(drv_[DROUGHT_SPI] < STRESS_Z)]:
                if yrs.min() <= y <= yrs.max(): axt.axvspan(y - 0.5, y + 0.5, color=C_DRY_L, alpha=0.45, lw=0, zorder=0)
        axt.set_ylim(0, 105); axt.set_ylabel("% of natural vegetation in drought condition\n(index below 35)"); axt.set_xlabel("Year"); axt.set_title("Drought condition through time", loc="left", fontsize=10.5); axt.legend(fontsize=8.5, loc="upper right")
        hs = [Patch(color=c[3], label=f"{c[2]} (index {c[0]}-{min(c[1], 100)})") for c in COND_CLASSES] + [Line2D([0], [0], color="#00e5ff", lw=2, label="Lake"), Patch(color="#eeeeee", label="Not natural vegetation"), Patch(color=C_DRY_L, alpha=0.6, label="Rainfall drought year (SPI-6 < -1)")]
        finish(fig, f"07_canopy_stress_{sid}", f"{SITES[sid]['name']}: how dry is the vegetation around the lake?",
               f"Each pixel is scored 0 to 100 between its own 5th and 95th percentile year ({season_label(VEG_SEASON)}, within 1.5 km; Landsat 5/7 harmonised to 8/9 first, otherwise every pre-2013 year would look like drought). Below 35 counts as drought condition. Maps: driest year on record vs the latest year.",
               source="Landsat 5/7/8/9 C2-L2 (30 m), ESA WorldCover 2021", handles=hs, legend_cols=4, bot_extra=0.75)
    except Exception as e:
        print(f"condition maps skipped for {SITES[sid]['name']}: {type(e).__name__}: {e}")

if COND:
    ct = pd.DataFrame(COND_ROWS); ADV_SHEETS["condition_timeline"] = ct; sites_c = list(COND); allyrs = list(range(int(ct.year.min()), int(ct.year.max()) + 1))     # continuous time axis: missing years stay as grey columns
    M = np.full((len(sites_c), len(allyrs)), np.nan)
    for i, s_ in enumerate(sites_c):
        for y, v in ct[(ct.site == s_) & (ct["index"] == "MCI")][["year", "pct_natural_veg_in_drought"]].values: M[i, allyrs.index(int(y))] = v
    fig, ax = plt.subplots(figsize=(15, 0.62 * len(sites_c) + 3.4)); cm_ = mpl.colormaps["YlOrBr"].copy(); cm_.set_bad("#e6e6e6")
    im = ax.imshow(np.ma.masked_invalid(M), cmap=cm_, vmin=0, vmax=100, aspect="auto")
    ax.set_yticks(range(len(sites_c))); ax.set_yticklabels([SITES[s]["name"] for s in sites_c]); ax.set_xticks(range(0, len(allyrs), 2)); ax.set_xticklabels(allyrs[::2], fontsize=8.5); ax.set_xlabel("Year (grey = no usable image)"); ax.grid(False)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            if np.isfinite(M[i, j]) and M[i, j] >= 40: ax.text(j, i, f"{M[i, j]:.0f}", ha="center", va="center", fontsize=7.5, color="white" if M[i, j] > 65 else "black")
    cb = fig.colorbar(im, ax=ax, shrink=0.8, pad=0.02); cb.set_label("% of natural vegetation in drought condition")
    finish(fig, "S6_drought_condition_timeline", "Which years were worst for the vegetation around each dam?",
           f"Share of natural vegetation within 1.5 km of each dam whose canopy water content (MCI, {season_label(VEG_SEASON)}) fell in drought condition. Dark cells are years when most of the vegetation around that dam was in drought. Grey = year not usable.",
           source="Landsat 5/7/8/9 C2-L2 (30 m)")
    lat_ = [COND[s]["pct"]["MCI"][-1] for s in COND]; ADV_LINES.append(f"Vegetation condition: in {COND[sites_c[0]]['latest']}, {min(lat_):.0f}% to {max(lat_):.0f}% of the natural vegetation around the dams was in drought condition on the canopy-moisture index; the worst years were " + ", ".join(f"{SITES[s]['name'].replace(' Dam', '')} {COND[s]['worst']}" for s in sites_c) + ".")


## Step 4. One-page site sheets (figures `05_site_*`)
For each dam: the satellite view, where water was lost or gained across the three decades 1996-2005, 2006-2015 and 2016-2025 (decade means are drawn on the water-cover panel), water cover in both seasons with its trend, canopy NDWI of the shoreline vegetation, and the rainy-season drought index.

In [ ]:
# Cell 11: ONE-PAGE "SITE SHEET" FOR EACH WATER BODY (picture, water over time, shoreline vegetation, rainfall, verdict)
def site_headline(sid, r):
    s = SITES[sid]
    if s["kind"] == "new_reservoir":
        st = r["stats"].get((r["prim"], WATER_SEASON)); ok = st[st.ok_site]
        fy = ok.loc[ok.water_pct_foot >= 50, "year"]
        return f"{s['name']}: new reservoir, filling since about {int(fy.min())}" if len(fy) else f"{s['name']}: new reservoir, not yet filled in the record"
    x = pct_row(r["trend"].get(WATER_SEASON), "water_pct_foot")
    if x is None: return f"{s['name']}: not enough clear years to test a trend"
    hedge = {"High": "", "Medium": " (medium reliability)", "Low": " - but measurement reliability is LOW, so treat as indicative only"}.get(r["rel"], "")
    if not x.get("full_record", True): hedge += f" [partial record: first usable year {int(x.first_year)}]"
    if x.mk_trend != "no trend":
        verb = "shrinking" if x.mk_trend == "decreasing" else "growing"
        return f"{s['name']}: end-of-dry-season water cover is {verb} ({x.sen_slope_per_year * 10:+.1f} points per decade){hedge}"
    return f"{s['name']}: no clear long-term change in end-of-dry-season water cover{hedge if r['rel'] == 'Low' else ''}"

def drought_years(r):
    drv = r.get("drivers")
    if drv is None: return []
    return [int(y) for y in drv.index[drv[DROUGHT_SPI] < STRESS_Z] if LANDSAT_YEARS[0] <= y <= LANDSAT_YEARS[-1]]

def site_sheet(sid):
    r = RES[sid]; s = SITES[sid]; sc = SCREEN.get(sid); prim = r["prim"]
    fig = plt.figure(figsize=(16.5, 10.6)); gs = fig.add_gridspec(1, 2, width_ratios=[1, 1.75], wspace=0.16)
    gl = gs[0].subgridspec(2, 1, hspace=0.45); gr = gs[1].subgridspec(3, 1, hspace=0.78)
    a0 = fig.add_subplot(gl[0])
    if sc is not None:
        a0.imshow(sc["rgb"], extent=sc["extent_km"], origin="upper")
        if sc["comp"].any():
            ex = sc["extent_km"]; ny, nx = sc["comp"].shape
            a0.contour(np.linspace(ex[0], ex[1], nx), np.linspace(ex[3], ex[2], ny), sc["comp"].astype(float), levels=[0.5], colors="#00e5ff", linewidths=1.2)
        a0.plot(*sc["pin_km"], marker="*", ms=12, mfc="#ffd400", mec="black", ls="none")
    a0.set_xlabel("km east"); a0.set_ylabel("km north"); a0.grid(False); a0.set_aspect("equal"); a0.set_title("A. Satellite view, end of dry season 2024-25", loc="left", fontsize=10.5)
    a1 = fig.add_subplot(gl[1])
    if r.get("change") is not None:
        CH = ["#f2f2f2", "#9ecae1", "#8c510a", "#f4a261", "#2166ac", "#08306b", "#c2a5cf"]
        CH_LAB = ["", "Water in all 3 decades", f"Lost by {DECADE_LABEL[1]}", f"Lost by {DECADE_LABEL[2]}", f"New by {DECADE_LABEL[1]}", f"New by {DECADE_LABEL[2]}", "Intermittent (on/off)"]
        dz = r.get("dam_zone"); ca = {k: float(((r["change"] == k) & dz).sum()) * 0.09 for k in range(1, 7)}        # this dam only
        dwh = r.get("decade_water_ha", {})
        show_map(a1, r["change"], ListedColormap(CH), -0.5, 6.5, title="B. Usual dry-season water at this dam\n" + " | ".join(f"{d}: {v:.0f} ha" for d, v in dwh.items()))
        fz = r["foot_da"].values; fz = fz[::-1] if float(r["foot_da"].y[0]) < float(r["foot_da"].y[-1]) else fz; ex_ = km_extent(r["foot_da"])
        a1.contour(np.linspace(ex_[0], ex_[1], fz.shape[1]), np.linspace(ex_[3], ex_[2], fz.shape[0]), fz, levels=[0.5], colors="k", linewidths=0.7, linestyles="--")
        map_handles = [Patch(color=CH[k], label=f"{CH_LAB[k]} ({ca[k]:.0f} ha at this dam)") for k in range(1, 7) if ca[k] > 0] + \
                      [Line2D([0], [0], color="k", lw=0.8, ls="--", label="dam's usual outline (hectares count only this dam; other ponds are shown but not counted)")]
    else:
        map_handles = []
        im = show_map(a1, r["occ_da"] * 100, "Blues", 0, 100, title="B. How often is each spot water?"); fig.colorbar(im, ax=a1, shrink=0.75).set_label("% of years")
    yrs = (LANDSAT_YEARS[0] - 0.8, LANDSAT_YEARS[-1] + 0.8) if prim == "landsat" else (S2_YEARS[0] - 0.8, S2_YEARS[-1] + 0.8)
    ax1 = fig.add_subplot(gr[0]); ax2 = fig.add_subplot(gr[1], sharex=ax1); ax3 = fig.add_subplot(gr[2], sharex=ax1)
    dys = drought_years(r)
    for a in (ax1, ax2, ax3):
        for y in dys: a.axvspan(y - 0.5, y + 0.5, color=C_DRY_L, alpha=0.40, lw=0, zorder=0)
    for sk, col, mk_ in [("mam", C_RAINS, "o"), ("dry", C_DRYS, "s")]:
        st = r["stats"].get((prim, sk))
        if st is None: continue
        ok = st[st.ok_site & st.water_pct_foot.notna()]; gap_line(ax1, ok.year, ok.water_pct_foot, col, mk_, ms=5, lw=1.5, label=season_label(sk))
        x = pct_row(r["trend"].get(sk), "water_pct_foot")
        if x is not None and len(ok) >= 8: sen_band(ax1, ok.year, ok.water_pct_foot, col, sig=x.mk_trend != "no trend", band=x.mk_trend != "no trend")
        if "phys_fail" in st:
            f_ = st[st.ok_site & st.phys_fail]
            if len(f_): ax1.plot(f_.year, np.full(len(f_), 4.0), "x", color=C_HOT, ms=6, mew=1.6, ls="none", label="end-of-rains value removed (physics check)", zorder=6)
    ax1.plot([], [], color="#e08214", lw=3, label="decade mean, end of dry season")
    st_d = r["stats"].get((prim, WATER_SEASON))
    if st_d is not None:                                           # decade means of the end-of-dry-season water cover (horizontal bars, labelled)
        okd = st_d[st_d.ok_site & st_d.water_pct_foot.notna()]
        for (w0, w1), lab in zip(DECADES, DECADE_LABEL):
            v = okd[(okd.year >= w0) & (okd.year <= w1)].water_pct_foot
            if len(v) >= 3:
                ax1.plot([w0 - 0.4, w1 + 0.4], [v.mean(), v.mean()], color="#e08214", lw=3, alpha=0.85, zorder=6, solid_capstyle="butt")
                ax1.text(w0 + 0.2, v.mean() - 3, f"{lab}: {v.mean():.0f}%", ha="left", va="top", fontsize=7.8, color="#b35806", fontweight="bold", zorder=8,
                         bbox=dict(fc="white", ec="none", alpha=0.8, pad=0.8))
    for w0, _ in DECADES[1:]: ax1.axvline(w0 - 0.5, color="#bbbbbb", lw=0.8, ls="--", zorder=1)
    ax1.set_ylim(0, 105); ax1.set_xlim(*yrs); ax1.set_ylabel("% of usual footprint\ncovered by water")
    ax1.legend(loc="upper left", bbox_to_anchor=(0.0, -0.13), ncol=3, fontsize=8, frameon=False)
    ax1.set_title("C. Share of the lake's usual surface under water (open-water NDWI)", loc="left", fontsize=10.5)
    xd = pct_row(r["trend"].get(WATER_SEASON), "water_pct_foot")
    if xd is not None and xd.pettitt_p < 0.05:
        ax1.axvline(xd.pettitt_change_year - 0.5, color="#7b3294", ls=":", lw=1.6); ax1.text(xd.pettitt_change_year - 0.7, 104, f"step change ~{int(xd.pettitt_change_year)}", color="#7b3294", fontsize=8, va="top", ha="right", bbox=dict(fc="white", ec="none", alpha=0.8, pad=0.5), zorder=8)
    if xd is not None:
        ax1.text(0.99, 0.04, f"End of dry season: {phrase_p(xd.mk_p, xd.mk_trend)}\n{span_label(xd)}", transform=ax1.transAxes, ha="right", va="bottom", fontsize=8.6, bbox=dict(fc="white", ec="#cccccc", alpha=0.93, boxstyle="round,pad=0.3"), zorder=7)
    st = r["stats"].get((prim, VEG_SEASON))
    if st is not None and st.ring_moist_mean_z.notna().any():
        ok = st[st.ok_site & st.ring_moist_mean_z.notna()]
        anomaly_bars(ax2, ok.year, ok.ring_moist_mean_z.values, "moist"); q = quick_trend(ok.year, ok.ring_moist_mean_z)
        ax2.axhline(STRESS_Z, color=C_DRY, ls=":", lw=1.1)
        if q: ax2.text(0.01, 0.95, phrase_p(q["p"], q["trend"]), transform=ax2.transAxes, va="top", fontsize=8.6, bbox=dict(fc="white", ec="#cccccc", alpha=0.93, boxstyle="round,pad=0.3"))
    lim2 = max(2.5, float(np.nanmax(np.abs(ax2.get_ylim())))); ax2.set_ylim(-lim2, lim2)
    ax2.set_ylabel("Anomaly (SD from\nits own average)"); ax2.set_title(f"D. Canopy water content (NDWI) of shoreline vegetation within {RING_M} m, {season_label(VEG_SEASON)}", loc="left", fontsize=10.5)
    drv = r.get("drivers")
    if drv is not None:
        sp = drv[DROUGHT_SPI].reindex(range(int(yrs[0] + 0.8), int(yrs[1] - 0.8) + 1)).dropna()
        ax3.bar(sp.index, sp.values, color=np.where(sp.values < 0, C_DRY, C_WATER), width=0.8, zorder=3); ax3.axhline(0, c="k", lw=0.8); ax3.axhline(STRESS_Z, color=C_DRY, ls=":", lw=1.2)
        lim3 = max(2.5, float(np.nanmax(np.abs(sp.values))) * 1.1); ax3.set_ylim(-lim3, lim3)
    else:
        ax3.text(0.5, 0.5, "climate data unavailable", transform=ax3.transAxes, ha="center")
    ax3.set_ylabel("SPI-6 (Dec-May)"); ax3.set_xlabel("Year")
    ax3.set_title("E. Rainy-season drought index (brown = drier than normal; shaded columns = drought years, SPI-6 < -1)", loc="left", fontsize=10.5)
    rr = r.get("rain", {}).get(WATER_SEASON, {})
    extra = (f" Water cover vs wet-season rain: rho = {rr['rain_rho']:+.2f} ({fmt_p(rr['rain_p'])}); trend left after removing rainfall: {rr.get('resid_trend', 'n/a')} ({fmt_p(rr.get('resid_p', np.nan)) if np.isfinite(rr.get('resid_p', np.nan)) else 'n/a'})." if rr else "")
    nf = r.get("mam_fail", {}).get(prim, (0, 0))
    extra += f" {nf[0]} of {nf[1]} end-of-rains water values failed the physics check and were removed." if nf[0] else ""
    finish(fig, f"05_site_{sid}", site_headline(sid, r),
           f"Measurement reliability: {r['rel']}" + (f" ({'; '.join(r['why'])})" if r["why"] else "") + "." + extra + " Solid line + band = significant Sen trend (95% range); dashed = not significant.",
           source="Landsat 5/7/8/9 C2-L2 (30 m), Sentinel-2 L2A, ESA WorldCover; " + str(r.get("climate_src")), handles=map_handles + [Patch(color=C_DRY_L, alpha=0.6, label="drought year (SPI-6 < -1)")], legend_cols=4, bot_extra=0.35, adjust=dict(left=0.05, right=0.985))       # nested grids: explicit margins (tight_layout cannot place them)

for sid in RES:
    if not RES[sid].get("error"):
        try: site_sheet(sid)
        except Exception as e: print(f"site sheet failed for {sid}: {type(e).__name__}: {e}")


# ---------- land cover around each lake: natural vegetation vs farmland ----------
def landuse_sheet(sid):
    r = RES[sid]; lu = r.get("landuse")
    if lu is None: return
    s = SITES[sid]; fig, ax = plt.subplots(1, 3, figsize=(17, 5.6), gridspec_kw={"width_ratios": [1.2, 1, 1.25]})
    img = np.ones(lu["cls"].shape + (3,))
    for k, c in LC_COLORS.items(): img[lu["cls"] == k] = [int(c[i:i + 2], 16) / 255 for i in (1, 3, 5)]
    img[~lu["zone"] & ~r["fp"]["foot"]] = 0.5 * img[~lu["zone"] & ~r["fp"]["foot"]] + 0.5
    ny, nx = lu["cls"].shape; ex = [0, nx * 0.03, 0, ny * 0.03]
    ax[0].imshow(img, extent=ex, origin="upper" if lu["y"][0] > lu["y"][-1] else "lower")
    ax[0].contour(np.linspace(0, ex[1], nx), np.linspace(ex[3], 0, ny) if lu["y"][0] > lu["y"][-1] else np.linspace(0, ex[3], ny), r["fp"]["foot"].astype(float), levels=[0.5], colors="#00e5ff", linewidths=1.4)
    ax[0].set_xlabel("km east"); ax[0].set_ylabel("km north"); ax[0].grid(False); ax[0].set_aspect("equal"); ax[0].set_title("A. Land cover (bright = within 1.5 km of the lake)", loc="left", fontsize=10.5)
    items_ = sorted(lu["share"].items(), key=lambda z: z[1]); cols = {v: LC_COLORS[k] for k, v in LC_NAMES.items()}
    ax[1].barh([k for k, _ in items_], [v for _, v in items_], color=[cols[k] for k, _ in items_], edgecolor="#555555", lw=0.5)
    for i_, (k, v) in enumerate(items_): ax[1].text(v + 0.8, i_, f"{v:.0f}%", va="center", fontsize=9)
    ax[1].set_xlim(0, max(v for _, v in items_) * 1.2); ax[1].set_xlabel("% of the land within 1.5 km"); ax[1].set_title("B. What surrounds the lake?", loc="left", fontsize=10.5); ax[1].grid(True, axis="x"); ax[1].grid(False, axis="y")
    t = lu["table"][lu["table"].variable == "ndvi"]
    if len(t):
        x = np.arange(len(t)); colv = [C_VEG if n == "Natural vegetation" else C_CROP for n in t.land_cover]
        ax[2].bar(x, t.mean_change_per_decade, color=colv, edgecolor="#444444", width=0.55, zorder=3); ax[2].axhline(0, c="k", lw=0.8)
        ax[2].set_xticks(x); ax[2].set_xticklabels([f"{n}\n({int(p):,} pixels)" for n, p in zip(t.land_cover, t.n_pixels)])
        for xi, v in zip(x, t.mean_change_per_decade): ax[2].text(xi, v + (0.003 if v >= 0 else -0.003), f"{v:+.3f}", ha="center", va="bottom" if v >= 0 else "top", fontsize=9.5)
        ax[2].set_ylabel("Greenness (NDVI) change per decade"); ax[2].set_title(f"C. Is the greening on farmland or on natural vegetation? ({season_label(VEG_SEASON)})", loc="left", fontsize=10.5)
        ax[2].margins(y=0.25)
    else:
        ax[2].text(0.5, 0.5, "too few pixels of each type", transform=ax[2].transAxes, ha="center"); ax[2].axis("off")
    hs = [Patch(color=LC_COLORS[k], label=LC_NAMES[k]) for k in LC_NAMES if LC_NAMES[k] in lu["share"]] + [Line2D([0], [0], color="#00e5ff", lw=2, label="Lake footprint")]
    finish(fig, f"S3_landuse_{sid}", f"{s['name']}: natural vegetation and farmland around the lake",
           "Farming changes greenness on its own, so natural vegetation is the climate signal and cropland is shown separately. Bars: average greenness change per decade of each land type.",
           source="ESA WorldCover 2021, Landsat 5/7/8/9 C2-L2 (30 m)", handles=hs, legend_cols=5)

for sid in RES:
    if not RES[sid].get("error"):
        try: landuse_sheet(sid)
        except Exception as e: print(f"land-use sheet failed for {sid}: {type(e).__name__}: {e}")


## Step 4b. Early warning: does canopy NDWI drop before greenness?
Core figure `08_early_warning_yearly` tests the claim that falling NDWI warns of plant water deficit before visible chlorosis (NDVI decline). Supplementary: `S7` drought-year panel, `S8` NDWI threshold sensitivity, `S9` climate-driver correlations, plus GeoTIFFs for GIS.

In [ ]:
# Cell 11b: EXTRA TESTS. (1) Is vegetation NDWI an EARLY warning? (2) What happened in each drought year? (3) Is the water area robust to the threshold? (4) GeoTIFFs for GIS.
import rioxarray  # adds the .rio accessor used for GeoTIFF export
from matplotlib.colors import TwoSlopeNorm
EXTRA_LINES = ([EXTRA_AV_LINE] if globals().get("EXTRA_AV_LINE") else []) + list(globals().get("ADV_LINES", [])) + ([FDR_LINE] if globals().get("FDR_LINE") else [])
EXTRA_SHEETS = dict(globals().get("ADV_SHEETS", {})); EXTRA_SHEETS.update(globals().get("ADV_SHEETS_AV", {}))

# ---------- 1. EARLY WARNING: in drought years, does shoreline-vegetation NDWI fall while NDVI (greenness) is still normal? ----------
# A drought year = the Dec-May rain index (SPI-6 at end of rains) is below STRESS_Z (-1). "Stress" = anomaly below STRESS_Z standard deviations of the site's own record.
EW = {}; ew_rows = []
for sid, r in RES.items():
    if r.get("error") or r.get("drivers") is None or r.get("prim") != "landsat": continue
    m = r["stats"].get(("landsat", VEG_SEASON))
    if m is None: continue
    t = m[m.ok_site][["year", "ring_moist_mean_z", "ring_ndvi_mean_z"]].dropna().merge(r["drivers"].reset_index()[["year", "spi6_mam"]], on="year").dropna()
    if len(t) < 8: continue
    t = t.assign(drought=t.spi6_mam < STRESS_Z, moist_low=t.ring_moist_mean_z < STRESS_Z, ndvi_low=t.ring_ndvi_mean_z < STRESS_Z)
    dy = t[t.drought]
    rho_m = stats.spearmanr(t.ring_moist_mean_z, t.spi6_mam)[0]; rho_n = stats.spearmanr(t.ring_ndvi_mean_z, t.spi6_mam)[0]
    EW[sid] = dict(t=t, n_drought=len(dy), early=int((dy.moist_low & ~dy.ndvi_low).sum()), both=int((dy.moist_low & dy.ndvi_low).sum()),
                   ndvi_only=int((~dy.moist_low & dy.ndvi_low).sum()), neither=int((~dy.moist_low & ~dy.ndvi_low).sum()), rho_m=rho_m, rho_n=rho_n)
    e = EW[sid]; ew_rows.append({"site": sid, "name": SITES[sid]["name"], "drought_years": e["n_drought"], "NDWI_low_NDVI_normal": e["early"], "both_low": e["both"],
                                 "NDVI_low_only": e["ndvi_only"], "neither_low": e["neither"], "rain_vs_NDWI_rho": round(rho_m, 2), "rain_vs_NDVI_rho": round(rho_n, 2),
                                 "mean_NDWI_z_drought": round(float(dy.ring_moist_mean_z.mean()), 2) if len(dy) else np.nan, "mean_NDVI_z_drought": round(float(dy.ring_ndvi_mean_z.mean()), 2) if len(dy) else np.nan})
ew_tab = pd.DataFrame(ew_rows); EXTRA_SHEETS["early_warning"] = ew_tab
if len(ew_tab):
    display(ew_tab)
    N = int(ew_tab.drought_years.sum()); a = int(ew_tab.NDWI_low_NDVI_normal.sum() + ew_tab.both_low.sum()); b = int(ew_tab.NDVI_low_only.sum() + ew_tab.both_low.sum()); c_ = int(ew_tab.NDWI_low_NDVI_normal.sum())
    if N >= 4:
        verdict_ew = ("consistent with NDWI acting as an early indicator" if (a > b and c_ >= 1) else "no clear early-warning advantage at yearly resolution")
        EXTRA_LINES.append(f"Early-warning test ({N} drought site-years): shoreline vegetation water content (NDWI) was stressed in {a}, greenness (NDVI) in {b}, and NDWI flagged stress while NDVI was still normal in {c_}: {verdict_ew}. Yearly composites cannot show timing within a season, so treat this as supporting evidence, not proof.")
    else:
        EXTRA_LINES.append(f"Early-warning test: only {N} drought site-years in the record, too few to judge whether NDWI warns before NDVI.")
    n_ = len(EW); fig = plt.figure(figsize=(17, 1.2 + 1.9 * n_)); gs = fig.add_gridspec(n_, 2, width_ratios=[1.7, 1], wspace=0.18, hspace=0.6)
    axs = []
    for i, (sid, e) in enumerate(EW.items()):
        ax = fig.add_subplot(gs[i, 0], sharex=axs[0] if axs else None); axs.append(ax); t = e["t"]
        for y in t.year[t.drought]: ax.axvspan(y - 0.5, y + 0.5, color=C_DRY_L, alpha=0.45, lw=0)
        ax.axhline(0, c="k", lw=0.7); ax.axhline(STRESS_Z, color=C_DRY, ls=":", lw=1.1)
        gap_line(ax, t.year, t.ring_moist_mean_z, C_MOIST, "o", ms=4, lw=1.6)
        gap_line(ax, t.year, t.ring_ndvi_mean_z, C_VEG, "s", ms=4, lw=1.3)
        ax.set_ylim(-3, 3); ax.set_ylabel("Anomaly (SD)"); ax.set_title(f"{SITES[sid]['name']}  (rain vs NDWI rho = {e['rho_m']:+.2f}, rain vs NDVI rho = {e['rho_n']:+.2f})", loc="left", fontsize=9.5, pad=3)
        if i < n_ - 1: plt.setp(ax.get_xticklabels(), visible=False)
    axs[-1].set_xlabel("Year")
    axb = fig.add_subplot(gs[:, 1]); names = [SITES[s]["name"] for s in EW]; yy = np.arange(len(names))[::-1]; left = np.zeros(len(names))
    for key, lab_, col in [("early", "NDWI stressed, NDVI still normal (early warning)", C_MOIST), ("both", "Both stressed", C_DRY), ("ndvi_only", "Only NDVI stressed", C_VEG), ("neither", "Neither stressed", "#cccccc")]:
        v = np.array([EW[s][key] for s in EW], float); axb.barh(yy, v, left=left, color=col, edgecolor="white", label=lab_, zorder=3, height=0.6)
        for yi, l0, vi in zip(yy, left, v):
            if vi > 0: axb.text(l0 + vi / 2, yi, f"{int(vi)}", ha="center", va="center", color="white" if col != "#cccccc" else "#333333", fontsize=9.5, fontweight="bold")
        left += v
    axb.set_yticks(yy); axb.set_yticklabels(names); axb.set_xlabel("Number of drought years"); axb.grid(True, axis="x"); axb.grid(False, axis="y")
    axb.xaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True))
    axb.set_title("What did the shoreline vegetation do in each drought year?", loc="left", fontsize=10.5)
    hs = [Line2D([0], [0], color=C_MOIST, marker="o", mec="white", label="Canopy water content (NDWI)"), Line2D([0], [0], color=C_VEG, marker="s", mec="white", label="Greenness (NDVI)"),
          Line2D([0], [0], color=C_DRY, ls=":", label=f"stress threshold ({STRESS_Z:+.0f} SD)"), Patch(color=C_DRY_L, alpha=0.6, label="drought year (SPI-6 Dec-May < -1)")] + \
         [Patch(color=c_, label=l_) for l_, c_ in [("NDWI stressed, NDVI normal = early warning", C_MOIST), ("both stressed", C_DRY), ("only NDVI stressed", C_VEG), ("neither", "#cccccc")]]
    finish(fig, "08_early_warning_yearly", "Does canopy water content (NDWI) flag drought before greenness (NDVI) does?",
           "Shoreline vegetation of each dam at the end of the rains, as anomalies from its own (sensor-harmonised) record. An early warning is a drought year in which NDWI drops below the dotted line while NDVI is still normal. "
           "These are yearly composites, so this tests sensitivity, not timing within a season; the month-by-month test is figure 09.",
           source="Landsat 5/7/8/9 C2-L2 (30 m); " + str(next(iter(RES.values())).get("climate_src")), handles=hs, legend_cols=4, bot_extra=0.45)

# ---------- 2. DROUGHT-YEAR PANEL: what did each dam do in each REGIONAL drought year? ----------
_spis = pd.concat([r["drivers"][DROUGHT_SPI].rename(s) for s, r in RES.items() if r.get("drivers") is not None], axis=1)
REG_SPI = _spis.mean(axis=1) if len(_spis.columns) else pd.Series(dtype=float)          # regional mean of the per-site index (same product at every site)
dyears = [int(y) for y in REG_SPI.index[REG_SPI < STRESS_Z] if LANDSAT_YEARS[0] <= y <= LANDSAT_YEARS[-1]]
if dyears:
    ok_sites = [s for s, r in RES.items() if not r.get("error") and r.get("prim") == "landsat"]
    def zmat(metric, sk):
        M = np.full((len(ok_sites), len(dyears)), np.nan)
        for i, s in enumerate(ok_sites):
            st = RES[s]["stats"].get(("landsat", sk))
            if st is None: continue
            z = st[st.ok_site].set_index("year")[metric + "_z"]
            for j, y in enumerate(dyears):
                if y in z.index: M[i, j] = z[y]
        return M
    fig = plt.figure(figsize=(max(9, 1.15 * len(dyears) + 5), 1.6 + 1.0 * len(ok_sites) * 2)); gs2 = fig.add_gridspec(2, 2, width_ratios=[1, 0.03])
    axx = [fig.add_subplot(gs2[0, 0]), fig.add_subplot(gs2[1, 0])]; cax = fig.add_subplot(gs2[:, 1])
    norm = TwoSlopeNorm(vmin=-2.5, vcenter=0, vmax=2.5); cm = mpl.colormaps["BrBG"].copy(); cm.set_bad("#e6e6e6")
    for ax, (metric, sk, ttl) in zip(axx, [("water_pct_foot", WATER_SEASON, "A. Water cover at the END of the dry season that followed the failed rains"), ("ring_moist_mean", VEG_SEASON, "B. Canopy water content (NDWI) of shoreline vegetation at the end of the rains")]):
        M = zmat(metric, sk); im = ax.imshow(np.ma.masked_invalid(M), cmap=cm, norm=norm, aspect="auto")
        for i in range(M.shape[0]):
            for j in range(M.shape[1]):
                ax.text(j, i, f"{M[i, j]:+.1f}" if np.isfinite(M[i, j]) else "no data", ha="center", va="center", fontsize=9 if np.isfinite(M[i, j]) else 7.5, color=("white" if abs(M[i, j]) > 1.4 else "black") if np.isfinite(M[i, j]) else "#888888")
        ax.set_yticks(range(len(ok_sites))); ax.set_yticklabels([SITES[s]["name"] for s in ok_sites]); ax.set_xticks(range(len(dyears)))
        ax.set_xticklabels([f"{y}\nSPI-6 {REG_SPI.get(y, np.nan):+.1f}" for y in dyears], fontsize=8.5); ax.grid(False); ax.set_title(ttl, loc="left", fontsize=10.5)
        for sp in ax.spines.values(): sp.set_visible(False)
    cb = fig.colorbar(im, cax=cax); cb.set_label("Anomaly vs the site's own average (SD)\nbrown = less water / drier than normal, blue-green = more")
    finish(fig, "S7_drought_years", "How did each dam respond in the regional drought years?",
           f"Columns are years when the regional rainy-season drought index (SPI-6, Dec-May, mean of the {len(_spis.columns)} sites, one climate product) fell below {STRESS_Z:+.0f}. Brown = that dam had less water or drier shoreline vegetation than its own normal.",
           source="Landsat 5/7/8/9 C2-L2 (30 m); rainfall: " + str(next(iter(RES.values())).get("climate_src")), adjust=dict(left=0.17, right=0.88, wspace=0.05, hspace=0.55))
else:
    print("No regional drought years (SPI-6 below -1) in the record; drought-year panel skipped.")

# ---------- 3. THRESHOLD SENSITIVITY: does the 'what counts as water' threshold change the area? ----------
THR = np.round(np.arange(-0.2, 0.301, 0.025), 3); sens_rows = []
def _near_area(wat, res, A, lon, lat):
    lab, n = ndimage.label(wat, structure=np.ones((3, 3)))
    if n == 0: return 0.0
    px, py = _tr.transform(lon, lat); ix = int(np.abs(A.x.values - px).argmin()); iy = int(np.abs(A.y.values - py).argmin())
    d, (jy, jx) = ndimage.distance_transform_edt(lab == 0, return_indices=True); c = lab[jy[iy, ix], jx[iy, ix]]
    return float((lab == c).sum() * res * res / 1e4)
for sid, r in RES.items():
    if r.get("error") or r.get("prim") != "landsat": continue
    D = SITE_DATA[sid]["D"]; s = SITES[sid]; yrs = {}
    for tag, key in [("Landsat", ("landsat", WATER_SEASON)), ("Sentinel-2 (30 m)", ("s2_30", WATER_SEASON))]:
        if key in D: yrs[tag] = set(D[key]["qc"].loc[D[key]["qc"].ok, "year"].astype(int))
    common = set.intersection(*yrs.values()) if len(yrs) == 2 else set()
    for tag, key in [("Landsat", ("landsat", WATER_SEASON)), ("Sentinel-2 (30 m)", ("s2_30", WATER_SEASON))]:
        if tag not in yrs or not yrs[tag]: continue
        yr = max(common) if common else max(yrs[tag]); a = D[key]["annual"].sel(year=yr); obs = (a.n_obs > 0)
        for t in THR:
            sens_rows.append({"site": sid, "sensor": tag, "year": yr, "threshold": t,
                              "ndwi_ha": _near_area(((a.ndwi_water > t) & obs).values, 30, a, s["lon"], s["lat"]), "mndwi_ha": _near_area(((a.mndwi > t) & obs).values, 30, a, s["lon"], s["lat"])})
sens = pd.DataFrame(sens_rows)
if len(sens):
    EXTRA_SHEETS["threshold_sensitivity"] = sens; sites_s = list(dict.fromkeys(sens.site)); ncol = 3; nrow = int(np.ceil(len(sites_s) / ncol))
    fig, axs_ = plt.subplots(nrow, ncol, figsize=(5.4 * ncol, 4.1 * nrow), squeeze=False); [a_.axis("off") for a_ in axs_.ravel()]; summ = {}
    for ax, sid in zip(axs_.ravel(), sites_s):
        ax.axis("on"); d_all = sens[sens.site == sid]; txt = []
        for tag, col in [("Landsat", C_WATER), ("Sentinel-2 (30 m)", SENSOR_COL["Sentinel-2"])]:
            d_ = d_all[d_all.sensor == tag]
            if not len(d_): continue
            ref = d_.loc[np.isclose(d_.threshold, WATER_THR), "ndwi_ha"].iloc[0]
            if ref <= 0: continue
            ax.plot(d_.threshold, d_.ndwi_ha / ref, color=col, lw=2.3, label=f"{tag}: NDWI"); ax.plot(d_.threshold, d_.mndwi_ha / ref, color=col, ls="--", lw=1.4, label=f"{tag}: MNDWI (cross-check)")
            pl = d_[(d_.threshold >= -0.1) & (d_.threshold <= 0.1)].ndwi_ha; v = 100 * (pl.max() - pl.min()) / ref; summ[(sid, tag)] = v; txt.append(f"{tag}: {v:.0f}% spread")
        ax.axvspan(-0.1, 0.1, color=C_WATER_L, alpha=0.25, lw=0); ax.axvline(WATER_THR, c="k", lw=1.1); ax.axhspan(0.95, 1.05, color="grey", alpha=0.2, lw=0)
        ax.set_ylim(0, 3.5); ax.set_xlabel("NDWI threshold"); ax.set_ylabel("Area / area at chosen threshold"); ax.set_title(SITES[sid]["name"], loc="left", fontsize=10.5)
        ax.text(0.03, 0.04, "Spread across blue band:\n" + "\n".join(txt), transform=ax.transAxes, fontsize=8, va="bottom", bbox=dict(fc="white", ec="#cccccc", alpha=0.93, boxstyle="round,pad=0.3"))
    h_, l_ = axs_.ravel()[0].get_legend_handles_labels()
    finish(fig, "S8_threshold_sensitivity", "How much does the 'what counts as water' threshold change the area?",
           "A flat line at 1.0 inside the blue band means the area does not depend on the threshold. Lines that move mean small threshold changes add or remove many hectares, so treat hectares as approximate and trust the direction of change.",
           source="Landsat 5/7/8/9 C2-L2, Sentinel-2 L2A (30 m), latest usable dry season", handles=h_, legend_cols=4)
    ls_ = [v for (s_, t_), v in summ.items() if t_ == "Landsat"]
    if ls_: EXTRA_LINES.append(f"Threshold check: the Landsat water area of a dam changes by {min(ls_):.0f}% to {max(ls_):.0f}% when the NDWI threshold moves between -0.1 and +0.1, so quote hectares as approximate and rely on the direction of change.")

# ---------- 4. GeoTIFFs for GIS (water occurrence, usual footprint, change) ----------
n_tif = 0
for sid, r in RES.items():
    if r.get("error"): continue
    try:
        for name, da in [("water_occurrence", r.get("occ_da")), ("usual_footprint", r.get("foot_da")), ("water_change", r.get("change"))]:
            if da is None: continue
            da.astype("float32").rio.write_crs(CRS).rio.to_raster(DIRS["gis"] / f"{sid}_{name}.tif"); n_tif += 1
    except Exception as e: print(f"GeoTIFF export failed for {sid}: {type(e).__name__}: {e}")
open(DIRS["gis"] / "water_change_legend.txt", "w").write("water_change.tif values (usual end-of-dry-season water per decade " + ", ".join(DECADE_LABEL) + "):\n"
    "0 never water | 1 water in all 3 decades | 2 lost by " + DECADE_LABEL[1] + " | 3 lost by " + DECADE_LABEL[2] + " | 4 new by " + DECADE_LABEL[1] + " | 5 new by " + DECADE_LABEL[2] + " | 6 intermittent | NaN no data\n")
print(f"{n_tif} GeoTIFFs written to", DIRS["gis"].resolve())

# ---------- 5. FULL CLIMATE-DRIVER CORRELATIONS (as in v2 Cell 15): which climate variable does each dam respond to? ----------
resp = {("dry", "water_pct_foot"): "Water cover, end of dry season", ("mam", "water_pct_foot"): "Water cover, end of rains", ("mam", "ring_moist_mean"): "Shoreline vegetation water (NDWI)",
        ("mam", "ring_ndvi_mean"): "Shoreline greenness (NDVI)", ("mam", "stress_pct_z"): "Share of shoreline vegetation stressed"}
DRV_NAMES = {"wet_season_ppt_mm": "Wet-season rain (Nov-Apr)", "hydro_year_ppt_mm": "Rain, Nov-Oct", "water_balance_mm": "Rain minus evaporation demand", "spi3_dry": "Rain index 3-month (Oct)", "spi6_dry": "Rain index 6-month (Oct)",
             "spi12_dry": "Rain index 12-month (Oct)", "spi3_mam": "Rain index 3-month (May)", "spi6_mam": "Rain index 6-month (May)", "pdsi_dry": "Drought severity index (dry)", "pdsi_mam": "Drought severity index (rains)",
             "soil_mm_dry": "Soil moisture (dry)", "soil_mm_mam": "Soil moisture (rains)", "tmax_dry": "Max temperature (dry)", "tmax_mam": "Max temperature (rains)", "pet_dry_mm": "Evaporation demand (dry)",
             "spi12_dry_lag1": "Rain index 12-month, previous year", "wet_season_ppt_lag1": "Wet-season rain, previous year"}
crow = []
for sid, r in RES.items():
    if r.get("error") or r.get("drivers") is None: continue
    drv = r["drivers"].reset_index()
    for (sk, m), lab in resp.items():
        st = r["stats"].get((r["prim"], sk))
        if st is None or m not in st: continue
        mg = st[st.ok_site][["year", m]].merge(drv, on="year", how="inner")
        for dcol in DRV_NAMES:
            if dcol not in mg: continue
            dd_ = mg[[m, dcol]].dropna()
            if len(dd_) >= 8 and dd_[m].std() > 0 and dd_[dcol].std() > 0:
                rho, p = stats.spearmanr(dd_[m], dd_[dcol]); crow.append({"site": sid, "season": sk, "response": m, "response_label": lab, "driver": dcol, "driver_label": DRV_NAMES[dcol], "rho": rho, "p": p, "n": len(dd_)})
corr_all = pd.DataFrame(crow)
if len(corr_all): corr_all["q"] = corr_all.groupby("site").p.transform(lambda p_: pd.Series(bh_fdr(p_.values), index=p_.index))   # FDR within each dam's grid
if len(corr_all):
    EXTRA_SHEETS["climate_correlations"] = corr_all; sites_c = list(dict.fromkeys(corr_all.site)); drv_order = [d for d in DRV_NAMES if d in set(corr_all.driver)]; resp_labels = list(resp.values())
    fig = plt.figure(figsize=(3.9 * len(sites_c) + 2.5, 0.42 * len(drv_order) + 3.8)); gsc = fig.add_gridspec(1, len(sites_c) + 1, width_ratios=[1] * len(sites_c) + [0.07], wspace=0.08)
    axs_c = [[fig.add_subplot(gsc[0, i]) for i in range(len(sites_c))]]; cax_c = fig.add_subplot(gsc[0, -1])
    for ax, sid in zip(axs_c[0], sites_c):
        c = corr_all[corr_all.site == sid]; piv = c.pivot_table(index="driver", columns="response_label", values="rho").reindex(index=drv_order, columns=resp_labels)
        pp = c.pivot_table(index="driver", columns="response_label", values="p").reindex(index=drv_order, columns=resp_labels)
        qq = c.pivot_table(index="driver", columns="response_label", values="q").reindex(index=drv_order, columns=resp_labels)
        im = ax.imshow(np.ma.masked_invalid(piv.values), cmap="BrBG", vmin=-1, vmax=1, aspect="auto")
        for i in range(piv.shape[0]):
            for j in range(piv.shape[1]):
                if np.isfinite(piv.values[i, j]): ax.text(j, i, f"{piv.values[i, j]:.2f}" + ("*" if qq.values[i, j] < FDR_Q else ("°" if pp.values[i, j] < 0.05 else "")), ha="center", va="center", fontsize=7.5, color="white" if abs(piv.values[i, j]) > 0.6 else "black")
        ax.set_xticks(range(len(resp_labels))); ax.set_xticklabels([l.replace("Share of shoreline vegetation stressed", "Vegetation stress share") for l in resp_labels], rotation=40, ha="right", fontsize=7.5)
        ax.set_yticks(range(len(drv_order))); ax.set_yticklabels([DRV_NAMES[d] for d in drv_order] if ax is axs_c[0][0] else [], fontsize=8); ax.grid(False); ax.set_title(SITES[sid]["name"], loc="left", fontsize=10)
    cb = fig.colorbar(im, cax=cax_c); cb.set_label("Spearman correlation\nblue-green = rises with the climate variable, brown = falls")
    finish(fig, "S9_climate_drivers", "Which climate variable does each dam respond to?",
           f"Each cell is the rank correlation between a yearly measurement of the dam (columns) and a climate variable (rows). * = survives a false-discovery-rate correction across the whole grid of that dam (q < {FDR_Q}); ° = p < 0.05 only, which with ~70 cells per dam is expected a few times by chance. Climate variables the product does not provide are left out.",
           source="Landsat 5/7/8/9 C2-L2 (30 m); " + str(next(iter(RES.values())).get("climate_src")), adjust=dict(left=0.14, right=0.95, wspace=0.08), bot_extra=1.1)


## Step 4c. NDWI seasonal trend and the month-by-month early warning
Core: `03_ndwi_seasonal_trend` (open-water and canopy NDWI for every dam, 1996-2025, blue = high water / canopy moisture, brown = low moisture / stress), `04_ndwi_monthly_heatmap` (the same month by month) and `09_early_warning_monthly`. Supplementary: `S10` rain timescales and `S11` shoreline retreat in metres.

In [ ]:
# Cell 11c: THE NDWI SEASONAL TREND VISUAL, A MONTH-BY-MONTH EARLY-WARNING TEST, SPEI / RAIN-TIMESCALE ANALYSIS, AND SHORELINE RETREAT IN METRES.
from scipy.stats import norm
from contourpy import contour_generator
MONTHS_ALL = [3, 4, 5, 8, 9, 10]; MN = {3: "Mar", 4: "Apr", 5: "May", 8: "Aug", 9: "Sep", 10: "Oct"}
SEAS_M = {"mam": [3, 4, 5], "dry": [8, 9, 10]}

# ---------- monthly series from the scene tables (one value per year and month, anomaly against that month's own normal) ----------
# v5: vegetation values are the HARMONISED scene values (Cell 9); Landsat 7 after its orbit drift is dropped; water fractions need >= 80% clear pixels
# (in a half-cloudy scene clouds sit mostly over land, so the 'water share of clear pixels' would be inflated).
MON = {}
for sid, r in RES.items():
    if r.get("error") or r.get("prim") != "landsat": continue
    parts = []
    for sk in SEASONS:
        sc = SITE_DATA[sid]["D"][("landsat", sk)]["scenes"]; sc = sc[sc.scene_ok & ~((sc.platform == "landsat-7") & (sc.year >= L7_DRIFT_YEAR))].copy()
        sc.loc[sc.valid_frac < 0.8, ["water_frac_valid", "water_ndwi_mean"]] = np.nan
        parts.append(sc.groupby(["year", "month"])[["moist_nat_mean", "ndvi_nat_mean", "water_frac_valid", "water_ndwi_mean"]].median().reset_index())
    m = pd.concat(parts, ignore_index=True)
    for c in ["moist_nat_mean", "ndvi_nat_mean", "water_frac_valid", "water_ndwi_mean"]:
        gb = m.groupby("month")[c]; sd = gb.transform("std"); n_ = gb.transform("count")
        m[c + "_z"] = ((m[c] - gb.transform("mean")) / sd.where(sd > 0)).where(n_ >= 8)        # a 'normal' needs at least 8 years of that month
    MON[sid] = m

# ---------- 1a. NDWI SEASONAL TREND ----------
# Row 1 = open-water NDWI (Green-NIR)/(Green+NIR) -> share of each dam's usual footprint under water at the END OF THE DRY SEASON (the annual low point = water stress).
# Row 2 = canopy NDWI (NIR-SWIR)/(NIR+SWIR) of natural vegetation within RING_M of the shore at the END OF THE RAINS (full canopy; in Aug-Oct the miombo
#          woodland is partly leafless, so a low dry-season value is phenology, not stress). Markers are coloured on ONE diverging scale:
#          blue = high water / canopy moisture, brown = low moisture / stress (standardised anomaly against that dam's own record).
CM_NDWI = mpl.colormaps["BrBG"].copy()
ROW_NORM = {"water_pct_foot": TwoSlopeNorm(vmin=-20, vcenter=0, vmax=20), "ring_moist_mean": TwoSlopeNorm(vmin=-2, vcenter=0, vmax=2)}
sites_l = [s for s, r in RES.items() if not r.get("error") and r.get("prim") == "landsat"]
if sites_l:
    n_ = len(sites_l); fig = plt.figure(figsize=(3.5 * n_ + 2.2, 9.2)); gsl = fig.add_gridspec(2, n_ + 1, width_ratios=[1] * n_ + [0.06], wspace=0.30, hspace=0.62)
    ROWS = [("water_pct_foot", WATER_SEASON, "Open water  NDWI = (Green - NIR) / (Green + NIR)", "% of usual footprint under water", "pts"),
            ("ring_moist_mean", VEG_SEASON, "Vegetation canopy water  NDWI = (NIR - SWIR) / (NIR + SWIR)", "canopy NDWI (harmonised)", "NDWI")]
    for row, (met, sk, ttl, ylab, unit) in enumerate(ROWS):
        for j, sid in enumerate(sites_l):
            ax = fig.add_subplot(gsl[row, j]); st = RES[sid]["stats"].get(("landsat", sk))
            if st is None: ax.axis("off"); continue
            ok = st[st.ok_site & st[met].notna()]; x, y = ok.year.values.astype(float), ok[met].values.astype(float)
            # colour = departure from this dam's own normal. Water: in % points of the footprint (a 3-point wobble at a full dam is NOT stress);
            # canopy NDWI: standardised anomaly (SD), because NDWI has no natural physical scale across sites.
            cval = (y - np.median(y)) if met == "water_pct_foot" else ((y - y.mean()) / y.std(ddof=1) if len(y) > 2 and y.std(ddof=1) > 0 else np.zeros_like(y))
            for yy_ in drought_years(RES[sid]): ax.axvspan(yy_ - 0.5, yy_ + 0.5, color="#ececec", lw=0, zorder=0)
            gx, gy = gap_xy(x, y); ax.plot(gx, gy, color="#9a9a9a", lw=0.9, zorder=2)
            ax.scatter(x, y, c=cval, cmap=CM_NDWI, norm=ROW_NORM[met], s=34, edgecolor="#555555", lw=0.4, zorder=4)
            q = quick_trend(x, y)
            if q is not None:
                pdec = 1000 * q["slope"] / abs(q["mean"]) if (met == "water_pct_foot" and q["mean"]) else np.nan
                if met == "water_pct_foot" and q["trend"] != "no trend" and abs(pdec) < MIN_EFFECT_PCT_DEC: q["trend"] = "small"     # same effect-size rule as the scoreboard
                sen_band(ax, x, y, "#222222", sig=q["trend"] in ("increasing", "decreasing"), band=True, lw=1.8)
                vt = {"increasing": "RISING", "decreasing": "DECLINING", "no trend": "no clear trend", "small": "detectable but < 5%/decade"}[q["trend"]]
                rc_ = record_cover(x); span_ = f"{int(x.min())}-{int(x.max())}, n={len(x)}" + ("" if rc_["full_record"] else " PARTIAL")
                ax.text(0.03, 0.04, f"{vt} ({fmt_p(q['p'])})\n{10 * q['slope']:+.2g} {unit}/decade" + (f" ({pdec:+.0f}%)" if np.isfinite(pdec) else "") + f"\n{span_}", transform=ax.transAxes, fontsize=7.8,
                        va="bottom", bbox=dict(fc="white", ec="#cccccc", alpha=0.9, boxstyle="round,pad=0.25"), zorder=7)
            if met == "water_pct_foot": ax.set_ylim(0, 105)
            else:
                lo_, hi_ = np.nanmin(y), np.nanmax(y); pad_ = 0.40 * (hi_ - lo_) + 1e-3; ax.set_ylim(lo_ - pad_, hi_ + 0.12 * (hi_ - lo_))
            ax.set_xlim(LANDSAT_YEARS[0] - 1, LANDSAT_YEARS[-1] + 1); ax.set_xticks([1990, 2000, 2010, 2020])
            ax.set_title(SITES[sid]["name"] + f"  [{RES[sid]['rel']}]", loc="left", fontsize=9.8)
            if j == 0: ax.set_ylabel(ylab)
            if j == 0: ax.text(0.0, 1.17, f"{'AB'[row]}. {ttl}   ({season_label(sk)})", transform=ax.transAxes, fontsize=10.5, fontweight="bold", color="#222222")
        cax = fig.add_subplot(gsl[row, -1]); cb = fig.colorbar(mpl.cm.ScalarMappable(norm=ROW_NORM[met], cmap=CM_NDWI), cax=cax)
        if met == "water_pct_foot": cb.set_ticks([-20, -10, 0, 10, 20]); cb.set_ticklabels(["-20 pts  Low water", "-10", "0  dam's median", "+10", "+20  High water"])
        else: cb.set_ticks([-2, -1, 0, 1, 2]); cb.set_ticklabels(["-2 SD  Low moisture / stress", "-1", "0  normal", "+1", "+2 SD  High canopy moisture"])
        cb.ax.tick_params(labelsize=8)
    hs = [Line2D([0], [0], color="#222222", lw=1.8, label="Sen trend, significant (band = 95% range)"), Line2D([0], [0], color="#222222", lw=1.3, ls="--", label="Sen trend, not significant"),
          Patch(color="#ececec", label="rainfall drought year (SPI-6 Dec-May < -1)"), Line2D([0], [0], color="#9a9a9a", lw=0.9, label="consecutive years (broken where a year is missing)")]
    finish(fig, "03_ndwi_seasonal_trend", "NDWI seasonal trend: is the open water, and the water in the plants around it, declining?",
           f"A: open-water NDWI at the end of the dry season, when the dams are lowest. B: canopy NDWI of natural vegetation within {RING_M} m of the shore at the end of the rains (full leaf). "
           "Colour = departure from that dam's own normal (blue = high water / canopy moisture, brown = low moisture / stress). A declining canopy NDWI is the early sign of plant water deficit. "
           "Canopy NDWI is harmonised across the Landsat 7 to 8 switch AND corrected for the date mix of each year's images (it dries ~0.08 per month from March to May). Trend significance allows for year-to-year memory (Hamed-Rao).",
           source="Landsat 5/7/8/9 C2-L2 (30 m)", handles=hs, legend_cols=4, adjust=dict(left=0.06, right=0.90, wspace=0.30, hspace=0.62), top_extra=0.35)

# ---------- 1b. The same NDWI, month by month (heat-map view) ----------
if MON:
    sites_m = list(MON); n_ = len(sites_m); fig = plt.figure(figsize=(3.3 * n_ + 2.4, 11.5)); gsm = fig.add_gridspec(2, n_ + 1, width_ratios=[1] * n_ + [0.07], wspace=0.12, hspace=0.32)
    cm_ = mpl.colormaps["BrBG"].copy(); cm_.set_bad("#e6e6e6"); yrs_all = list(LANDSAT_YEARS)
    for row, (zc, ttl) in enumerate([("moist_nat_mean_z", "Canopy water content of natural vegetation in the study box  (NIR - SWIR) / (NIR + SWIR)"), ("water_frac_valid_z", "Open water in the study box (all water bodies)  (Green - NIR) / (Green + NIR) > 0")]):
        for j, sid in enumerate(sites_m):
            ax = fig.add_subplot(gsm[row, j]); pv = MON[sid].pivot_table(index="year", columns="month", values=zc).reindex(index=yrs_all, columns=MONTHS_ALL)
            im = ax.imshow(np.ma.masked_invalid(pv.values), cmap=cm_, vmin=-2.5, vmax=2.5, aspect="auto", interpolation="nearest"); ax.axvline(2.5, color="white", lw=4)
            ax.set_xticks(range(6)); ax.set_xticklabels([MN[m_] for m_ in MONTHS_ALL], fontsize=8.5); ax.set_yticks(range(0, len(yrs_all), 5)); ax.set_yticklabels(yrs_all[::5], fontsize=8.5); ax.grid(False)
            ax.text(1, -0.9, "end of rains", ha="center", fontsize=7.5, color="#555555"); ax.text(4, -0.9, "end of dry season", ha="center", fontsize=7.5, color="#555555")
            if j == 0: ax.set_ylabel("Year")
            else: ax.set_yticklabels([])
            ax.set_title(SITES[sid]["name"] if row == 0 else "", loc="left", fontsize=10.5, pad=14)
            if j == 0: ax.text(0.0, 1.13 if row == 0 else 1.06, ttl, transform=ax.transAxes, fontsize=9.5, fontweight="bold", color="#333333")
        if row == 0:
            cax = fig.add_subplot(gsm[:, -1]); cb = fig.colorbar(im, cax=cax); cb.set_ticks([-2, -1, 0, 1, 2])
            cb.set_ticklabels(["-2 SD  Low moisture /\nstress", "-1", "0  normal", "+1", "+2 SD  High water /\ncanopy moisture"]); cb.ax.tick_params(labelsize=8.5)
    finish(fig, "04_ndwi_monthly_heatmap", "NDWI month by month: where in the record are the dry and the wet months?",
           "Each cell is one month of one year, compared with the same calendar month's own normal (so the normal seasonal cycle is removed). Blue = high water / canopy moisture, brown = low moisture / stress, grey = no clear image. "
           "Vegetation values are harmonised across the 2013 Landsat 7 to 8 switch; without that step the whole pre-2013 block turned brown. Months with fewer than 8 years of data have no normal and stay grey.",
           source="Landsat 5/7/8/9 C2-L2 (30 m), scene-level medians for the study box", bot_extra=0.2, top_extra=0.35)
    mt_rows = []
    for sid in sites_m:
        for var, c in [("canopy_water_NDWI", "moist_nat_mean_z"), ("open_water_extent", "water_frac_valid_z")]:
            for mo in MONTHS_ALL:
                d_ = MON[sid][MON[sid].month == mo][["year", c]].dropna(); t = trend_table(d_, {c: var}, min_n=10)
                if len(t): t = harden_trends(t).iloc[0]; mt_rows.append({"site": sid, "variable": var, "month": mo, "n_years": int(t.n_years), "trend": t.mk_trend, "p": t.mk_p, "sd_per_decade": 10 * t.sen_slope_per_year})
    mt = pd.DataFrame(mt_rows)
    if len(mt): mt["q"] = bh_fdr(mt.p.values)
    EXTRA_SHEETS["ndwi_monthly_trends"] = mt
    if len(mt):
        cw = mt[mt.variable == "canopy_water_NDWI"]
        EXTRA_LINES.append(f"Seasonal NDWI trend (harmonised): canopy water content is declining in {int((cw.trend == 'decreasing').sum())} and rising in {int((cw.trend == 'increasing').sum())} of {len(cw)} dam-month series "
                           f"(p<0.05 after allowing for year-to-year memory); {int(((cw.trend != 'no trend') & (cw.q < FDR_Q)).sum())} survive a false-discovery-rate correction across all {len(mt)} monthly tests.")

# ---------- 2. EARLY WARNING WITHIN THE SEASON: in drought years, does canopy NDWI drop in an earlier month than NDVI? ----------
EWM_ROWS, EWM_LEAD, EWM_FIG = [], [], {}
for sid, r in RES.items():
    if sid not in MON or r.get("drivers") is None: continue
    m = MON[sid]; drv = r["drivers"]
    for sk, col_drought in [("mam", DROUGHT_SPI), ("dry", DROUGHT_SPI)]:          # one drought definition: the failed rains precede both seasons
        dyears = set(drv.index[drv[col_drought] < STRESS_Z].astype(int)); mm = m[m.month.isin(SEAS_M[sk])]
        for mo in SEAS_M[sk]:
            x = mm[mm.month == mo]; dr = x[x.year.isin(dyears)]; nr = x[~x.year.isin(dyears)]
            if len(dr) >= 2: EWM_ROWS.append({"site": sid, "season": sk, "month": mo, "n_drought_years": len(dr), "NDWI_z_drought": dr.moist_nat_mean_z.mean(), "NDWI_se": dr.moist_nat_mean_z.sem(), "NDVI_z_drought": dr.ndvi_nat_mean_z.mean(), "NDVI_se": dr.ndvi_nat_mean_z.sem(),
                                              "NDWI_z_normal": nr.moist_nat_mean_z.mean(), "NDVI_z_normal": nr.ndvi_nat_mean_z.mean()})
        for y in sorted(dyears):
            yy = mm[mm.year == y].set_index("month").reindex(SEAS_M[sk])
            if yy.moist_nat_mean_z.notna().sum() < 2 or yy.ndvi_nat_mean_z.notna().sum() < 2: continue
            first = lambda s_: next((i for i, v in enumerate(s_.values) if np.isfinite(v) and v < -0.5), None); fw, fn = first(yy.moist_nat_mean_z), first(yy.ndvi_nat_mean_z)
            cat = "neither stressed" if (fw is None and fn is None) else ("NDWI first" if (fn is None or (fw is not None and fw < fn)) else ("same month" if fw == fn else "NDVI first"))
            EWM_LEAD.append({"site": sid, "season": sk, "year": y, "first_NDWI_month": None if fw is None else SEAS_M[sk][fw], "first_NDVI_month": None if fn is None else SEAS_M[sk][fn], "result": cat})
ewm = pd.DataFrame(EWM_ROWS); lead = pd.DataFrame(EWM_LEAD)
# lead-lag correlation of consecutive months: does this month's NDWI anomaly predict NEXT month's NDVI better than NDVI predicts next NDWI?
LL_ROWS = []
for sid, m in MON.items():
    a, b = [], []
    for sk in SEAS_M:
        for m0, m1 in zip(SEAS_M[sk][:-1], SEAS_M[sk][1:]):
            x0, x1 = m[m.month == m0].set_index("year"), m[m.month == m1].set_index("year"); ix = x0.index.intersection(x1.index)
            a.append(pd.DataFrame({"w0": x0.loc[ix, "moist_nat_mean_z"], "v0": x0.loc[ix, "ndvi_nat_mean_z"], "w1": x1.loc[ix, "moist_nat_mean_z"], "v1": x1.loc[ix, "ndvi_nat_mean_z"]}))
    if a:
        p_ = pd.concat(a).dropna()
        if len(p_) >= 20: LL_ROWS.append({"site": sid, "n_month_pairs": len(p_), "NDWI_now_vs_NDVI_next": stats.spearmanr(p_.w0, p_.v1)[0], "NDVI_now_vs_NDWI_next": stats.spearmanr(p_.v0, p_.w1)[0]})
ll = pd.DataFrame(LL_ROWS)
if len(ewm) or len(lead) or len(ll):
    EXTRA_SHEETS["early_warning_monthly"] = ewm; EXTRA_SHEETS["early_warning_first_month"] = lead; EXTRA_SHEETS["early_warning_lead_lag"] = ll
    if len(lead):
        vc = lead.result.value_counts(); EXTRA_LINES.append("Month-by-month early-warning test (" + str(len(lead)) + " dam-seasons in drought): " + "; ".join(f"{k} in {v}" for k, v in vc.items()) + ". NDWI warning first means canopy water content fell below -0.5 SD in an earlier month than greenness.")
    if len(ll): EXTRA_LINES.append(f"Lead-lag check: this month's canopy-water anomaly correlates {ll.NDWI_now_vs_NDVI_next.median():+.2f} with next month's greenness anomaly, against {ll.NDVI_now_vs_NDWI_next.median():+.2f} the other way round (median over dams); a larger first number supports NDWI leading NDVI.")
    sites_e = [s for s in MON if len(ewm) and s in set(ewm.site)]
    if sites_e:
        fig, axs = plt.subplots(len(sites_e), 2, figsize=(13, 2.7 * len(sites_e) + 1.2), squeeze=False, sharey=True)
        for i, sid in enumerate(sites_e):
            for j, sk in enumerate(["mam", "dry"]):
                ax = axs[i][j]; e = ewm[(ewm.site == sid) & (ewm.season == sk)].sort_values("month"); ax.axhline(0, c="k", lw=0.8); ax.axhline(-0.5, color=C_DRY, ls=":", lw=1.1)
                if len(e):
                    xx = np.array([SEAS_M[sk].index(m_) for m_ in e.month]); big = e.n_drought_years.values >= 3     # a standard error from 1-2 years is meaningless: no bar drawn
                    for col_, c_se, mk_, off_, lab_ in [("NDWI_z_drought", "NDWI_se", "o", -0.06, "Canopy water content (NDWI)"), ("NDVI_z_drought", "NDVI_se", "s", 0.06, "Greenness (NDVI)")]:
                        cc = C_MOIST if mk_ == "o" else C_VEG
                        ax.plot(xx + off_, e[col_], "-", color=cc, lw=1.6); ax.plot(xx + off_, e[col_], mk_, color=cc, ms=6, mec="white", label=lab_, ls="none")
                        ax.errorbar((xx + off_)[big], e[col_].values[big], yerr=1.96 * e[c_se].values[big], fmt="none", ecolor=cc, capsize=3, lw=1.2)
                    for xi, nn in zip(xx, e.n_drought_years): ax.text(xi, 2.15, f"n={int(nn)}", ha="center", fontsize=7.5, color="#555555")
                    ax.set_xticks(range(3)); ax.set_xticklabels([MN[m_] for m_ in SEAS_M[sk]]); ax.set_xlim(-0.4, 2.4); ax.set_ylim(-2.6, 2.5)
                    if len(lead): lc_ = lead[(lead.site == sid) & (lead.season == sk)].result.value_counts(); ax.text(0.02, 0.05, "In drought years: " + (", ".join(f"{k} {v}" for k, v in lc_.items()) or "no complete years"), transform=ax.transAxes, fontsize=8, bbox=dict(fc="white", ec="#cccccc", alpha=0.93, boxstyle="round,pad=0.25"))
                if j == 0: ax.set_ylabel(SITES[sid]["name"] + "\nanomaly (SD)", fontsize=9)
                if i == 0: ax.set_title("End of the rains" if sk == "mam" else "End of the dry season that followed", loc="left", fontsize=10)
        hs = [Line2D([0], [0], color=C_MOIST, marker="o", mec="white", label="Canopy water content (NDWI)"), Line2D([0], [0], color=C_VEG, marker="s", mec="white", label="Greenness (NDVI)"), Line2D([0], [0], color=C_DRY, ls=":", label="-0.5 SD")]
        finish(fig, "09_early_warning_monthly", "Month by month in drought years: does canopy water content drop before greenness?",
               "Average anomaly of each month in the drought years (SPI-6 Dec-May < -1). Error bars = 95% confidence interval, drawn only where at least 3 drought years were observed in that month (n above each month). "
               "If the blue line sits below the green one early in the season, canopy water content is warning before greenness changes. The box counts, per drought year, which index first fell below -0.5 SD.",
               source="Landsat 5/7/8/9 C2-L2 (30 m), harmonised scene medians; rainfall: " + str(next(iter(RES.values())).get("climate_src")), handles=hs, legend_cols=3)

# ---------- 3. SPEI AND RAIN TIMESCALES: over how many months of rainfall does each dam respond, and does evaporation demand add information? ----------
def spei_np(wb, k):
    """Standardised (rainfall - evaporation demand) over k months. Non-parametric (rank-based normal scores per calendar month), so no distribution has to be assumed."""
    s = wb.rolling(k).sum(); out = pd.Series(np.nan, index=s.index)
    for mo in range(1, 13):
        sm = s[s.index.month == mo].dropna()
        if len(sm) >= 15: out[sm.index] = np.asarray(norm.ppf((sm.rank() - 0.44) / (len(sm) + 0.12)))
    return out
TS_ROWS = []; TSC = {}; K = list(range(1, 25))
for sid, r in RES.items():
    tc = r.get("tc")
    if r.get("error") or tc is None or "pet" not in tc or tc["pet"].isna().all(): continue
    ppt = tc["ppt"]; wb = tc["ppt"] - tc["pet"]; spi_k = {k: spi(ppt, k) for k in K}; spe_k = {k: spei_np(wb, k) for k in K}
    for (sk, metric, endm), lab in {("dry", "water_pct_foot", 10): "Water cover, end of dry season", ("mam", "ring_moist_mean", 5): "Shoreline vegetation water (NDWI)", ("mam", "ring_ndvi_mean", 5): "Shoreline greenness (NDVI)", ("mam", "stress_pct_z", 5): "Vegetation stress share"}.items():
        st = r["stats"].get(("landsat", sk))
        if st is None: continue
        y_ = st[st.ok_site].set_index("year")[metric].dropna()
        for nm, dct in [("SPI", spi_k), ("SPEI", spe_k)]:
            rho = []
            for k in K:
                s_ = dct[k]; x = pd.Series({int(y): float(s_[(s_.index.year == y) & (s_.index.month == endm)].iloc[0]) if ((s_.index.year == y) & (s_.index.month == endm)).any() else np.nan for y in y_.index})
                dd_ = pd.concat([y_, x], axis=1).dropna()
                rho.append(stats.spearmanr(dd_.iloc[:, 0], dd_.iloc[:, 1])[0] if len(dd_) >= 8 and dd_.iloc[:, 0].std() > 0 and dd_.iloc[:, 1].std() > 0 else np.nan)
            rho = np.array(rho); TSC[(sid, lab, nm)] = rho
            if np.isfinite(rho).any(): kb = int(np.nanargmax(np.abs(rho))); TS_ROWS.append({"site": sid, "response": lab, "index": nm, "best_timescale_months": K[kb], "rho_at_best": float(rho[kb])})
ts_tab = pd.DataFrame(TS_ROWS)
if len(ts_tab):
    EXTRA_SHEETS["drought_timescales"] = ts_tab; labs = list(dict.fromkeys(ts_tab.response)); sites_t = list(dict.fromkeys(ts_tab.site)); pal = {s_: SITE_COL.get(s_, "#555555") for s_ in sites_t}
    fig, axs = plt.subplots(2, 2, figsize=(13, 9.5), squeeze=False, sharey=True); fig.subplots_adjust(hspace=0.35)
    for ax, lab in zip(axs.ravel(), labs):
        for sid in sites_t:
            for nm, ls_ in [("SPI", "-"), ("SPEI", "--")]:
                if (sid, lab, nm) in TSC: ax.plot(K, TSC[(sid, lab, nm)], ls_, color=pal[sid], lw=1.6, label=f"{SITES[sid]['name'].replace(' Dam', '')} {nm}" if lab == labs[0] else None)
        n_typ = int(np.median([len(RES[s_]["stats"][("landsat", WATER_SEASON)].query("ok_site")) for s_ in sites_t]))
        rc = np.tanh(1.96 / np.sqrt(max(n_typ - 3, 1)))                       # |rho| a random series reaches at p = 0.05 with this many years
        ax.axhspan(-rc, rc, color="#eeeeee", lw=0, zorder=0); ax.text(23.8, rc + 0.02, f"noise level (p=0.05, n~{n_typ})", ha="right", fontsize=7.5, color="#777777")
        ax.axhline(0, c="k", lw=0.8); ax.set_xlabel("Months of rainfall accumulated"); ax.set_xticks([1, 3, 6, 9, 12, 18, 24]); ax.set_ylim(-1, 1); ax.set_title(lab, loc="left", fontsize=10)
    [a_.set_ylabel("Correlation (Spearman)") for a_ in axs[:, 0]]; axs[0][0].legend(fontsize=8, ncol=2, loc="upper left")
    finish(fig, "S10_rain_timescales", "How long a rain record does each dam respond to?",
           "Correlation between each yearly measurement and the rain index accumulated over 1 to 24 months before it. A peak at 3 months means the dam reacts to the last season's rain; a peak at 12 to 24 months means it integrates rain over years. Solid = rain only (SPI); dashed = rain minus evaporation demand (SPEI). Curves inside the grey band are indistinguishable from chance, and picking the best of 48 curves per panel overstates the peak, so read the shape, not the single best value.",
           source="Landsat 5/7/8/9 C2-L2 (30 m); " + str(next(iter(RES.values())).get("climate_src")), bot_extra=0.1)
    wat = ts_tab[(ts_tab.response == "Water cover, end of dry season")]; sp_b = wat[wat["index"] == "SPI"].set_index("site"); se_b = wat[wat["index"] == "SPEI"].set_index("site")
    if len(sp_b): EXTRA_LINES.append(f"Rain timescale: water cover at the end of the dry season responds most to rain accumulated over {int(sp_b.best_timescale_months.min())} to {int(sp_b.best_timescale_months.max())} months (median {sp_b.best_timescale_months.median():.0f}); the strongest correlation reached is {sp_b.rho_at_best.abs().max():.2f}" + (f" with rain only and {se_b.rho_at_best.abs().max():.2f} when evaporation demand is included (SPEI)." if len(se_b) else "."))
AR_ROWS = []
for sid, r in RES.items():
    tc = r.get("tc")
    if r.get("error") or tc is None or tc["pet"].isna().all(): continue
    hy = pd.DataFrame({"y": np.where(tc.index.month >= 11, tc.index.year + 1, tc.index.year), "ppt": tc.ppt.values, "pet": tc.pet.values}).groupby("y").sum(); hy = hy[(hy.index >= LANDSAT_YEARS[0]) & (hy.index <= LANDSAT_YEARS[-1])]
    hy["aridity_index"] = hy.ppt / hy.pet; t = trend_table(hy.reset_index().rename(columns={"y": "year"}), {"aridity_index": "Aridity index (rain / evaporation demand)"}, min_n=10)
    if len(t): t = harden_trends(t).iloc[0]; AR_ROWS.append({"site": sid, "mean_aridity_index": float(hy.aridity_index.mean()), "trend": t.mk_trend, "p": t.mk_p, "change_per_decade": 10 * t.sen_slope_per_year})
if AR_ROWS:
    ar = pd.DataFrame(AR_ROWS); EXTRA_SHEETS["aridity_trend"] = ar
    EXTRA_LINES.append(f"Aridity: the rain-to-evaporation ratio averages {ar.mean_aridity_index.mean():.2f} across the dams (below 1 means evaporation demand exceeds rain) and shows " + ("no significant trend." if (ar.trend == "no trend").all() else f"a significant trend at {int((ar.trend != 'no trend').sum())} of {len(ar)} sites."))

# ---------- 4. SHORELINE RETREAT IN METRES ----------
def outline_length_m(mask, res=30):
    cg = contour_generator(z=mask.astype(float)); tot = 0.0
    for ln in cg.lines(0.5):
        if len(ln) > 1: tot += float(np.hypot(np.diff(ln[:, 0]), np.diff(ln[:, 1])).sum())
    return tot * res
SH = {}; SH_ROWS = []; EDGE_ERR_M = 15.0          # one estimate of the mean band width is uncertain by about half a 30 m pixel at the water edge
DEC_COL = ["#c6dbef", "#6baed6", "#08519c"]       # decade 1 -> 3: light to dark (time order)
def mean_ci(v, conf=0.95):
    v = np.asarray(v, float); v = v[np.isfinite(v)]
    if len(v) < 2: return (float(v.mean()) if len(v) else np.nan), np.nan
    return float(v.mean()), float(stats.t.ppf(0.5 + conf / 2, len(v) - 1) * v.std(ddof=1) / np.sqrt(len(v)))
for sid, r in RES.items():
    if r.get("error") or r.get("prim") != "landsat": continue
    st = r["stats"].get(("landsat", WATER_SEASON))
    if st is None: continue
    L_ = outline_length_m(r["fp"]["foot"]); ok = st[st.ok_site & st.water_ha.notna()].copy()
    if L_ <= 0 or len(ok) < 8: continue
    ok["exposed_m"] = (r["fp"]["area_ha"] - ok.water_ha) * 1e4 / L_; SH[sid] = ok[["year", "exposed_m"]]
    t = trend_table(ok[["year", "exposed_m"]], {"exposed_m": "Exposed shoreline band (m)"}, min_n=8)
    row = {"site": sid, "shoreline_length_km": L_ / 1000}
    vals = [ok[(ok.year >= w0) & (ok.year <= w1)].exposed_m.values for w0, w1 in DECADES]
    for lab, v in zip(DECADE_LABEL, vals):
        m_, h_ = mean_ci(v); row.update({f"mean_{lab}": m_, f"ci95_{lab}": h_, f"n_{lab}": len(v)})
    for (l0, v0), (l1, v1) in zip(list(zip(DECADE_LABEL, vals))[:-1], list(zip(DECADE_LABEL, vals))[1:]):
        dlt, dlo, dhi = ci_mean_diff(v0, v1); row.update({f"change_{l0}_to_{l1}": dlt, f"change_{l0}_to_{l1}_lo": dlo, f"change_{l0}_to_{l1}_hi": dhi})
    if len(t): t = harden_trends(t).iloc[0]; row.update({"trend": t.mk_trend, "p": t.mk_p, "retreat_m_per_decade": 10 * t.sen_slope_per_year})
    SH_ROWS.append(row)
if SH_ROWS:
    sh = pd.DataFrame(SH_ROWS); EXTRA_SHEETS["shoreline_by_decade"] = sh; display(sh.round(1))
    sites_s = list(SH); ncol = 3; nrow = int(np.ceil((len(sites_s) + 1) / ncol)); ymax = max(float(SH[s_].exposed_m.max()) for s_ in sites_s) * 1.15 + 5
    fig, axs = plt.subplots(nrow, ncol, figsize=(5.5 * ncol, 3.7 * nrow), squeeze=False)
    for ax, sid in zip(axs.ravel(), sites_s):
        d_ = SH[sid]; col = SITE_COL.get(sid, "#555555")
        ax.axhspan(0, EDGE_ERR_M, color="#e5e5e5", lw=0, zorder=0)
        for k, ((w0, w1), lab) in enumerate(zip(DECADES, DECADE_LABEL)):
            ax.axvspan(w0 - 0.5, w1 + 0.5, ymin=0.965, ymax=1.0, color=DEC_COL[k], lw=0)
            v = d_[(d_.year >= w0) & (d_.year <= w1)].exposed_m
            if len(v) >= 3: ax.plot([w0 - 0.4, w1 + 0.4], [v.mean(), v.mean()], color=DEC_COL[k], lw=3.2, zorder=4, solid_capstyle="butt"); ax.text((w0 + w1) / 2, v.mean() + 2, f"{v.mean():.0f} m", ha="center", va="bottom", fontsize=7.8, color="#08306b")
        for w0, _ in DECADES[1:]: ax.axvline(w0 - 0.5, color="#bbbbbb", lw=0.8, ls="--", zorder=1)
        gap_line(ax, d_.year, d_.exposed_m, col, "o", ms=3.8, lw=1.1, zorder=3)
        q = quick_trend(d_.year, d_.exposed_m)
        if q is not None: sen_band(ax, d_.year, d_.exposed_m, "#222222", sig=q["trend"] != "no trend", band=False, lw=1.2)
        ax.set_ylim(0, ymax); ax.set_xlim(LANDSAT_YEARS[0] - 0.8, LANDSAT_YEARS[-1] + 0.8); ax.set_ylabel("Exposed shore width (m)")
        if q:
            tiny = abs(q["slope"] * (LANDSAT_YEARS[-1] - LANDSAT_YEARS[0])) < EDGE_ERR_M
            vt = "no clear trend" if q["trend"] == "no trend" else (f"{q['trend']} but sub-pixel" if tiny else q["trend"].upper())
        ax.set_title(f"{SITES[sid]['name']}" + (f": {vt}, {10 * q['slope']:+.0f} m/decade" if q else ""), loc="left", fontsize=10)
    axb = axs.ravel()[len(sites_s)]; xx = np.arange(len(sh)); wbar = 0.26
    axb.axhspan(0, EDGE_ERR_M, color="#e5e5e5", lw=0, zorder=0)
    for k, lab in enumerate(DECADE_LABEL):
        m_, h_ = sh[f"mean_{lab}"].values, sh[f"ci95_{lab}"].values
        axb.bar(xx + (k - 1) * wbar, m_, wbar, color=DEC_COL[k], edgecolor="#444444", lw=0.6, zorder=3, label=lab)
        axb.errorbar(xx + (k - 1) * wbar, m_, yerr=h_, fmt="none", ecolor="k", capsize=2.5, lw=0.9, zorder=4)
    axb.set_xticks(xx); axb.set_xticklabels([SITES[s_]["name"].replace(" Dam", "") for s_ in sh.site], fontsize=9); axb.set_ylim(bottom=0)
    axb.set_ylabel("Mean exposed width (m)"); axb.set_title("Decade means (95% CI)", loc="left", fontsize=10); axb.legend(fontsize=8, loc="upper left", ncol=3)
    for a_ in axs.ravel()[len(sites_s) + 1:]: a_.axis("off")
    hs = [Patch(color=DEC_COL[k], label=f"decade {lab} (bar on top, mean line in panel)") for k, lab in enumerate(DECADE_LABEL)] + \
         [Patch(color="#e5e5e5", label=f"within edge uncertainty (±{EDGE_ERR_M:.0f} m, half a pixel)"), Line2D([0], [0], color="#222222", lw=1.2, label="Sen trend (solid = significant, dashed = not)")]
    finish(fig, "S11_shoreline_retreat", "How far has the waterline pulled back at the end of the dry season, decade by decade?",
           "Exposed shore = (usual footprint area minus water area) / length of the usual outline: the mean width of lake bed left uncovered (it cannot be negative). "
           f"Each decade ({', '.join(DECADE_LABEL)}) is summarised by its mean; overlapping 95% intervals mean the decades are not distinguishable. Panels share one y-axis so dams can be compared.",
           source="Landsat 5/7/8/9 C2-L2 (30 m)", handles=hs, legend_cols=3)
    EXTRA_LINES.append("Shoreline (mean exposed width, end of dry season, " + " / ".join(DECADE_LABEL) + "): " +
                       "; ".join(f"{SITES[s_]['name'].replace(' Dam', '')} " + " / ".join(f"{r_[f'mean_{l}']:.0f}" for l in DECADE_LABEL) + " m" for s_, (_, r_) in zip(sh.site, sh.iterrows())) + " (higher = waterline further back).")


## Step 5. Copperbelt synthesis
Core: `10_trend_scoreboard`, all dams side by side with water and canopy trends, their 95% ranges and the years each trend really covers. Supplementary: `S12` synchrony between dams, `S13` site map, `S14` the auto-written conclusion as a page (also printed below and saved to `tables/copperbelt_conclusion.txt`).

In [ ]:
# Cell 12: COPPERBELT SYNTHESIS. One row per water body, then the regional picture and an automatic, hedged conclusion.
rows = []
for sid, r in RES.items():
    s = SITES[sid]; row = {"site": sid, "name": s["name"], "town": s["town"], "kind": s["kind"], "lon": s["lon"], "lat": s["lat"]}
    if r.get("error"):
        row.update({"status": r["error"]}); rows.append(row); continue
    row.update({"status": "ok", "footprint_ha": r["fp"]["area_ha"], "reliability": r["rel"], "why": "; ".join(r["why"])})
    for sk in SEASONS:
        tt = r["trend"].get(sk)
        x, xp, xr_ = pct_row(tt, "water_ha"), pct_row(tt, "water_pct_foot"), pct_row(tt, "ring_moist_mean")
        row.update({f"{sk}_first_year": np.nan if x is None else x.first_year, f"{sk}_full_record": None if x is None else bool(x.full_record), f"{sk}_span": "" if x is None else span_label(x)})
        row.update({f"{sk}_trend": None if x is None else x.mk_trend, f"{sk}_p": np.nan if x is None else x.mk_p, f"{sk}_pct_dec": np.nan if x is None else x.pct_dec,
                    f"{sk}_ci_lo": np.nan if x is None else x.ci_lo, f"{sk}_ci_hi": np.nan if x is None else x.ci_hi, f"{sk}_n": np.nan if x is None else x.n_years,
                    f"{sk}_pts_dec": np.nan if xp is None else 10 * xp.sen_slope_per_year,
                    f"{sk}_ring_slope_dec": np.nan if xr_ is None else 10 * xr_.sen_slope_per_year, f"{sk}_ring_p": np.nan if xr_ is None else xr_.mk_p,
                    f"{sk}_ring_lo": np.nan if xr_ is None else 10 * xr_.sen_ci_low, f"{sk}_ring_hi": np.nan if xr_ is None else 10 * xr_.sen_ci_high})
        rr = r.get("rain", {}).get(sk, {})
        row.update({f"{sk}_rain_rho": rr.get("rain_rho", np.nan), f"{sk}_rain_p": rr.get("rain_p", np.nan), f"{sk}_resid_trend": rr.get("resid_trend"), f"{sk}_resid_p": rr.get("resid_p", np.nan)})
    rows.append(row)
SUMM = pd.DataFrame(rows); SUMM.to_csv(DIRS["tables"] / "copperbelt_site_summary.csv", index=False)
show_cols = [c for c in ["name", "kind", "footprint_ha", "reliability", "dry_trend", "dry_p", "dry_pct_dec", "mam_trend", "mam_p", "mam_pct_dec", "dry_rain_rho", "dry_rain_p", "dry_resid_trend"] if c in SUMM.columns]
display(SUMM[show_cols].round(3))

REG = SUMM[(SUMM.status == "ok") & SUMM.kind.isin(["reservoir", "river"]) & SUMM.get("dry_pct_dec", pd.Series(dtype=float)).notna()].copy()
if len(REG) == 0:
    print("No reservoir/river site has a testable trend yet; check the screening table and the data-collection output.")
else:
    REG = REG.sort_values("dry_pct_dec").reset_index(drop=True); RELA = {"High": 1.0, "Medium": 0.7, "Low": 0.4}

    # ------------- Figure 13: the scoreboard -------------
    fig, ax = plt.subplots(1, 2, figsize=(16.5, 0.70 * len(REG) + 3.8), gridspec_kw={"width_ratios": [1.5, 1]})
    yy = np.arange(len(REG))[::-1]
    allci = np.r_[REG.dry_ci_lo.values, REG.dry_ci_hi.values, REG.mam_ci_lo.values, REG.mam_ci_hi.values]; lim = max(float(np.nanmax(np.abs(allci))) * 1.08, 10) if np.isfinite(allci).any() else 30
    ax[0].axvspan(-MIN_EFFECT_PCT_DEC, MIN_EFFECT_PCT_DEC, color="#eeeeee", lw=0, zorder=0)
    for i, r in REG.iterrows():
        y = yy[i]; nf, nt = RES[r.site].get("mam_fail", {}).get("landsat", (0, 1))
        for sk, col, mk_, off in [("mam", C_RAINS, "o", 0.17), ("dry", C_DRYS, "s", -0.17)]:
            v, lo, hi = r[f"{sk}_pct_dec"], r[f"{sk}_ci_lo"], r[f"{sk}_ci_hi"]
            if not np.isfinite(v): continue
            weak = (sk == "mam" and nt and nf / nt > 0.3)            # end-of-rains water too often undetected at this dam
            c_ = C_GREY if weak else col
            ax[0].plot([lo, hi], [y + off, y + off], color=c_, lw=2, solid_capstyle="round", ls=":" if weak else "-")
            ax[0].plot(v, y + off, mk_, color=c_, mfc=c_ if r[f"{sk}_trend"] in ("decreasing", "increasing") else "white", mew=2, ms=9)
            if weak: ax[0].text(hi, y + off, f"  {nf}/{nt} end-of-rains values failed physics check", va="center", fontsize=7.5, color=C_GREY)
    ax[0].axvline(0, c="k", lw=1); ax[0].set_yticks(yy)
    ax[0].set_yticklabels([f"{n}  [{rl} reliability]\ndry season: {sp}" for n, rl, sp in zip(REG.name, REG.reliability, REG.dry_span)], fontsize=8.5)
    ax[0].set_xlim(-lim, lim); ax[0].set_xlabel("Change in water-covered area per decade (% of the site's own average)"); ax[0].grid(True, axis="x"); ax[0].grid(False, axis="y")
    ax[0].text(0.01, 1.01, "< shrinking", transform=ax[0].transAxes, color=C_DRY, fontsize=9, fontweight="bold"); ax[0].text(0.99, 1.01, "growing >", transform=ax[0].transAxes, color=C_WATER, fontsize=9, fontweight="bold", ha="right")
    ax[0].set_title("A. Surface water (open-water NDWI)", loc="left", pad=16)
    ax[1].axvline(0, c="k", lw=1)
    for i, r in REG.iterrows():
        y = yy[i]; v, lo, hi, p = r.mam_ring_slope_dec, r.mam_ring_lo, r.mam_ring_hi, r.mam_ring_p
        if not np.isfinite(v): continue
        col = (C_MOIST if v > 0 else C_DRY) if p < 0.05 else C_GREY
        ax[1].plot([lo, hi], [y, y], color=col, lw=2); ax[1].plot(v, y, "D", color=col, mfc=col if p < 0.05 else "white", mew=2, ms=8)
    mm_ = np.nanmax(np.abs(np.r_[REG.mam_ring_lo.values, REG.mam_ring_hi.values])); mm_ = mm_ if np.isfinite(mm_) and mm_ > 0 else 0.01; ax[1].set_xlim(-mm_ * 1.15, mm_ * 1.15)
    ax[1].set_yticks(yy); ax[1].set_yticklabels([]); ax[1].set_xlabel("Change in canopy NDWI per decade (Landsat 5/7 harmonised to 8/9)"); ax[1].grid(True, axis="x"); ax[1].grid(False, axis="y")
    ax[1].text(0.01, 1.01, "< drying (stress)", transform=ax[1].transAxes, color=C_DRY, fontsize=9, fontweight="bold"); ax[1].text(0.99, 1.01, "wetter >", transform=ax[1].transAxes, color=C_MOIST, fontsize=9, fontweight="bold", ha="right")
    ax[1].set_title(f"B. Canopy water of vegetation within {RING_M} m of the shore ({season_label(VEG_SEASON)})", loc="left", pad=16)
    hs = [Line2D([0], [0], marker="o", color=C_RAINS, ls="-", mfc=C_RAINS, ms=9, label="End of rains (Mar-May)"), Line2D([0], [0], marker="s", color=C_DRYS, ls="-", mfc=C_DRYS, ms=9, label="End of dry season (Aug-Oct)"),
          Line2D([0], [0], marker="o", color=C_GREY, ls="none", mfc="white", mew=2, ms=9, label="Hollow = not significant / too small"), Line2D([0], [0], color=C_GREY, lw=2, label="Bar = 95% range of the Sen slope"),
          Patch(color="#eeeeee", label=f"within ±{MIN_EFFECT_PCT_DEC}% per decade (too small to matter)")]
    finish(fig, "10_trend_scoreboard", "Is surface water shrinking across the Copperbelt? Site-by-site scoreboard",
           f"Each row is one water body. Filled symbols are trends that are statistically significant (p < 0.05, Hamed-Rao, allowing for year-to-year memory) AND larger than ±{MIN_EFFECT_PCT_DEC}% per decade. "
           f"A range that crosses zero is not distinguishable from no change. Under each name: reliability and the years the trend really covers; PARTIAL = fewer than {MIN_YEARS_PER_END} usable years in {EARLY[0]}-{EARLY[1]}, so it is not a full {LANDSAT_YEARS[0]}-{LANDSAT_YEARS[-1]} trend.", handles=hs, legend_cols=5)

    # ------------- Figure 14: do they move together? -------------
    series, names = [], []
    for _, r in REG.iterrows():
        st = RES[r.site]["stats"].get((RES[r.site]["prim"], WATER_SEASON))
        if st is None: continue
        series.append(st[st.ok_site].set_index("year").water_pct_foot_z.reindex(LANDSAT_YEARS)); names.append(r["name"])
    ZS = pd.DataFrame(series, index=names)
    spis = [RES[s]["drivers"][DROUGHT_SPI].reindex(LANDSAT_YEARS) for s in REG.site if RES[s].get("drivers") is not None]
    if spis: ZS.loc["Regional rainy-season drought index (SPI-6, Dec-May)"] = pd.concat(spis, axis=1).mean(axis=1)
    fig, ax = plt.subplots(figsize=(15.5, 0.55 * len(ZS) + 3.2))
    stripes(fig, ax, ZS, list(ZS.index), vmax=2.5, cbar_label="Anomaly vs the site's own average (standard deviations)")
    if spis: ax.axhline(len(ZS) - 1.5, color="k", lw=1.5)
    ax.set_xlabel("Year (grey = year not usable)")
    rel_names = [n for n, r in zip(REG.name, REG.reliability) if r in ("High", "Medium")]
    sync = np.nan; Zr = ZS.loc[[n for n in ZS.index if n in rel_names]]
    if len(Zr) >= 3:
        cm_ = Zr.T.corr(method="spearman", min_periods=10).values; sync = float(np.nanmean(cm_[np.triu_indices(len(Zr), 1)]))
    finish(fig, "S12_synchrony", "Do the Copperbelt water bodies rise and fall together?",
           "Each row is the yearly water cover of one site (end of dry season), brown = less water than that site's own norm, blue-green = more. The bottom row is regional rainfall. If the rows share the same pattern as each other and as rainfall, climate is the common driver; "
           "if they look unrelated, local factors (abstraction, dam operation, siltation, mining) dominate." + ("" if not np.isfinite(sync) else f" Average agreement between reliable sites: rho = {sync:.2f}."),
           source="Landsat 5/7/8/9 C2-L2 (30 m); " + str(next((v[1] for v in CLIM.values() if v[1]), "")))

    # ------------- Figure 15: map -------------
    fig, ax = plt.subplots(figsize=(10.5, 8.2)); MK = {"reservoir": "o", "river": "^", "new_reservoir": "s", "sinkhole": "*"}
    for _, r in SUMM[SUMM.status == "ok"].iterrows():
        sig = r.get("dry_p", np.nan); tr = r.get("dry_trend")
        col = (C_DRY if tr == "decreasing" else C_WATER) if tr in ("decreasing", "increasing") else C_GREY
        if r.kind in ("new_reservoir", "sinkhole"): col = "#bbbbbb"
        ax.scatter(r.lon, r.lat, s=60 + 25 * np.sqrt(max(r.footprint_ha, 1)), marker=MK.get(r.kind, "o"), c=col, edgecolor="black", lw=1.2, linestyle="--" if r.reliability == "Low" else "-", zorder=3)
        ax.annotate(f"{r['name']}\n{r.reliability} reliability", (r.lon, r.lat), xytext=(9, 6), textcoords="offset points", fontsize=8.5)
    ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude"); ax.set_aspect(1 / np.cos(np.deg2rad(-13))); ax.grid(True, axis="both")
    ax.xaxis.set_major_formatter(mpl.ticker.FuncFormatter(lon_fmt)); ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lat_fmt)); ax.margins(0.15)
    x0_, x1_ = ax.get_xlim(); y0_, y1_ = ax.get_ylim(); dlon = 10 / (111.32 * np.cos(np.deg2rad(13)))          # 10 km scale bar
    bx, by = x0_ + 0.05 * (x1_ - x0_), y0_ + 0.06 * (y1_ - y0_); ax.plot([bx, bx + dlon], [by, by], color="k", lw=3); ax.text(bx + dlon / 2, by + 0.012 * (y1_ - y0_), "10 km", ha="center", va="bottom", fontsize=8.5)
    hs = [Line2D([0], [0], marker="o", ls="none", mfc=C_DRY, mec="k", ms=10, label="Significant decline (dry season)"), Line2D([0], [0], marker="o", ls="none", mfc=C_WATER, mec="k", ms=10, label="Significant increase"),
          Line2D([0], [0], marker="o", ls="none", mfc=C_GREY, mec="k", ms=10, label="No clear trend"), Line2D([0], [0], marker="s", ls="none", mfc="#bbbbbb", mec="k", ms=10, label="New dam (no 30-yr baseline)"),
          Line2D([0], [0], marker="*", ls="none", mfc="#bbbbbb", mec="k", ms=13, label="Reference sinkhole lake"), Line2D([0], [0], marker="^", ls="none", mfc="white", mec="k", ms=10, label="River reach")]
    finish(fig, "S13_site_map", f"Where is the water shrinking? Copperbelt sites, {LANDSAT_YEARS[0]}-{LANDSAT_YEARS[-1]}", "Colour = end-of-dry-season water trend; symbol size = the lake's usual footprint; dashed outline = low measurement reliability.", handles=hs, legend_cols=3,
           source="Landsat 5/7/8/9 C2-L2 (30 m); site pins from Google Places")

    # ------------- Conclusion -------------
    rel = REG[REG.reliability.isin(["High", "Medium"])]; N, Nall = len(rel), len(REG)
    cnt = lambda d, c: (int(((d[f"{c}_trend"] == "decreasing") & (d[f"{c}_p"] < 0.05)).sum()), int(((d[f"{c}_trend"] == "increasing") & (d[f"{c}_p"] < 0.05)).sum()))
    dec_d, inc_d = cnt(rel, "dry") if N else (0, 0); dec_m, inc_m = cnt(rel, "mam") if N else (0, 0)
    lines = [f"Analysis period {LANDSAT_YEARS[0]}-{LANDSAT_YEARS[-1]} for every site. {N} of {Nall} reservoir/river sites passed the reliability checks (High or Medium)."]
    _part = REG[REG.dry_full_record == False]
    # ---- decade summary: mean end-of-dry-season water cover and end-of-rains canopy NDWI per dam and decade ----
    DEC_ROWS = []
    for sid_ in REG.site:
        rr_ = RES[sid_]
        for met_, sk_, lab_ in [("water_pct_foot", WATER_SEASON, "water cover (% of footprint), end of dry season"), ("ring_moist_mean", VEG_SEASON, "canopy NDWI (harmonised), end of rains")]:
            st_ = rr_["stats"].get((rr_["prim"], sk_))
            if st_ is None: continue
            ok_ = st_[st_.ok_site & st_[met_].notna()]; row_ = {"site": sid_, "name": SITES[sid_]["name"], "variable": lab_}
            for (w0, w1), dl in zip(DECADES, DECADE_LABEL):
                v_ = ok_[(ok_.year >= w0) & (ok_.year <= w1)][met_]; row_[dl] = v_.mean() if len(v_) >= 3 else np.nan; row_[f"n {dl}"] = len(v_)
            DEC_ROWS.append(row_)
    DEC_TAB = pd.DataFrame(DEC_ROWS)
    if len(DEC_TAB):
        print("Decade means (a decade needs >= 3 usable years):"); display(DEC_TAB.round(3)); globals().setdefault("EXTRA_SHEETS", {})["decade_summary"] = DEC_TAB
        w_ = DEC_TAB[DEC_TAB.variable.str.startswith("water")]
        lines.append("Water cover by decade (" + " / ".join(DECADE_LABEL) + ", % of usual footprint at the end of the dry season): " +
                     "; ".join(f"{r_['name'].replace(' Dam', '')} " + " / ".join("n/a" if not np.isfinite(r_[d]) else f"{r_[d]:.0f}" for d in DECADE_LABEL) for _, r_ in w_.iterrows()) + ".")
    if len(_part): lines.append("Partial records (fewer than " + str(MIN_YEARS_PER_END) + f" usable dry-season years in {EARLY[0]}-{EARLY[1]}): " + "; ".join(f"{r_['name']} {r_.dry_span.replace('PARTIAL record ', '')}" for _, r_ in _part.iterrows()) + ". Their trends are not directly comparable with full-record sites.")
    if N:
        lines.append(f"End of dry season: {dec_d} of {N} reliable sites lost water cover significantly, {inc_d} gained, {N - dec_d - inc_d} show no clear trend. End of rains: {dec_m} lost, {inc_m} gained, {N - dec_m - inc_m} no clear trend.")
    rains = [RES[s]["drivers"].wet_season_ppt_mm.reindex(LANDSAT_YEARS) for s in REG.site if RES[s].get("drivers") is not None]
    if rains:
        qr = quick_trend(LANDSAT_YEARS, pd.concat(rains, axis=1).mean(axis=1).values)
        if qr: lines.append(f"Regional wet-season rainfall: {phrase_p(qr['p'], qr['trend']).lower()}, {qr['slope'] * 10:+.0f} mm per decade.")
    if np.isfinite(sync):
        lines.append(f"Synchrony between reliable sites: rho = {sync:.2f} " + ("(they move together, which points to a shared regional driver such as rainfall)." if sync > 0.4 else "(weak, so each site is mostly following its own local drivers: dam operation, abstraction, siltation or mining)." if sync < 0.2 else "(moderate, so regional climate and local factors both matter)."))
    if N:
        follow = int((rel.dry_rain_p < 0.05).sum()); persist = int(((rel.dry_resid_trend == "decreasing") & (rel.dry_resid_p < 0.05)).sum())
        lines.append(f"Water cover tracks wet-season rainfall significantly at {follow} of {N} reliable sites; a significant decline is still present after removing the rainfall effect at {persist} of {N}" + (" (so rainfall alone does not explain the loss)." if persist and dec_d and persist >= dec_d * 0.5 else "."))
    ref = SUMM[(SUMM.kind == "sinkhole") & (SUMM.status == "ok")]
    if len(ref): lines.append("Reference sinkhole lakes: " + "; ".join(f"{r['name']} dry-season trend: {r.dry_trend or 'n/a'}" for _, r in ref.iterrows()) + ". Stable references suggest the satellite signal itself is not drifting (Sentinel-2 period only, so this is a partial check).")
    for _, r in SUMM[(SUMM.kind == "new_reservoir") & (SUMM.status == "ok")].iterrows(): lines.append(f"{r['name']} is a new reservoir: the water-cover curve in its site sheet shows the filling, there is no 30-year baseline.")
    if len(globals().get("HARM_TAB", [])):
        hv = HARM_TAB[(HARM_TAB.season == VEG_SEASON)]
        lines.append(f"Sensor harmonisation: Landsat 8/9 read canopy NDWI {hv[hv.variable == 'ndwi_moist'].offset_OLI_minus_TM_ETM.median():+.3f} and NDVI {hv[hv.variable == 'ndvi'].offset_OLI_minus_TM_ETM.median():+.3f} higher than Landsat 5/7 for the same vegetation (median over dams, end of rains). "
                     "These offsets were removed before every vegetation trend; without that, the 2013 sensor switch appears as region-wide 'greening and wetting'.")
    nf_all = sum(v.get("mam_fail", {}).get("landsat", (0, 0))[0] for v in RES.values() if not v.get("error")); nt_all = sum(v.get("mam_fail", {}).get("landsat", (0, 0))[1] for v in RES.values() if not v.get("error"))
    if nt_all: lines.append(f"Physics check: {nf_all} of {nt_all} end-of-rains water values were lower than the following dry season by more than {MAX_REFILL_PTS} points, which is impossible without dry-season inflow; they were treated as detection failures and removed.")
    lines.extend(globals().get("EXTRA_LINES", []))
    lines.append("Caution: significant is not the same as caused by climate. Management, abstraction and mining can change these water bodies independently of rainfall.")
    print("\n" + "=" * 78 + "\nCOPPERBELT CONCLUSION (auto-generated; read against the figures)\n" + "=" * 78)
    for l in lines: print("-", l)
    open(DIRS["tables"] / "copperbelt_conclusion.txt", "w").write("\n".join(lines))
    body = "\n\n".join("• " + textwrap.fill(l, 125, subsequent_indent="   ") for l in lines); nl = body.count("\n") + 1
    fig = plt.figure(figsize=(14, 2.6 + 0.30 * nl)); ax = fig.add_subplot(111); ax.axis("off")
    ax.text(0.0, 1.0, body, va="top", ha="left", fontsize=11.5, linespacing=1.35)
    finish(fig, "S14_conclusion", "What the satellites say about water in the Copperbelt", "Generated from the statistics in this run; read it together with the scoreboard and the reliability flags.")


## Step 6. Export

In [ ]:
# Cell 13: EXPORT every table into one workbook + metadata
import platform, importlib
sheets = {"site_summary": SUMM, "screening": scr}
for sid, r in RES.items():
    if r.get("error"): continue
    for (tag, sk), st in r["stats"].items(): sheets[f"{sid[:10]}_{tag}_{sk}"[:31]] = st
    for sk, tt in r["trend"].items(): sheets[f"{sid[:10]}_trend_{sk}"[:31]] = tt
sheets.update(globals().get("EXTRA_SHEETS", {}))
lu = [r["landuse"]["table"] for r in RES.values() if r.get("landuse") is not None]
if lu: sheets["landcover_trends"] = pd.concat(lu, ignore_index=True)
with pd.ExcelWriter(DIRS["tables"] / "Copperbelt_results.xlsx") as w:
    for name, df in sheets.items():
        if df is not None and len(df): df.to_excel(w, sheet_name=name[:31], index=False)
json.dump({"sites": SITES, "crs": CRS, "seasons": SEASONS, "landsat_years": [LANDSAT_YEARS[0], LANDSAT_YEARS[-1]], "water_threshold": WATER_THR,
           "python": platform.python_version(), "packages": {p: getattr(importlib.import_module(p), "__version__", "?") for p in ["numpy", "pandas", "xarray", "scipy", "pymannkendall"]},
           "created": pd.Timestamp.now().isoformat()}, open(ROOT / "metadata.json", "w"), indent=2, default=str)
print("Saved:", DIRS["tables"] / "Copperbelt_results.xlsx", "\nFigures in:", DIRS["figures"].resolve())
